# Chapter 5 forecasting and inventory workflow

This notebook is the main computational record for the thesis. It defines the five forecasting methods, prepares the fixed M5 sample, generates rolling-origin forecasts, evaluates point and quantile accuracy, constructs residual-based inventory targets, runs the baseline inventory simulation, and produces the original paired-bootstrap results.

Run the cells in order. The ARIMA/SARIMA production sections are computationally expensive and use checkpoint files in `chapter5_results`. Set the environment variable `M5_DATA_DIR` to the folder containing the three M5 CSV files and `selected_sample_ids.csv`. If the variable is absent, the original `~/Desktop/m5_data` location is used.


## Software environment

Run this cell first and retain its output with any new full execution. The historical notebook metadata record Python 3.12.7; exact microversions of all dependencies were not separately saved during the original long-running fit.


In [ ]:
import importlib.metadata as metadata
import json
import os
import platform
import sys

PACKAGE_NAMES = [
    "numpy",
    "pandas",
    "scipy",
    "statsmodels",
    "pmdarima",
    "scikit-learn",
    "xgboost",
    "torch",
    "tirex2",
    "matplotlib",
]

software_environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
}

for package_name in PACKAGE_NAMES:
    try:
        software_environment[package_name] = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        software_environment[package_name] = "not installed"

print(json.dumps(software_environment, indent=2))


In [15]:
import itertools
import warnings

import numpy as np
import pandas as pd

FORECAST_HORIZON = 28
SEASONAL_PERIOD = 7
RANDOM_SEED = 2026


Seasonal Naive Model

In [18]:
def seasonal_naive_forecast(
    training_values,
    horizon=FORECAST_HORIZON,
    seasonal_period=SEASONAL_PERIOD
):
    training_values = np.asarray(training_values, dtype=float)

    if len(training_values) < seasonal_period:
        raise ValueError("At least seven observations are required.")

    final_week = training_values[-seasonal_period:]

    repetitions = int(np.ceil(horizon / seasonal_period))

    forecast = np.tile(final_week, repetitions)[:horizon]

    # Demand cannot be negative.
    forecast = np.maximum(forecast, 0)

    return forecast


Automatic ETS

In [21]:
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.exponential_smoothing.ets import ETSModel


def automatic_ets_forecast(
    training_values,
    horizon=FORECAST_HORIZON
):
    training_values = np.asarray(training_values, dtype=float)

    candidates = [
        {
            "trend": None,
            "damped_trend": False,
            "name": "no_trend"
        },
        {
            "trend": "add",
            "damped_trend": False,
            "name": "additive_trend"
        },
        {
            "trend": "add",
            "damped_trend": True,
            "name": "damped_additive_trend"
        }
    ]

    successful_models = []

    for candidate in candidates:

        try:
            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore",
                    ConvergenceWarning
                )

                model = ETSModel(
                    training_values,
                    error="add",
                    trend=candidate["trend"],
                    damped_trend=candidate["damped_trend"],
                    seasonal="add",
                    seasonal_periods=SEASONAL_PERIOD,
                    initialization_method="estimated"
                )

                result = model.fit(
                    maxiter=1000,
                    disp=False
                )

            aicc = float(result.aicc)

            if np.isfinite(aicc):

                converged = result.mle_retvals.get(
                    "converged",
                    False
                )

                successful_models.append({
                    "result": result,
                    "name": candidate["name"],
                    "aicc": aicc,
                    "converged": converged
                })

        except Exception as error:
            print(
                "ETS candidate failed:",
                candidate["name"],
                error
            )

    if len(successful_models) == 0:
        raise RuntimeError("All ETS candidates failed.")

    converged_models = [
        model
        for model in successful_models
        if model["converged"]
    ]

    if len(converged_models) > 0:
        eligible_models = converged_models
    else:
        eligible_models = successful_models

    selected_model = min(
        eligible_models,
        key=lambda model: model["aicc"]
    )

    forecast = selected_model["result"].forecast(horizon)

    forecast = np.asarray(forecast, dtype=float)
    forecast = np.maximum(forecast, 0)

    information = {
        "selected_specification": selected_model["name"],
        "aicc": selected_model["aicc"],
        "converged": selected_model["converged"]
    }

    return forecast, information


Automatic ARIMA/SARIMA

In [24]:
from pmdarima.arima import auto_arima


def automatic_arima_forecast(
    training_values,
    horizon=FORECAST_HORIZON
):
    training_values = np.asarray(training_values, dtype=float)

    model = auto_arima(
        training_values,

        # Non-seasonal orders
        start_p=1,
        start_q=1,
        max_p=3,
        max_q=3,
        d=None,
        max_d=2,

        # Seasonal orders
        start_P=0,
        start_Q=0,
        max_P=1,
        max_Q=1,
        D=None,
        max_D=1,

        m=SEASONAL_PERIOD,
        seasonal=True,

        # Model selection
        information_criterion="aicc",
        stepwise=True,
        max_order=None,

        # Differencing tests
        test="kpss",
        seasonal_test="ocsb",

        # Estimation
        method="lbfgs",
        maxiter=200,

        suppress_warnings=True,
        error_action="raise",
        trace=False,

        with_intercept="auto",

        sarimax_kwargs={
            "simple_differencing": False,
            "enforce_stationarity": False,
            "enforce_invertibility": False
        }
    )

    forecast = model.predict(
        n_periods=horizon
    )

    forecast = np.asarray(forecast, dtype=float)
    forecast = np.maximum(forecast, 0)

    converged = model.arima_res_.mle_retvals.get(
        "converged",
        False
    )

    information = {
        "order": model.order,
        "seasonal_order": model.seasonal_order,
        "aicc": model.aicc(),
        "converged": converged
    }

    return forecast, information


Global XGBoost

Feature Construction

In [8]:
LAGS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 28]


def create_xgboost_features(panel):

    panel = panel.sort_values(
        ["id", "day_number"]
    ).copy()

    def calculate_series_features(series):

        series = series.sort_values(
            "day_number"
        ).copy()

        past_demand = series["demand"].shift(1)

        for lag in LAGS:
            series[f"lag_{lag}"] = (
                series["demand"].shift(lag)
            )

        for window in ROLLING_WINDOWS:

            series[f"rolling_mean_{window}"] = (
                past_demand
                .rolling(window)
                .mean()
            )

            series[f"rolling_sd_{window}"] = (
                past_demand
                .rolling(window)
                .std(ddof=0)
            )

            series[f"zero_proportion_{window}"] = (
                past_demand
                .eq(0)
                .rolling(window)
                .mean()
            )

        # Only prices already observed before the target day.
        series["price_level"] = (
            series["sell_price"].shift(1)
        )

        series["price_change_7"] = (
            series["sell_price"].shift(1)
            - series["sell_price"].shift(8)
        )

        series["price_pct_change_7"] = (
            series["price_change_7"]
            / series["sell_price"].shift(8)
        )

        return series

    panel = (
        panel
        .groupby("id", group_keys=False)
        .apply(calculate_series_features)
        .reset_index(drop=True)
    )

    panel = panel.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return panel


In [10]:
NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_sd_7",
    "zero_proportion_7",
    "rolling_mean_28",
    "rolling_sd_28",
    "zero_proportion_28",
    "weekday",
    "month",
    "snap",
    "price_level",
    "price_change_7",
    "price_pct_change_7"
]

CATEGORICAL_FEATURES = [
    "item_id",
    "cat_id",
    "dept_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]


Categorical Encoding

In [13]:
from sklearn.preprocessing import OrdinalEncoder


def prepare_xgboost_data(training_frame):

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32
    )

    categorical_values = encoder.fit_transform(
        training_frame[
            CATEGORICAL_FEATURES
        ].astype(str)
    )

    numeric_values = training_frame[
        NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    features = np.column_stack([
        numeric_values,
        categorical_values
    ])

    target = training_frame[
        "demand"
    ].to_numpy(dtype=np.float32)

    return features, target, encoder


Hyperparameter Combinations

In [16]:
def xgboost_parameter_grid():

    parameter_grid = []

    for n_estimators, learning_rate, max_depth in itertools.product(
        [300, 600],
        [0.03, 0.05],
        [6, 10]
    ):

        parameter_grid.append({
            "n_estimators": n_estimators,
            "learning_rate": learning_rate,
            "max_depth": max_depth
        })

    return parameter_grid


Model Estimation

In [18]:
from xgboost import XGBRegressor


def fit_xgboost_model(
    features,
    target,
    parameters
):

    model = XGBRegressor(
        n_estimators=parameters["n_estimators"],
        learning_rate=parameters["learning_rate"],
        max_depth=parameters["max_depth"],

        subsample=0.8,
        colsample_bytree=0.8,

        objective="reg:squarederror",

        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        features,
        target,
        verbose=False
    )

    return model


Recursive 28-day forecasting

In [21]:
def recursive_xgboost_forecast(
    model,
    observed_history,
    create_next_day_features,
    horizon=FORECAST_HORIZON
):
    """
    observed_history:
        Array with dimensions:
        number of series x historical days

    create_next_day_features:
        Function that uses the current history to create
        the predictors for the next forecast day.
    """

    history = np.asarray(
        observed_history,
        dtype=float
    ).copy()

    forecasts = []

    for forecast_step in range(1, horizon + 1):

        next_day_features = create_next_day_features(
            history,
            forecast_step
        )

        prediction = model.predict(
            next_day_features
        )

        prediction = np.maximum(
            prediction,
            0
        )

        forecasts.append(prediction)

        # The prediction becomes part of the history.
        # Future realised demand is never inserted.
        history = np.column_stack([
            history,
            prediction
        ])

    forecasts = np.column_stack(forecasts)

    return forecasts


RMSSE for Hyperparamter Selection

In [24]:
def calculate_rmsse(
    actual,
    forecast,
    training_values
):

    forecast_mse = np.mean(
        (actual - forecast) ** 2
    )

    naive_mse = np.mean(
        np.diff(training_values) ** 2
    )

    if naive_mse == 0:
        return np.nan

    rmsse = np.sqrt(
        forecast_mse / naive_mse
    )

    return rmsse


In [26]:
def select_xgboost_configuration(validation_results):

    summary = (
        validation_results
        .groupby(
            [
                "n_estimators",
                "learning_rate",
                "max_depth"
            ],
            as_index=False
        )["rmsse"]
        .mean()
        .rename(
            columns={
                "rmsse": "mean_validation_rmsse"
            }
        )
        .sort_values(
            [
                "mean_validation_rmsse",
                "max_depth",
                "n_estimators"
            ]
        )
    )

    selected_configuration = summary.iloc[0]

    return selected_configuration, summary


Zero-Shot TiRex-2

In [50]:
import torch

from tirex2 import TimeseriesType, load_model


In [52]:
def load_tirex_model(device="auto"):

    if device == "auto":

        if torch.cuda.is_available():
            device = "cuda"

        elif torch.backends.mps.is_available():
            device = "mps"

        else:
            device = "cpu"

    model = load_model(
        "NX-AI/TiRex-2",
        device=device
    )

    return model, device


In [54]:
def create_tirex_input(
    target_history,
    past_price,
    future_known_covariates
):
    """
    target_history:
        One-dimensional demand history.

    past_price:
        One-dimensional historical price series.
        It must have the same length as target_history.

    future_known_covariates:
        Matrix with dimensions:
        number of covariates x
        (history length + 28 future days).
    """

    target_tensor = torch.tensor(
        target_history,
        dtype=torch.float32
    ).unsqueeze(0)

    price_tensor = torch.tensor(
        past_price,
        dtype=torch.float32
    ).unsqueeze(0)

    future_covariate_tensor = torch.tensor(
        future_known_covariates,
        dtype=torch.float32
    )

    tirex_input = TimeseriesType(
        target=target_tensor,
        past_covariates=price_tensor,
        future_covariates=future_covariate_tensor
    )

    return tirex_input


In [56]:
def tirex_forecast(
    model,
    tirex_input,
    horizon=FORECAST_HORIZON
):

    output = model.forecast(
        [tirex_input],
        prediction_length=horizon,
        output_type="numpy",
        batch_size=1
    )[0]

    # Output dimensions:
    # one target x nine quantiles x 28 days
    quantile_forecasts = np.asarray(
        output[0],
        dtype=float
    )

    quantile_forecasts = np.maximum(
        quantile_forecasts,
        0
    )

    quantile_levels = np.array([
        round(float(level), 6)
        for level in model.model.quantiles
    ])

    expected_levels = np.arange(
        0.1,
        1.0,
        0.1
    )

    if not np.allclose(
        quantile_levels,
        expected_levels
    ):
        raise ValueError(
            "Unexpected TiRex-2 quantile levels."
        )

    median_position = np.where(
        np.isclose(quantile_levels, 0.50)
    )[0][0]

    central_forecast = quantile_forecasts[
        median_position
    ]

    return {
        "quantile_levels": quantile_levels,
        "quantile_forecasts": quantile_forecasts,
        "central_forecast": central_forecast
    }


Common 28-day Forecast Error

In [59]:
def protection_period_error(
    actual_demand,
    daily_forecast
):

    actual_total = np.sum(
        actual_demand
    )

    forecast_total = np.sum(
        daily_forecast
    )

    error = (
        actual_total
        - forecast_total
    )

    return {
        "actual_protection_demand": actual_total,
        "forecast_protection_demand": forecast_total,
        "protection_period_error": error
    }


Define the file locations

In [62]:
from pathlib import Path
import numpy as np
import pandas as pd

# Folder containing the M5 files
DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))

# Input files
SALES_FILE = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_FILE = DATA_DIR / "calendar.csv"
PRICES_FILE = DATA_DIR / "sell_prices.csv"
SAMPLE_FILE = DATA_DIR / "selected_sample_ids.csv"

# Folder in which the Chapter 5 results will be saved
OUTPUT_DIR = DATA_DIR / "chapter5_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check whether all required files exist
required_files = [
    SALES_FILE,
    CALENDAR_FILE,
    PRICES_FILE,
    SAMPLE_FILE
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(f"File not found: {file}")

print("All required files were found.")
print("Data folder:", DATA_DIR)
print("Results folder:", OUTPUT_DIR)


All required files were found.
Data folder: <M5_DATA_DIR>
Results folder: <M5_DATA_DIR>/chapter5_results


Load and check the fixed sample

In [65]:
sales = pd.read_csv(SALES_FILE)
sample = pd.read_csv(SAMPLE_FILE)

# Allow either of the two column names.
if "demand_class" not in sample.columns:
    if "demand_pattern" in sample.columns:
        sample = sample.rename(
            columns={"demand_pattern": "demand_class"}
        )
    else:
        raise ValueError(
            "The sample file needs a demand_class or demand_pattern column."
        )

# Standardise the class names.
class_names = {
    "smooth": "Smooth",
    "erratic": "Erratic",
    "intermittent": "Intermittent",
    "lumpy": "Lumpy"
}

sample["demand_class"] = (
    sample["demand_class"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(class_names)
)

if sample["demand_class"].isna().any():
    raise ValueError("An unknown demand class was found.")

if sample["id"].duplicated().any():
    raise ValueError("The selected sample contains duplicated IDs.")

class_counts = sample["demand_class"].value_counts()

expected_counts = {
    "Smooth": 125,
    "Erratic": 125,
    "Intermittent": 125,
    "Lumpy": 125
}

if len(sample) != 500:
    raise ValueError(f"The sample contains {len(sample)} instead of 500 series.")

if class_counts.to_dict() != expected_counts:
    raise ValueError(
        f"The demand classes are not balanced: {class_counts.to_dict()}"
    )

# Retain the frozen IDs and demand classes.
sample_for_merge = (
    sample[["id", "demand_class"]]
    .sort_values("id")
    .reset_index(drop=True)
)

selected_sales = sample_for_merge.merge(
    sales,
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True
)

missing_ids = selected_sales.loc[
    selected_sales["_merge"] != "both", "id"
]

if len(missing_ids) > 0:
    raise ValueError(
        f"Some selected IDs were not found: {missing_ids.head().tolist()}"
    )

selected_sales = selected_sales.drop(columns="_merge")

def day_number(column_name):
    return int(column_name.replace("d_", ""))

day_columns = sorted(
    [column for column in sales.columns if column.startswith("d_")],
    key=day_number
)

if len(day_columns) < 1941:
    raise ValueError(
        "sales_train_evaluation.csv must contain demand through d_1941."
    )

demand_matrix = selected_sales[day_columns].to_numpy(dtype=float)

print("Selected series:", len(selected_sales))
print("Demand matrix:", demand_matrix.shape)
print(class_counts.sort_index())


Selected series: 500
Demand matrix: (500, 1941)
demand_class
Erratic         125
Intermittent    125
Lumpy           125
Smooth          125
Name: count, dtype: int64


/var/folders/qg/nvrwk7_51bngv6xjz8w6p5f40000gn/T/ipykernel_10516/2785134939.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  selected_sales = sample_for_merge.merge(
/var/folders/qg/nvrwk7_51bngv6xjz8w6p5f40000gn/T/ipykernel_10516/2785134939.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  selected_sales = sample_for_merge.merge(


Determine each series active period

In [67]:
calendar = pd.read_csv(CALENDAR_FILE)
prices = pd.read_csv(PRICES_FILE)

calendar["day_number"] = (
    calendar["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

calendar["date"] = pd.to_datetime(calendar["date"])
calendar["weekday"] = calendar["date"].dt.dayofweek
calendar["month"] = calendar["date"].dt.month

event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for column in event_columns:
    calendar[column] = calendar[column].fillna("none").astype(str)

calendar = (
    calendar
    .sort_values("day_number")
    .reset_index(drop=True)
)

# Find the first week with a valid positive price.
valid_prices = prices[
    prices["sell_price"].notna()
    & (prices["sell_price"] > 0)
].copy()

first_price_week = (
    valid_prices
    .groupby(["store_id", "item_id"], as_index=False)["wm_yr_wk"]
    .min()
    .rename(columns={"wm_yr_wk": "first_price_week"})
)

first_day_of_week = (
    calendar
    .groupby("wm_yr_wk")["day_number"]
    .min()
)

first_price_week["first_active_day"] = (
    first_price_week["first_price_week"]
    .map(first_day_of_week)
)

selected_sales = selected_sales.merge(
    first_price_week[
        [
            "store_id",
            "item_id",
            "first_price_week",
            "first_active_day"
        ]
    ],
    on=["store_id", "item_id"],
    how="left",
    validate="one_to_one"
)

if selected_sales["first_active_day"].isna().any():
    problematic_ids = selected_sales.loc[
        selected_sales["first_active_day"].isna(), "id"
    ]

    raise ValueError(
        "No positive price was found for: "
        f"{problematic_ids.head().tolist()}"
    )

selected_sales["first_active_day"] = (
    selected_sales["first_active_day"].astype(int)
)

print("Active periods were determined successfully.")
print(
    selected_sales[
        ["id", "demand_class", "first_active_day"]
    ].head()
)


Active periods were determined successfully.
                            id  demand_class  first_active_day
0  FOODS_1_005_WI_1_evaluation         Lumpy                 1
1  FOODS_1_010_WI_1_evaluation  Intermittent               519
2  FOODS_1_012_WI_3_evaluation         Lumpy               183
3  FOODS_1_015_WI_2_evaluation         Lumpy                64
4  FOODS_1_018_CA_2_evaluation       Erratic                 1


Create the daily historical price matrix

In [70]:
number_of_series = len(selected_sales)
number_of_days = len(day_columns)

price_matrix = np.full(
    shape=(number_of_series, number_of_days),
    fill_value=np.nan,
    dtype=float
)

weeks_by_day = calendar.loc[
    :number_of_days - 1, "wm_yr_wk"
].to_numpy()

price_groups = {
    key: group.set_index("wm_yr_wk")["sell_price"]
    for key, group in prices.groupby(["store_id", "item_id"])
}

for series_index, row in selected_sales.iterrows():

    key = (row["store_id"], row["item_id"])

    if key not in price_groups:
        continue

    daily_prices = (
        pd.Series(weeks_by_day)
        .map(price_groups[key])
        .astype(float)
        .ffill()
    )

    price_matrix[series_index] = daily_prices.to_numpy()

print("Price matrix:", price_matrix.shape)


Price matrix: (500, 1941)


Define the forecast periods

In [73]:
FORECAST_HORIZON = 28

forecast_schedule = pd.DataFrame({
    "forecast_origin": [
        1661, 1689, 1717, 1745, 1773, 1801,
        1829, 1857, 1885,
        1913
    ],
    "purpose": [
        "calibration", "calibration", "calibration",
        "calibration", "calibration", "calibration",
        "validation", "validation", "validation",
        "final_test"
    ],
    "round": [
        1, 2, 3, 4, 5, 6,
        1, 2, 3,
        1
    ]
})

forecast_schedule["forecast_start"] = (
    forecast_schedule["forecast_origin"] + 1
)

forecast_schedule["forecast_end"] = (
    forecast_schedule["forecast_origin"] + FORECAST_HORIZON
)

forecast_schedule


,forecast_origin,purpose,round,forecast_start,forecast_end
0,1661,calibration,1,1662,1689
1,1689,calibration,2,1690,1717
2,1717,calibration,3,1718,1745
3,1745,calibration,4,1746,1773
4,1773,calibration,5,1774,1801
5,1801,calibration,6,1802,1829
6,1829,validation,1,1830,1857
7,1857,validation,2,1858,1885
8,1885,validation,3,1886,1913
9,1913,final_test,1,1914,1941


Function for extracting one model input: This function prevents future demand from entering the training data

In [75]:
def get_series_input(series_index, forecast_origin):

    row = selected_sales.iloc[series_index]

    active_start = int(row["first_active_day"]) - 1

    training_demand = demand_matrix[
        series_index,
        active_start:forecast_origin
    ]

    actual_future_demand = demand_matrix[
        series_index,
        forecast_origin:forecast_origin + FORECAST_HORIZON
    ]

    historical_prices = price_matrix[
        series_index,
        active_start:forecast_origin
    ]

    future_calendar = calendar.iloc[
        forecast_origin:forecast_origin + FORECAST_HORIZON
    ].copy()

    return {
        "id": row["id"],
        "demand_class": row["demand_class"],
        "state_id": row["state_id"],
        "first_active_day": active_start + 1,
        "training_demand": training_demand,
        "historical_prices": historical_prices,
        "future_calendar": future_calendar,
        "actual_future_demand": actual_future_demand
    }


In [76]:
example_input = get_series_input(
    series_index=0,
    forecast_origin=1829
)

print("Series:", example_input["id"])
print("Training observations:", len(example_input["training_demand"]))
print("Forecast observations:", len(example_input["actual_future_demand"]))
print("Last training day: d_1829")
print("Forecast period: d_1830 to d_1857")


Series: FOODS_1_005_WI_1_evaluation
Training observations: 1829
Forecast observations: 28
Last training day: d_1829
Forecast period: d_1830 to d_1857


First smoke test for the individual-series models

In [78]:
y_train = example_input["training_demand"]

seasonal_naive_test = seasonal_naive_forecast(
    y_train,
    horizon=28
)

ets_test = automatic_ets_forecast(
    y_train,
    horizon=28
)

arima_test = automatic_arima_forecast(
    y_train,
    horizon=28
)

print("Seasonal-naïve forecast length:", len(seasonal_naive_test))
print("ETS forecast length:", len(ets_test))
print("ARIMA/SARIMA forecast length:", len(arima_test))

print("Seasonal naïve:", seasonal_naive_test[:5])
print("ETS:", ets_test[:5])
print("ARIMA/SARIMA:", arima_test[:5])


Seasonal-naïve forecast length: 28
ETS forecast length: 2
ARIMA/SARIMA forecast length: 2
Seasonal naïve: [1. 1. 3. 0. 1.]
ETS: (array([1.09187421, 1.15826403, 1.14791086, 1.2677745 , 1.35736524,
       1.36731962, 1.06713389, 1.09187421, 1.15826403, 1.14791086,
       1.2677745 , 1.35736524, 1.36731962, 1.06713389, 1.09187421,
       1.15826403, 1.14791086, 1.2677745 , 1.35736524, 1.36731962,
       1.06713389, 1.09187421, 1.15826403, 1.14791086, 1.2677745 ,
       1.35736524, 1.36731962, 1.06713389]), {'selected_specification': 'no_trend', 'aicc': 5841.054709534786, 'converged': True})
ARIMA/SARIMA: (array([1.22229835, 1.22201366, 1.24615824, 1.21036836, 1.22215942,
       1.2336607 , 1.22230175, 1.22480804, 1.22480804, 1.22480804,
       1.22480804, 1.22480804, 1.22480804, 1.22480804, 1.22480804,
       1.22480804, 1.22480804, 1.22480804, 1.22480804, 1.22480804,
       1.22480804, 1.22480804, 1.22480804, 1.22480804, 1.22480804,
       1.22480804, 1.22480804, 1.22480804]), {'order': 

Feed the data to XGBoost

In [84]:
import numpy as np
import pandas as pd

FINAL_TEST_ORIGIN = 1913

LAGS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 28]

CATEGORICAL_FEATURES = [
    "item_id",
    "cat_id",
    "dept_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_sd_7",
    "zero_proportion_7",
    "rolling_mean_28",
    "rolling_sd_28",
    "zero_proportion_28",
    "weekday",
    "month",
    "snap",
    "price_level",
    "price_change_7",
    "price_pct_change_7"
]


def safe_price_features(price_level, earlier_price):

    price_change = price_level - earlier_price

    price_pct_change = np.divide(
        price_change,
        earlier_price,
        out=np.zeros_like(price_change, dtype=float),
        where=np.isfinite(earlier_price) & (earlier_price != 0)
    )

    price_change = np.nan_to_num(
        price_change,
        nan=0.0
    )

    price_pct_change = np.nan_to_num(
        price_pct_change,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return price_change, price_pct_change


In [85]:
def build_historical_feature_panel(
    selected_sales,
    day_columns,
    calendar,
    price_matrix
):

    demand_values = selected_sales[
        day_columns
    ].to_numpy(dtype=float)

    calendar_by_day = calendar.set_index("day_number")

    panel_parts = []

    for series_index, row in selected_sales.reset_index(drop=True).iterrows():

        # At least 28 preceding observations are required.
        first_target_day = max(
            int(row["first_active_day"]) + 28,
            29
        )

        target_days = np.arange(
            first_target_day,
            FINAL_TEST_ORIGIN + 1
        )

        # d_1 has array position 0.
        target_indices = target_days - 1

        series_demand = demand_values[series_index]

        series_panel = pd.DataFrame({
            "id": row["id"],
            "target_day": target_days,
            "target": series_demand[target_indices],
            "item_id": row["item_id"],
            "cat_id": row["cat_id"],
            "dept_id": row["dept_id"],
            "store_id": row["store_id"],
            "state_id": row["state_id"]
        })

        # Lagged demand values
        for lag in LAGS:
            series_panel[f"lag_{lag}"] = (
                series_demand[target_indices - lag]
            )

        # Rolling demand features
        for window in ROLLING_WINDOWS:

            rolling_values = np.vstack([
                series_demand[index - window:index]
                for index in target_indices
            ])

            series_panel[f"rolling_mean_{window}"] = (
                rolling_values.mean(axis=1)
            )

            series_panel[f"rolling_sd_{window}"] = (
                rolling_values.std(axis=1, ddof=0)
            )

            series_panel[f"zero_proportion_{window}"] = (
                rolling_values == 0
            ).mean(axis=1)

        # Calendar information
        calendar_rows = calendar_by_day.loc[target_days]

        series_panel["weekday"] = (
            calendar_rows["weekday"].to_numpy()
        )

        series_panel["month"] = (
            calendar_rows["month"].to_numpy()
        )

        for event_column in [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]:
            series_panel[event_column] = (
                calendar_rows[event_column].to_numpy()
            )

        # State-specific SNAP indicator
        snap_column = f"snap_{row['state_id']}"

        if snap_column not in calendar_rows.columns:
            raise ValueError(
                f"The calendar does not contain {snap_column}."
            )

        series_panel["snap"] = (
            calendar_rows[snap_column].to_numpy(dtype=float)
        )

        # Historical price information
        price_level = price_matrix[
            series_index,
            target_indices - 1
        ]

        price_seven_days_earlier = price_matrix[
            series_index,
            target_indices - 8
        ]

        price_change, price_pct_change = safe_price_features(
            price_level,
            price_seven_days_earlier
        )

        series_panel["price_level"] = np.nan_to_num(
            price_level,
            nan=0.0
        )

        series_panel["price_change_7"] = price_change
        series_panel["price_pct_change_7"] = price_pct_change

        panel_parts.append(series_panel)

    historical_panel = pd.concat(
        panel_parts,
        ignore_index=True
    )

    return historical_panel


In [86]:
xgboost_panel = build_historical_feature_panel(
    selected_sales=selected_sales,
    day_columns=day_columns,
    calendar=calendar,
    price_matrix=price_matrix
)

print("XGBoost training rows:", len(xgboost_panel))
print("XGBoost columns:", xgboost_panel.columns.tolist())
print("Missing values:", xgboost_panel.isna().sum().sum())
print("Panel shape:", xgboost_panel.shape)


XGBoost training rows: 825229
XGBoost columns: ['id', 'target_day', 'target', 'item_id', 'cat_id', 'dept_id', 'store_id', 'state_id', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_sd_7', 'zero_proportion_7', 'rolling_mean_28', 'rolling_sd_28', 'zero_proportion_28', 'weekday', 'month', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap', 'price_level', 'price_change_7', 'price_pct_change_7']
Missing values: 0
Panel shape: (825229, 28)


In [91]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor


RANDOM_SEED = 2026
FORECAST_HORIZON = 28


def fit_feature_encoder(training_frame):

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32
    )

    encoder.fit(
        training_frame[CATEGORICAL_FEATURES].astype(str)
    )

    return encoder


def transform_features(frame, encoder):

    numeric_features = frame[
        NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    categorical_features = encoder.transform(
        frame[CATEGORICAL_FEATURES].astype(str)
    ).astype(np.float32)

    features = np.column_stack([
        numeric_features,
        categorical_features
    ])

    if not np.isfinite(features).all():
        raise ValueError(
            "The XGBoost feature matrix contains missing or infinite values."
        )

    return features


def fit_xgboost_at_origin(
    panel,
    origin,
    configuration
):

    # Only observations available by the forecast origin are used.
    training_frame = panel[
        panel["target_day"] <= origin
    ].copy()

    encoder = fit_feature_encoder(training_frame)

    x_train = transform_features(
        training_frame,
        encoder
    )

    y_train = training_frame[
        "target"
    ].to_numpy(dtype=np.float32)

    model = XGBRegressor(
        n_estimators=int(configuration["n_estimators"]),
        learning_rate=float(configuration["learning_rate"]),
        max_depth=int(configuration["max_depth"]),
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        x_train,
        y_train,
        verbose=False
    )

    return model, encoder, len(training_frame)


In [93]:
def recursive_xgboost_forecast(
    model,
    encoder,
    selected_sales,
    day_columns,
    calendar,
    price_matrix,
    origin
):

    demand_values = selected_sales[
        day_columns
    ].to_numpy(dtype=float)

    number_of_series = len(selected_sales)

    # Actual demand is entered only through the forecast origin.
    history = np.full(
        (
            number_of_series,
            origin + FORECAST_HORIZON
        ),
        np.nan,
        dtype=float
    )

    history[:, :origin] = demand_values[:, :origin]

    metadata = selected_sales.reset_index(drop=True)
    calendar_by_day = calendar.set_index("day_number")

    # Use only the price known at the forecast origin.
    origin_index = origin - 1

    known_price = price_matrix[
        :,
        origin_index
    ]

    earlier_price = price_matrix[
        :,
        origin_index - 7
    ]

    price_change, price_pct_change = safe_price_features(
        known_price,
        earlier_price
    )

    # Forecast one day at a time.
    for horizon in range(1, FORECAST_HORIZON + 1):

        target_day = origin + horizon
        target_index = target_day - 1

        calendar_row = calendar_by_day.loc[target_day]

        forecast_frame = metadata[
            [
                "item_id",
                "cat_id",
                "dept_id",
                "store_id",
                "state_id"
            ]
        ].copy()

        # Lagged demand features
        for lag in LAGS:
            forecast_frame[f"lag_{lag}"] = (
                history[:, target_index - lag]
            )

        # Rolling features
        for window in ROLLING_WINDOWS:

            rolling_values = history[
                :,
                target_index - window:target_index
            ]

            forecast_frame[f"rolling_mean_{window}"] = (
                rolling_values.mean(axis=1)
            )

            forecast_frame[f"rolling_sd_{window}"] = (
                rolling_values.std(axis=1, ddof=0)
            )

            forecast_frame[f"zero_proportion_{window}"] = (
                rolling_values == 0
            ).mean(axis=1)

        # Calendar variables
        forecast_frame["weekday"] = float(
            calendar_row["weekday"]
        )

        forecast_frame["month"] = float(
            calendar_row["month"]
        )

        for event_column in [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]:
            forecast_frame[event_column] = str(
                calendar_row[event_column]
            )

        # State-specific SNAP indicator
        forecast_frame["snap"] = [
            float(calendar_row[f"snap_{state}"])
            for state in metadata["state_id"]
        ]

        # The last known price is held constant over the forecast horizon.
        forecast_frame["price_level"] = np.nan_to_num(
            known_price,
            nan=0.0
        )

        forecast_frame["price_change_7"] = price_change
        forecast_frame["price_pct_change_7"] = price_pct_change

        x_future = transform_features(
            forecast_frame,
            encoder
        )

        daily_prediction = model.predict(x_future)
        daily_prediction = np.asarray(
            daily_prediction,
            dtype=float
        )

        # Demand forecasts cannot be negative.
        daily_prediction = np.maximum(
            daily_prediction,
            0.0
        )

        if not np.isfinite(daily_prediction).all():
            raise ValueError(
                f"Invalid predictions at forecast horizon {horizon}."
            )

        # The prediction becomes part of the history for the next day.
        history[:, target_index] = daily_prediction

    forecasts = history[
        :,
        origin:origin + FORECAST_HORIZON
    ]

    return forecasts


In [ ]:
test_configuration = {
    "configuration_id": "test",
    "n_estimators": 300,
    "learning_rate": 0.03,
    "max_depth": 6
}

xgb_model, xgb_encoder, training_rows = fit_xgboost_at_origin(
    panel=xgboost_panel,
    origin=1829,
    configuration=test_configuration
)

print("XGBoost model fitted.")
print("Training rows:", training_rows)

xgb_test = recursive_xgboost_forecast(
    model=xgb_model,
    encoder=xgb_encoder,
    selected_sales=selected_sales,
    day_columns=day_columns,
    calendar=calendar,
    price_matrix=price_matrix,
    origin=1829
)

print("XGBoost forecast matrix:", xgb_test.shape)
print("Missing forecasts:", np.isnan(xgb_test).sum())
print("Minimum forecast:", xgb_test.min())
print("Maximum forecast:", xgb_test.max())


Feed the data to TiRex-2

Calendar Covariates

In [3]:
import numpy as np
import pandas as pd
import torch

from tirex2 import TimeseriesType


FORECAST_HORIZON = 28


def calendar_covariate_matrix(
    calendar_slice,
    state_id
):

    # Weekly seasonality represented by sine and cosine.
    weekday = calendar_slice[
        "weekday"
    ].to_numpy(dtype=float)

    weekday_angle = 2 * np.pi * weekday / 7

    covariates = [
        np.sin(weekday_angle),
        np.cos(weekday_angle)
    ]

    # Indicators showing whether an event occurs.
    event_name_1 = (
        calendar_slice["event_name_1"]
        .fillna("none")
        .astype(str)
    )

    event_name_2 = (
        calendar_slice["event_name_2"]
        .fillna("none")
        .astype(str)
    )

    covariates.append(
        (event_name_1 != "none").to_numpy(dtype=float)
    )

    covariates.append(
        (event_name_2 != "none").to_numpy(dtype=float)
    )

    # Binary indicators for the different event types.
    event_type_1 = (
        calendar_slice["event_type_1"]
        .fillna("none")
        .astype(str)
    )

    event_type_2 = (
        calendar_slice["event_type_2"]
        .fillna("none")
        .astype(str)
    )

    event_types = sorted(
        (
            set(event_type_1)
            | set(event_type_2)
        )
        - {"none"}
    )

    for event_type in event_types:

        event_indicator = (
            (event_type_1 == event_type)
            | (event_type_2 == event_type)
        )

        covariates.append(
            event_indicator.to_numpy(dtype=float)
        )

    # State-specific SNAP indicator.
    snap_column = f"snap_{state_id}"

    if snap_column not in calendar_slice.columns:
        raise ValueError(
            f"The calendar does not contain {snap_column}."
        )

    covariates.append(
        calendar_slice[snap_column].to_numpy(dtype=float)
    )

    return np.vstack(covariates).astype(np.float32)


Construct the TiRex-2 inputs

In [5]:
def build_tirex_timeseries(
    selected_sales,
    day_columns,
    calendar,
    price_matrix,
    origin
):

    demand_values = selected_sales[
        day_columns
    ].to_numpy(dtype=float)

    tirex_timeseries = []

    metadata = selected_sales.reset_index(drop=True)

    for series_index, row in metadata.iterrows():

        active_start_index = (
            int(row["first_active_day"]) - 1
        )

        # Demand is included only through the forecast origin.
        target_history = demand_values[
            series_index,
            active_start_index:origin
        ]

        # Prices are past covariates only.
        past_prices = pd.Series(
            price_matrix[
                series_index,
                active_start_index:origin
            ]
        ).ffill().bfill()

        if past_prices.isna().any():
            raise ValueError(
                f"Historical prices are missing for {row['id']}."
            )

        # Calendar variables include the historical period and the
        # following 28 days because they are known in advance.
        calendar_slice = calendar.iloc[
            active_start_index:
            origin + FORECAST_HORIZON
        ].copy()

        future_known_covariates = calendar_covariate_matrix(
            calendar_slice=calendar_slice,
            state_id=str(row["state_id"])
        )

        tirex_series = TimeseriesType(
            target=torch.tensor(
                target_history,
                dtype=torch.float32
            ).unsqueeze(0),

            past_covariates=torch.tensor(
                past_prices.to_numpy(dtype=np.float32),
                dtype=torch.float32
            ).unsqueeze(0),

            future_covariates=torch.tensor(
                future_known_covariates,
                dtype=torch.float32
            )
        )

        tirex_timeseries.append(tirex_series)

    return tirex_timeseries


Device Selection

In [9]:
def resolve_device(requested_device="auto"):

    if requested_device != "auto":
        return requested_device

    if torch.cuda.is_available():
        return "cuda"

    if torch.backends.mps.is_available():
        return "mps"

    return "cpu"


Rerun the input construction

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. File locations
# --------------------------------------------------

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))

SALES_FILE = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_FILE = DATA_DIR / "calendar.csv"
PRICES_FILE = DATA_DIR / "sell_prices.csv"
SAMPLE_FILE = DATA_DIR / "selected_sample_ids.csv"

OUTPUT_DIR = DATA_DIR / "chapter5_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


required_files = [
    SALES_FILE,
    CALENDAR_FILE,
    PRICES_FILE,
    SAMPLE_FILE
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(f"File not found: {file}")

print("All required files were found.")


# --------------------------------------------------
# 2. Load sales and selected sample
# --------------------------------------------------

sales = pd.read_csv(SALES_FILE)
sample = pd.read_csv(SAMPLE_FILE)

if "demand_class" not in sample.columns:

    if "demand_pattern" in sample.columns:
        sample = sample.rename(
            columns={"demand_pattern": "demand_class"}
        )

    else:
        raise ValueError(
            "The sample file must contain demand_class or demand_pattern."
        )


class_names = {
    "smooth": "Smooth",
    "erratic": "Erratic",
    "intermittent": "Intermittent",
    "lumpy": "Lumpy"
}

sample["demand_class"] = (
    sample["demand_class"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(class_names)
)

if sample["demand_class"].isna().any():
    raise ValueError("An unknown demand class was found.")

if sample["id"].duplicated().any():
    raise ValueError("The selected sample contains duplicated IDs.")


sample_for_merge = (
    sample[["id", "demand_class"]]
    .sort_values("id")
    .reset_index(drop=True)
)


selected_sales = sample_for_merge.merge(
    sales,
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True
)


missing_ids = selected_sales.loc[
    selected_sales["_merge"] != "both",
    "id"
]

if len(missing_ids) > 0:
    raise ValueError(
        f"Selected IDs not found in sales data: "
        f"{missing_ids.head().tolist()}"
    )

selected_sales = selected_sales.drop(columns="_merge")


# --------------------------------------------------
# 3. Identify demand columns
# --------------------------------------------------

def day_number(column_name):
    return int(column_name.replace("d_", ""))


day_columns = sorted(
    [
        column
        for column in sales.columns
        if column.startswith("d_")
    ],
    key=day_number
)

if len(day_columns) < 1941:
    raise ValueError(
        "The sales data must contain demand through d_1941."
    )


demand_matrix = selected_sales[
    day_columns
].to_numpy(dtype=float)


# --------------------------------------------------
# 4. Load and prepare calendar
# --------------------------------------------------

calendar = pd.read_csv(CALENDAR_FILE)

calendar["day_number"] = (
    calendar["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

calendar["date"] = pd.to_datetime(calendar["date"])
calendar["weekday"] = calendar["date"].dt.dayofweek
calendar["month"] = calendar["date"].dt.month


event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for column in event_columns:
    calendar[column] = (
        calendar[column]
        .fillna("none")
        .astype(str)
    )


calendar = (
    calendar
    .sort_values("day_number")
    .reset_index(drop=True)
)


# --------------------------------------------------
# 5. Load prices and determine active periods
# --------------------------------------------------

prices = pd.read_csv(PRICES_FILE)

valid_prices = prices[
    prices["sell_price"].notna()
    & (prices["sell_price"] > 0)
].copy()


first_price_week = (
    valid_prices
    .groupby(
        ["store_id", "item_id"],
        as_index=False
    )["wm_yr_wk"]
    .min()
    .rename(
        columns={"wm_yr_wk": "first_price_week"}
    )
)


first_day_of_week = (
    calendar
    .groupby("wm_yr_wk")["day_number"]
    .min()
)


first_price_week["first_active_day"] = (
    first_price_week["first_price_week"]
    .map(first_day_of_week)
)


selected_sales = selected_sales.merge(
    first_price_week[
        [
            "store_id",
            "item_id",
            "first_price_week",
            "first_active_day"
        ]
    ],
    on=["store_id", "item_id"],
    how="left",
    validate="one_to_one"
)


if selected_sales["first_active_day"].isna().any():

    missing_active_periods = selected_sales.loc[
        selected_sales["first_active_day"].isna(),
        "id"
    ]

    raise ValueError(
        "No valid active period was found for: "
        f"{missing_active_periods.head().tolist()}"
    )


selected_sales["first_active_day"] = (
    selected_sales["first_active_day"].astype(int)
)


# --------------------------------------------------
# 6. Construct daily price matrix
# --------------------------------------------------

number_of_series = len(selected_sales)
number_of_days = len(day_columns)

price_matrix = np.full(
    (
        number_of_series,
        number_of_days
    ),
    np.nan,
    dtype=float
)


weeks_by_day = calendar.loc[
    :number_of_days - 1,
    "wm_yr_wk"
].to_numpy()


price_groups = {
    key: group.set_index("wm_yr_wk")["sell_price"]
    for key, group in prices.groupby(
        ["store_id", "item_id"]
    )
}


for series_index, row in selected_sales.iterrows():

    key = (
        row["store_id"],
        row["item_id"]
    )

    if key not in price_groups:
        continue

    daily_prices = (
        pd.Series(weeks_by_day)
        .map(price_groups[key])
        .astype(float)
        .ffill()
    )

    price_matrix[series_index] = (
        daily_prices.to_numpy()
    )


# --------------------------------------------------
# 7. Final checks
# --------------------------------------------------

class_counts = (
    selected_sales["demand_class"]
    .value_counts()
    .sort_index()
)

assert len(selected_sales) == 500
assert demand_matrix.shape == (500, 1941)
assert price_matrix.shape == (500, 1941)
assert selected_sales["first_active_day"].notna().all()


print("\nData preparation completed successfully.")
print("Selected series:", len(selected_sales))
print("Demand matrix:", demand_matrix.shape)
print("Price matrix:", price_matrix.shape)
print("\nDemand classes:")
print(class_counts)


All required files were found.


/var/folders/qg/nvrwk7_51bngv6xjz8w6p5f40000gn/T/ipykernel_10537/2492210997.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  selected_sales = sample_for_merge.merge(
/var/folders/qg/nvrwk7_51bngv6xjz8w6p5f40000gn/T/ipykernel_10537/2492210997.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  selected_sales = sample_for_merge.merge(



Data preparation completed successfully.
Selected series: 500
Demand matrix: (500, 1941)
Price matrix: (500, 1941)

Demand classes:
demand_class
Erratic         125
Intermittent    125
Lumpy           125
Smooth          125
Name: count, dtype: int64


In [16]:
tirex_inputs = build_tirex_timeseries(
    selected_sales=selected_sales,
    day_columns=day_columns,
    calendar=calendar,
    price_matrix=price_matrix,
    origin=1829
)

print("Number of TiRex-2 inputs:", len(tirex_inputs))


Number of TiRex-2 inputs: 500


Run TiRex-2

In [19]:
from tirex2 import load_model

device = resolve_device("auto")

print("Selected device:", device)

tirex_model = load_model(
    "NX-AI/TiRex-2",
    device=device
)

tirex_test = tirex_model.forecast(
    tirex_inputs,
    prediction_length=28,
    output_type="numpy",
    batch_size=16
)

print("Number of forecasted series:", len(tirex_test))
print(
    "First output shape:",
    np.asarray(tirex_test[0]).shape
)


Selected device: mps


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

W0825 15:20:03.724000 10537 site-packages/torch/_inductor/utils.py:1953] [0/0] Not enough SMs to use max_autotune_gemm mode


Number of forecasted series: 500
First output shape: (1, 9, 28)


Save the results

In [22]:
ORIGIN = 1829
HORIZON = 28

quantile_levels = np.asarray(
    tirex_model._quantile_levels(),
    dtype=float
)

# Convert the list of outputs into:
# 500 series × 9 quantiles × 28 days
tirex_quantiles = np.stack(
    [
        np.asarray(output, dtype=float)[0]
        for output in tirex_test
    ],
    axis=0
)

# Demand forecasts cannot be negative.
tirex_quantiles = np.maximum(
    tirex_quantiles,
    0.0
)

expected_shape = (
    len(selected_sales),
    9,
    HORIZON
)

if tirex_quantiles.shape != expected_shape:
    raise ValueError(
        f"Unexpected shape: {tirex_quantiles.shape}"
    )

# Check for quantile crossings.
quantile_crossings = np.sum(
    np.diff(tirex_quantiles, axis=1) < -1e-8
)

print("Quantile levels:", quantile_levels)
print("Forecast array:", tirex_quantiles.shape)
print("Quantile crossings:", quantile_crossings)

# Save the complete array.
np.save(
    OUTPUT_DIR / "tirex2_quantiles_origin_1829.npy",
    tirex_quantiles
)

print("Raw TiRex-2 forecasts saved.")


Quantile levels: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
Forecast array: (500, 9, 28)
Quantile crossings: 779
Raw TiRex-2 forecasts saved.


In [24]:
median_index = int(
    np.where(
        np.isclose(quantile_levels, 0.50)
    )[0][0]
)

tirex_median = tirex_quantiles[
    :,
    median_index,
    :
]

actual_demand = demand_matrix[
    :,
    ORIGIN:ORIGIN + HORIZON
]

central_forecasts = pd.DataFrame({
    "id": np.repeat(
        selected_sales["id"].to_numpy(),
        HORIZON
    ),
    "demand_class": np.repeat(
        selected_sales["demand_class"].to_numpy(),
        HORIZON
    ),
    "model": "tirex2",
    "purpose": "validation",
    "round": 1,
    "forecast_origin": ORIGIN,
    "horizon": np.tile(
        np.arange(1, HORIZON + 1),
        len(selected_sales)
    ),
    "target_day": np.tile(
        ORIGIN + np.arange(1, HORIZON + 1),
        len(selected_sales)
    ),
    "actual": actual_demand.reshape(-1),
    "forecast": tirex_median.reshape(-1)
})

central_forecasts.to_csv(
    OUTPUT_DIR / "tirex2_central_forecasts_origin_1829.csv",
    index=False
)

print("Central forecasts saved:", central_forecasts.shape)


Central forecasts saved: (14000, 10)


In [26]:
import gc
import numpy as np
import pandas as pd
import torch


remaining_origins = [
    {"origin": 1661, "purpose": "calibration", "round": 1},
    {"origin": 1689, "purpose": "calibration", "round": 2},
    {"origin": 1717, "purpose": "calibration", "round": 3},
    {"origin": 1745, "purpose": "calibration", "round": 4},
    {"origin": 1773, "purpose": "calibration", "round": 5},
    {"origin": 1801, "purpose": "calibration", "round": 6},
    {"origin": 1857, "purpose": "validation", "round": 2},
    {"origin": 1885, "purpose": "validation", "round": 3},
    {"origin": 1913, "purpose": "final_test", "round": 1}
]

quantile_levels = np.asarray(
    tirex_model._quantile_levels(),
    dtype=float
)

median_index = int(
    np.where(
        np.isclose(quantile_levels, 0.50)
    )[0][0]
)


for schedule_row in remaining_origins:

    origin = schedule_row["origin"]
    purpose = schedule_row["purpose"]
    round_number = schedule_row["round"]

    quantile_file = (
        OUTPUT_DIR
        / f"tirex2_quantiles_origin_{origin}.npy"
    )

    central_file = (
        OUTPUT_DIR
        / f"tirex2_central_forecasts_origin_{origin}.csv"
    )

    # Skip an origin if both output files already exist.
    if quantile_file.exists() and central_file.exists():
        print(f"Origin d_{origin} already completed. Skipping.")
        continue

    print(f"\nStarting TiRex-2 at origin d_{origin}...")

    model_inputs = build_tirex_timeseries(
        selected_sales=selected_sales,
        day_columns=day_columns,
        calendar=calendar,
        price_matrix=price_matrix,
        origin=origin
    )

    model_outputs = tirex_model.forecast(
        model_inputs,
        prediction_length=28,
        output_type="numpy",
        batch_size=16
    )

    quantile_forecasts = np.stack(
        [
            np.asarray(output, dtype=float)[0]
            for output in model_outputs
        ],
        axis=0
    )

    quantile_forecasts = np.maximum(
        quantile_forecasts,
        0.0
    )

    expected_shape = (
        len(selected_sales),
        9,
        28
    )

    if quantile_forecasts.shape != expected_shape:
        raise ValueError(
            f"Origin d_{origin}: expected {expected_shape}, "
            f"but received {quantile_forecasts.shape}."
        )

    if not np.isfinite(quantile_forecasts).all():
        raise ValueError(
            f"Origin d_{origin} contains invalid forecasts."
        )

    crossing_count = int(
        np.sum(
            np.diff(
                quantile_forecasts,
                axis=1
            ) < -1e-8
        )
    )

    median_forecasts = quantile_forecasts[
        :,
        median_index,
        :
    ]

    actual_demand = demand_matrix[
        :,
        origin:origin + 28
    ]

    central_forecasts = pd.DataFrame({
        "id": np.repeat(
            selected_sales["id"].to_numpy(),
            28
        ),
        "demand_class": np.repeat(
            selected_sales["demand_class"].to_numpy(),
            28
        ),
        "model": "tirex2",
        "purpose": purpose,
        "round": round_number,
        "forecast_origin": origin,
        "horizon": np.tile(
            np.arange(1, 29),
            len(selected_sales)
        ),
        "target_day": np.tile(
            origin + np.arange(1, 29),
            len(selected_sales)
        ),
        "actual": actual_demand.reshape(-1),
        "forecast": median_forecasts.reshape(-1)
    })

    # Save immediately after completing the origin.
    np.save(
        quantile_file,
        quantile_forecasts
    )

    central_forecasts.to_csv(
        central_file,
        index=False
    )

    print(f"Completed origin d_{origin}.")
    print("Quantile crossings:", crossing_count)
    print("Saved:", quantile_file.name)
    print("Saved:", central_file.name)

    # Release memory before starting the next origin.
    del model_inputs
    del model_outputs
    del quantile_forecasts
    del median_forecasts
    del central_forecasts

    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


print("\nAll remaining TiRex-2 origins are complete.")



Starting TiRex-2 at origin d_1661...
Completed origin d_1661.
Quantile crossings: 688
Saved: tirex2_quantiles_origin_1661.npy
Saved: tirex2_central_forecasts_origin_1661.csv

Starting TiRex-2 at origin d_1689...
Completed origin d_1689.
Quantile crossings: 523
Saved: tirex2_quantiles_origin_1689.npy
Saved: tirex2_central_forecasts_origin_1689.csv

Starting TiRex-2 at origin d_1717...
Completed origin d_1717.
Quantile crossings: 561
Saved: tirex2_quantiles_origin_1717.npy
Saved: tirex2_central_forecasts_origin_1717.csv

Starting TiRex-2 at origin d_1745...
Completed origin d_1745.
Quantile crossings: 698
Saved: tirex2_quantiles_origin_1745.npy
Saved: tirex2_central_forecasts_origin_1745.csv

Starting TiRex-2 at origin d_1773...
Completed origin d_1773.
Quantile crossings: 705
Saved: tirex2_quantiles_origin_1773.npy
Saved: tirex2_central_forecasts_origin_1773.csv

Starting TiRex-2 at origin d_1801...
Completed origin d_1801.
Quantile crossings: 769
Saved: tirex2_quantiles_origin_1801.np

Seasonal Naive Forecasts for all 500 series and all 10 origins

In [29]:
import numpy as np
import pandas as pd


FORECAST_HORIZON = 28
SEASONAL_PERIOD = 7


# --------------------------------------------------
# 1. Forecast schedule
# --------------------------------------------------

forecast_schedule = pd.DataFrame({
    "forecast_origin": [
        1661, 1689, 1717, 1745, 1773, 1801,
        1829, 1857, 1885,
        1913
    ],
    "purpose": [
        "calibration", "calibration", "calibration",
        "calibration", "calibration", "calibration",
        "validation", "validation", "validation",
        "final_test"
    ],
    "round": [
        1, 2, 3, 4, 5, 6,
        1, 2, 3,
        1
    ]
})

forecast_schedule["forecast_start"] = (
    forecast_schedule["forecast_origin"] + 1
)

forecast_schedule["forecast_end"] = (
    forecast_schedule["forecast_origin"]
    + FORECAST_HORIZON
)


# --------------------------------------------------
# 2. Generate seasonal-naïve forecasts
# --------------------------------------------------

forecast_parts = []

number_of_series = len(selected_sales)
horizons = np.arange(1, FORECAST_HORIZON + 1)

series_ids = selected_sales["id"].to_numpy()
demand_classes = selected_sales["demand_class"].to_numpy()


for schedule_row in forecast_schedule.itertuples(index=False):

    origin = int(schedule_row.forecast_origin)

    # Training data end exactly at the forecast origin.
    training_demand = demand_matrix[:, :origin]

    # Retain the last observed seven days.
    final_week = training_demand[
        :,
        -SEASONAL_PERIOD:
    ]

    # Repeat the final week four times.
    forecasts = np.tile(
        final_week,
        (
            1,
            FORECAST_HORIZON // SEASONAL_PERIOD
        )
    )

    forecasts = np.maximum(
        forecasts,
        0.0
    )

    # Actual demand is used only for evaluation.
    actual_demand = demand_matrix[
        :,
        origin:origin + FORECAST_HORIZON
    ]

    origin_results = pd.DataFrame({
        "id": np.repeat(
            series_ids,
            FORECAST_HORIZON
        ),
        "demand_class": np.repeat(
            demand_classes,
            FORECAST_HORIZON
        ),
        "model": "seasonal_naive",
        "purpose": schedule_row.purpose,
        "round": int(schedule_row.round),
        "forecast_origin": origin,
        "forecast_origin_label": f"d_{origin}",
        "horizon": np.tile(
            horizons,
            number_of_series
        ),
        "target_day": np.tile(
            origin + horizons,
            number_of_series
        ),
        "actual": actual_demand.reshape(-1),
        "forecast": forecasts.reshape(-1)
    })

    origin_results["target_day_label"] = (
        "d_"
        + origin_results["target_day"].astype(str)
    )

    forecast_parts.append(origin_results)

    print(f"Completed seasonal-naïve origin d_{origin}.")


seasonal_naive_forecasts = pd.concat(
    forecast_parts,
    ignore_index=True
)


Completed seasonal-naïve origin d_1661.
Completed seasonal-naïve origin d_1689.
Completed seasonal-naïve origin d_1717.
Completed seasonal-naïve origin d_1745.
Completed seasonal-naïve origin d_1773.
Completed seasonal-naïve origin d_1801.
Completed seasonal-naïve origin d_1829.
Completed seasonal-naïve origin d_1857.
Completed seasonal-naïve origin d_1885.
Completed seasonal-naïve origin d_1913.


Calculate the 28-days forecast errors

In [32]:
group_columns = [
    "id",
    "demand_class",
    "model",
    "purpose",
    "round",
    "forecast_origin"
]

seasonal_naive_errors = (
    seasonal_naive_forecasts
    .groupby(
        group_columns,
        as_index=False
    )
    .agg(
        actual_protection_demand=("actual", "sum"),
        forecast_protection_demand=("forecast", "sum"),
        forecast_days=("horizon", "count")
    )
)

seasonal_naive_errors["protection_period_error"] = (
    seasonal_naive_errors["actual_protection_demand"]
    - seasonal_naive_errors["forecast_protection_demand"]
)


Implementation checks

In [35]:
expected_rows = (
    500
    * 10
    * FORECAST_HORIZON
)

seasonal_naive_checks = pd.DataFrame({
    "check": [
        "Expected number of rows",
        "Exactly 500 series",
        "No missing forecasts",
        "All forecasts are finite",
        "All forecasts are non-negative",
        "28 forecasts per series and origin",
        "Target day equals origin plus horizon"
    ],
    "passed": [
        len(seasonal_naive_forecasts) == expected_rows,

        seasonal_naive_forecasts[
            "id"
        ].nunique() == 500,

        seasonal_naive_forecasts[
            "forecast"
        ].notna().all(),

        np.isfinite(
            seasonal_naive_forecasts["forecast"]
        ).all(),

        (
            seasonal_naive_forecasts["forecast"] >= 0
        ).all(),

        seasonal_naive_forecasts.groupby(
            ["id", "forecast_origin"]
        ).size().eq(28).all(),

        (
            seasonal_naive_forecasts["target_day"]
            == seasonal_naive_forecasts["forecast_origin"]
            + seasonal_naive_forecasts["horizon"]
        ).all()
    ]
})

print(seasonal_naive_checks)

if not seasonal_naive_checks["passed"].all():
    raise AssertionError(
        "At least one seasonal-naïve check failed."
    )


                                   check  passed
0                Expected number of rows    True
1                     Exactly 500 series    True
2                   No missing forecasts    True
3               All forecasts are finite    True
4         All forecasts are non-negative    True
5     28 forecasts per series and origin    True
6  Target day equals origin plus horizon    True


In [37]:
seasonal_naive_forecasts.to_csv(
    OUTPUT_DIR / "seasonal_naive_forecasts.csv",
    index=False
)

seasonal_naive_errors.to_csv(
    OUTPUT_DIR
    / "seasonal_naive_protection_period_errors.csv",
    index=False
)

seasonal_naive_checks.to_csv(
    OUTPUT_DIR
    / "seasonal_naive_implementation_checks.csv",
    index=False
)

forecast_schedule.to_csv(
    OUTPUT_DIR / "forecast_schedule.csv",
    index=False
)

print("\nSeasonal-naïve implementation completed.")
print("Forecast rows:", len(seasonal_naive_forecasts))
print("Protection-period error rows:", len(seasonal_naive_errors))
print("Results saved in:", OUTPUT_DIR)



Seasonal-naïve implementation completed.
Forecast rows: 140000
Protection-period error rows: 5000
Results saved in: <M5_DATA_DIR>/chapter5_results


Forecasts for ETS and ARIMA/SARIMA for 500 series at 10 origins

In [41]:
import json
import time
import warnings

import numpy as np
import pandas as pd

from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from pmdarima.arima import auto_arima


FORECAST_HORIZON = 28
SEASONAL_PERIOD = 7


Seasonal naive fallback: If an individual ETS or ARIMA fit fails, the code uses seasonal naïve for that particular series and records the fallback.

In [44]:
def seasonal_naive_fallback(
    training_values,
    horizon=28
):

    final_week = np.asarray(
        training_values[-7:],
        dtype=float
    )

    repetitions = int(
        np.ceil(horizon / 7)
    )

    forecast = np.tile(
        final_week,
        repetitions
    )[:horizon]

    return np.maximum(forecast, 0.0)


Automatic ETS

In [47]:
def fit_automatic_ets(
    training_values,
    horizon=28
):

    candidates = [
        {
            "trend": None,
            "damped": False,
            "name": "no_trend"
        },
        {
            "trend": "add",
            "damped": False,
            "name": "additive_trend"
        },
        {
            "trend": "add",
            "damped": True,
            "name": "damped_additive_trend"
        }
    ]

    successful_candidates = []
    candidate_failures = []

    for candidate in candidates:

        try:
            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore",
                    ConvergenceWarning
                )

                model = ETSModel(
                    training_values,
                    error="add",
                    trend=candidate["trend"],
                    damped_trend=candidate["damped"],
                    seasonal="add",
                    seasonal_periods=7,
                    initialization_method="estimated"
                )

                fitted_model = model.fit(
                    maxiter=1000,
                    disp=False
                )

            convergence_information = getattr(
                fitted_model,
                "mle_retvals",
                {}
            ) or {}

            converged = bool(
                convergence_information.get(
                    "converged",
                    False
                )
            )

            aicc = float(fitted_model.aicc)

            if np.isfinite(aicc):
                successful_candidates.append({
                    "model": fitted_model,
                    "name": candidate["name"],
                    "trend": candidate["trend"],
                    "damped": candidate["damped"],
                    "aicc": aicc,
                    "converged": converged
                })

        except Exception as error:
            candidate_failures.append(
                f"{candidate['name']}: "
                f"{type(error).__name__}: {error}"
            )

    if len(successful_candidates) == 0:
        raise RuntimeError(
            "All ETS candidates failed. "
            + " | ".join(candidate_failures)
        )

    converged_candidates = [
        candidate
        for candidate in successful_candidates
        if candidate["converged"]
    ]

    if len(converged_candidates) > 0:
        eligible_candidates = converged_candidates
    else:
        eligible_candidates = successful_candidates

    selected_candidate = min(
        eligible_candidates,
        key=lambda candidate: candidate["aicc"]
    )

    forecast = np.asarray(
        selected_candidate["model"].forecast(horizon),
        dtype=float
    )

    forecast = np.maximum(forecast, 0.0)

    diagnostics = {
        "selected_specification":
            selected_candidate["name"],

        "trend":
            str(selected_candidate["trend"]),

        "damped_trend":
            selected_candidate["damped"],

        "aicc":
            selected_candidate["aicc"],

        "converged":
            selected_candidate["converged"],

        "used_nonconverged_candidate":
            len(converged_candidates) == 0,

        "candidate_failures":
            " | ".join(candidate_failures)
    }

    return forecast, diagnostics


Automatic ARIMA/SARIMA

In [50]:
def fit_automatic_arima(
    training_values,
    horizon=28
):

    model = auto_arima(
        training_values,

        start_p=1,
        start_q=1,
        max_p=3,
        max_q=3,

        d=None,
        max_d=2,

        start_P=0,
        start_Q=0,
        max_P=1,
        max_Q=1,

        D=None,
        max_D=1,

        max_order=None,

        m=7,
        seasonal=True,

        information_criterion="aicc",

        test="kpss",
        seasonal_test="ocsb",

        stepwise=True,
        n_jobs=1,

        method="lbfgs",
        maxiter=200,

        suppress_warnings=True,
        error_action="raise",
        trace=False,

        random=False,
        with_intercept="auto",

        sarimax_kwargs={
            "simple_differencing": False,
            "enforce_stationarity": False,
            "enforce_invertibility": False
        }
    )

    forecast = np.asarray(
        model.predict(n_periods=horizon),
        dtype=float
    )

    forecast = np.maximum(forecast, 0.0)

    fitted_result = model.arima_res_

    convergence_information = getattr(
        fitted_result,
        "mle_retvals",
        {}
    ) or {}

    diagnostics = {
        "order": json.dumps(
            list(model.order)
        ),

        "seasonal_order": json.dumps(
            list(model.seasonal_order)
        ),

        "aicc": float(model.aicc()),

        "converged": bool(
            convergence_information.get(
                "converged",
                False
            )
        ),

        "stepwise_search": True,
        "likelihood_approximation": False
    }

    return forecast, diagnostics


Resumable production runner

In [53]:
def fit_automatic_arima(
    training_values,
    horizon=28
):

    model = auto_arima(
        training_values,

        start_p=1,
        start_q=1,
        max_p=3,
        max_q=3,

        d=None,
        max_d=2,

        start_P=0,
        start_Q=0,
        max_P=1,
        max_Q=1,

        D=None,
        max_D=1,

        max_order=None,

        m=7,
        seasonal=True,

        information_criterion="aicc",

        test="kpss",
        seasonal_test="ocsb",

        stepwise=True,
        n_jobs=1,

        method="lbfgs",
        maxiter=200,

        suppress_warnings=True,
        error_action="raise",
        trace=False,

        random=False,
        with_intercept="auto",

        sarimax_kwargs={
            "simple_differencing": False,
            "enforce_stationarity": False,
            "enforce_invertibility": False
        }
    )

    forecast = np.asarray(
        model.predict(n_periods=horizon),
        dtype=float
    )

    forecast = np.maximum(forecast, 0.0)

    fitted_result = model.arima_res_

    convergence_information = getattr(
        fitted_result,
        "mle_retvals",
        {}
    ) or {}

    diagnostics = {
        "order": json.dumps(
            list(model.order)
        ),

        "seasonal_order": json.dumps(
            list(model.seasonal_order)
        ),

        "aicc": float(model.aicc()),

        "converged": bool(
            convergence_information.get(
                "converged",
                False
            )
        ),

        "stepwise_search": True,
        "likelihood_approximation": False
    }

    return forecast, diagnostics


Run ETS

In [58]:
required_objects = [
    "selected_sales",
    "demand_matrix",
    "forecast_schedule",
    "OUTPUT_DIR",
    "seasonal_naive_fallback",
    "fit_automatic_ets",
    "run_classical_model"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

print("Missing objects:", missing_objects)


Missing objects: ['run_classical_model']


In [60]:
def run_classical_model(
    model_name,
    fitting_function,
    checkpoint_interval=25
):

    all_origin_forecasts = []
    all_origin_diagnostics = []

    horizons = np.arange(
        1,
        FORECAST_HORIZON + 1
    )

    for schedule_row in forecast_schedule.itertuples(index=False):

        origin = int(schedule_row.forecast_origin)
        purpose = str(schedule_row.purpose)
        round_number = int(schedule_row.round)

        final_forecast_file = (
            OUTPUT_DIR
            / f"{model_name}_forecasts_origin_{origin}.csv"
        )

        final_diagnostic_file = (
            OUTPUT_DIR
            / f"{model_name}_diagnostics_origin_{origin}.csv"
        )

        partial_forecast_file = (
            OUTPUT_DIR
            / f"{model_name}_forecasts_origin_{origin}_partial.csv"
        )

        partial_diagnostic_file = (
            OUTPUT_DIR
            / f"{model_name}_diagnostics_origin_{origin}_partial.csv"
        )

        # Skip a completely finished origin.
        if (
            final_forecast_file.exists()
            and final_diagnostic_file.exists()
        ):
            existing_final = pd.read_csv(
                final_forecast_file
            )

            if len(existing_final) == len(selected_sales) * 28:
                print(
                    f"{model_name}: origin d_{origin} "
                    "already completed. Skipping."
                )

                all_origin_forecasts.append(existing_final)

                all_origin_diagnostics.append(
                    pd.read_csv(final_diagnostic_file)
                )

                continue

        # Load an incomplete checkpoint if one exists.
        if partial_forecast_file.exists():

            saved_forecasts = pd.read_csv(
                partial_forecast_file
            )

            completed_counts = (
                saved_forecasts
                .groupby("id")
                .size()
            )

            completed_ids = set(
                completed_counts[
                    completed_counts == 28
                ].index
            )

            print(
                f"{model_name}, d_{origin}: resuming after "
                f"{len(completed_ids)} completed series."
            )

        else:
            saved_forecasts = pd.DataFrame()
            completed_ids = set()

        if partial_diagnostic_file.exists():
            saved_diagnostics = pd.read_csv(
                partial_diagnostic_file
            )
        else:
            saved_diagnostics = pd.DataFrame()

        new_forecast_parts = []
        new_diagnostic_rows = []
        completed_since_save = 0

        for series_index, series_row in (
            selected_sales
            .reset_index(drop=True)
            .iterrows()
        ):

            series_id = series_row["id"]

            if series_id in completed_ids:
                continue

            started = time.perf_counter()

            active_start_index = (
                int(series_row["first_active_day"]) - 1
            )

            training_demand = demand_matrix[
                series_index,
                active_start_index:origin
            ]

            actual_demand = demand_matrix[
                series_index,
                origin:origin + FORECAST_HORIZON
            ]

            fallback_used = False
            error_message = ""

            try:
                forecast, diagnostics = fitting_function(
                    training_demand,
                    FORECAST_HORIZON
                )

                if len(forecast) != FORECAST_HORIZON:
                    raise ValueError(
                        "The model did not return 28 forecasts."
                    )

                if not np.isfinite(forecast).all():
                    raise ValueError(
                        "The model produced an invalid forecast."
                    )

            except Exception as error:

                forecast = seasonal_naive_fallback(
                    training_demand,
                    FORECAST_HORIZON
                )

                diagnostics = {}
                fallback_used = True

                error_message = (
                    f"{type(error).__name__}: {error}"
                )

            forecast_frame = pd.DataFrame({
                "id": series_id,
                "demand_class":
                    series_row["demand_class"],

                "model": model_name,
                "purpose": purpose,
                "round": round_number,

                "forecast_origin": origin,
                "forecast_origin_label": f"d_{origin}",

                "horizon": horizons,
                "target_day": origin + horizons,

                "actual": actual_demand,
                "forecast": np.maximum(
                    forecast,
                    0.0
                )
            })

            forecast_frame["target_day_label"] = (
                "d_"
                + forecast_frame["target_day"].astype(str)
            )

            new_forecast_parts.append(
                forecast_frame
            )

            new_diagnostic_rows.append({
                "id": series_id,
                "demand_class":
                    series_row["demand_class"],

                "model": model_name,
                "forecast_origin": origin,

                "active_training_start":
                    active_start_index + 1,

                "training_observations":
                    len(training_demand),

                "fit_seconds":
                    time.perf_counter() - started,

                "fallback_used":
                    fallback_used,

                "error_message":
                    error_message,

                **diagnostics
            })

            completed_since_save += 1

            # Save an intermediate checkpoint.
            if completed_since_save >= checkpoint_interval:

                new_forecasts = pd.concat(
                    new_forecast_parts,
                    ignore_index=True
                )

                saved_forecasts = pd.concat(
                    [
                        saved_forecasts,
                        new_forecasts
                    ],
                    ignore_index=True
                )

                saved_forecasts = (
                    saved_forecasts
                    .drop_duplicates(
                        subset=[
                            "id",
                            "forecast_origin",
                            "horizon"
                        ],
                        keep="last"
                    )
                )

                new_diagnostics = pd.DataFrame(
                    new_diagnostic_rows
                )

                saved_diagnostics = pd.concat(
                    [
                        saved_diagnostics,
                        new_diagnostics
                    ],
                    ignore_index=True
                )

                saved_diagnostics = (
                    saved_diagnostics
                    .drop_duplicates(
                        subset=[
                            "id",
                            "forecast_origin",
                            "model"
                        ],
                        keep="last"
                    )
                )

                saved_forecasts.to_csv(
                    partial_forecast_file,
                    index=False
                )

                saved_diagnostics.to_csv(
                    partial_diagnostic_file,
                    index=False
                )

                completed_total = (
                    saved_forecasts["id"].nunique()
                )

                print(
                    f"{model_name}, d_{origin}: "
                    f"{completed_total}/500 series completed."
                )

                new_forecast_parts = []
                new_diagnostic_rows = []
                completed_since_save = 0

        # Save the final incomplete batch.
        if len(new_forecast_parts) > 0:

            saved_forecasts = pd.concat(
                [
                    saved_forecasts,
                    pd.concat(
                        new_forecast_parts,
                        ignore_index=True
                    )
                ],
                ignore_index=True
            )

            saved_diagnostics = pd.concat(
                [
                    saved_diagnostics,
                    pd.DataFrame(new_diagnostic_rows)
                ],
                ignore_index=True
            )

        saved_forecasts = (
            saved_forecasts
            .drop_duplicates(
                subset=[
                    "id",
                    "forecast_origin",
                    "horizon"
                ],
                keep="last"
            )
            .sort_values(
                ["id", "horizon"]
            )
            .reset_index(drop=True)
        )

        saved_diagnostics = (
            saved_diagnostics
            .drop_duplicates(
                subset=[
                    "id",
                    "forecast_origin",
                    "model"
                ],
                keep="last"
            )
            .sort_values("id")
            .reset_index(drop=True)
        )

        expected_origin_rows = (
            len(selected_sales)
            * FORECAST_HORIZON
        )

        if len(saved_forecasts) != expected_origin_rows:
            raise AssertionError(
                f"{model_name}, d_{origin}: expected "
                f"{expected_origin_rows} rows but found "
                f"{len(saved_forecasts)}."
            )

        saved_forecasts.to_csv(
            final_forecast_file,
            index=False
        )

        saved_diagnostics.to_csv(
            final_diagnostic_file,
            index=False
        )

        # The complete origin is retained in memory for consolidation.
        all_origin_forecasts.append(
            saved_forecasts
        )

        all_origin_diagnostics.append(
            saved_diagnostics
        )

        print(
            f"{model_name}: completed origin d_{origin}."
        )

    # Combine all ten origins.
    combined_forecasts = pd.concat(
        all_origin_forecasts,
        ignore_index=True
    )

    combined_diagnostics = pd.concat(
        all_origin_diagnostics,
        ignore_index=True
    )

    group_columns = [
        "id",
        "demand_class",
        "model",
        "purpose",
        "round",
        "forecast_origin"
    ]

    protection_errors = (
        combined_forecasts
        .groupby(
            group_columns,
            as_index=False
        )
        .agg(
            actual_protection_demand=("actual", "sum"),
            forecast_protection_demand=("forecast", "sum"),
            forecast_days=("horizon", "count")
        )
    )

    protection_errors["protection_period_error"] = (
        protection_errors["actual_protection_demand"]
        - protection_errors["forecast_protection_demand"]
    )

    checks = pd.DataFrame({
        "check": [
            "Expected number of forecast rows",
            "Exactly 500 series",
            "Exactly ten forecast origins",
            "No missing forecasts",
            "Finite forecasts",
            "Non-negative forecasts",
            "28 rows per series and origin"
        ],
        "passed": [
            len(combined_forecasts)
            == 500 * 10 * 28,

            combined_forecasts["id"].nunique()
            == 500,

            combined_forecasts[
                "forecast_origin"
            ].nunique() == 10,

            combined_forecasts[
                "forecast"
            ].notna().all(),

            np.isfinite(
                combined_forecasts["forecast"]
            ).all(),

            (
                combined_forecasts["forecast"] >= 0
            ).all(),

            combined_forecasts.groupby(
                ["id", "forecast_origin"]
            ).size().eq(28).all()
        ]
    })

    if not checks["passed"].all():
        raise AssertionError(
            f"{model_name} implementation checks failed."
        )

    combined_forecasts.to_csv(
        OUTPUT_DIR / f"{model_name}_forecasts.csv",
        index=False
    )

    combined_diagnostics.to_csv(
        OUTPUT_DIR / f"{model_name}_fit_diagnostics.csv",
        index=False
    )

    protection_errors.to_csv(
        OUTPUT_DIR
        / f"{model_name}_protection_period_errors.csv",
        index=False
    )

    checks.to_csv(
        OUTPUT_DIR
        / f"{model_name}_implementation_checks.csv",
        index=False
    )

    print(f"\n{model_name} completed successfully.")
    print("Forecast rows:", len(combined_forecasts))
    print(
        "Fallback fits:",
        int(combined_diagnostics["fallback_used"].sum())
    )

    return (
        combined_forecasts,
        combined_diagnostics,
        protection_errors,
        checks
    )


In [62]:
print("run_classical_model" in globals())


True


In [64]:
ets_forecasts, ets_diagnostics, ets_errors, ets_checks = (
    run_classical_model(
        model_name="ets",
        fitting_function=fit_automatic_ets,
        checkpoint_interval=25
    )
)


ets, d_1661: 25/500 series completed.
ets, d_1661: 50/500 series completed.
ets, d_1661: 75/500 series completed.
ets, d_1661: 100/500 series completed.
ets, d_1661: 125/500 series completed.
ets, d_1661: 150/500 series completed.
ets, d_1661: 175/500 series completed.
ets, d_1661: 200/500 series completed.
ets, d_1661: 225/500 series completed.
ets, d_1661: 250/500 series completed.
ets, d_1661: 275/500 series completed.
ets, d_1661: 300/500 series completed.
ets, d_1661: 325/500 series completed.
ets, d_1661: 350/500 series completed.
ets, d_1661: 375/500 series completed.
ets, d_1661: 400/500 series completed.
ets, d_1661: 425/500 series completed.
ets, d_1661: 450/500 series completed.
ets, d_1661: 475/500 series completed.
ets, d_1661: 500/500 series completed.
ets: completed origin d_1661.
ets, d_1689: 25/500 series completed.
ets, d_1689: 50/500 series completed.
ets, d_1689: 75/500 series completed.
ets, d_1689: 100/500 series completed.
ets, d_1689: 125/500 series completed.
e

In [66]:
print(ets_checks)
print(
    ets_diagnostics["fallback_used"].value_counts()
)


                              check  passed
0  Expected number of forecast rows    True
1                Exactly 500 series    True
2      Exactly ten forecast origins    True
3              No missing forecasts    True
4                  Finite forecasts    True
5            Non-negative forecasts    True
6     28 rows per series and origin    True
fallback_used
False    5000
Name: count, dtype: int64


Run ARIMA/SARIMA

In [69]:
(
    arima_forecasts,
    arima_diagnostics,
    arima_errors,
    arima_checks
) = run_classical_model(
    model_name="arima_sarima",
    fitting_function=fit_automatic_arima,
    checkpoint_interval=25
)


arima_sarima, d_1661: 25/500 series completed.
arima_sarima, d_1661: 50/500 series completed.
arima_sarima, d_1661: 75/500 series completed.
arima_sarima, d_1661: 100/500 series completed.
arima_sarima, d_1661: 125/500 series completed.
arima_sarima, d_1661: 150/500 series completed.
arima_sarima, d_1661: 175/500 series completed.
arima_sarima, d_1661: 200/500 series completed.
arima_sarima, d_1661: 225/500 series completed.
arima_sarima, d_1661: 250/500 series completed.
arima_sarima, d_1661: 275/500 series completed.
arima_sarima, d_1661: 300/500 series completed.
arima_sarima, d_1661: 325/500 series completed.
arima_sarima, d_1661: 350/500 series completed.
arima_sarima, d_1661: 375/500 series completed.
arima_sarima, d_1661: 400/500 series completed.
arima_sarima, d_1661: 425/500 series completed.
arima_sarima, d_1661: 450/500 series completed.
arima_sarima, d_1661: 475/500 series completed.
arima_sarima, d_1661: 500/500 series completed.
arima_sarima: completed origin d_1661.
arim

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:1901: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ar)**-1


arima_sarima, d_1829: 500/500 series completed.
arima_sarima: completed origin d_1829.
arima_sarima, d_1857: 25/500 series completed.
arima_sarima, d_1857: 50/500 series completed.
arima_sarima, d_1857: 75/500 series completed.
arima_sarima, d_1857: 100/500 series completed.
arima_sarima, d_1857: 125/500 series completed.
arima_sarima, d_1857: 150/500 series completed.
arima_sarima, d_1857: 175/500 series completed.
arima_sarima, d_1857: 200/500 series completed.
arima_sarima, d_1857: 225/500 series completed.
arima_sarima, d_1857: 250/500 series completed.
arima_sarima, d_1857: 275/500 series completed.
arima_sarima, d_1857: 300/500 series completed.
arima_sarima, d_1857: 325/500 series completed.
arima_sarima, d_1857: 350/500 series completed.
arima_sarima, d_1857: 375/500 series completed.
arima_sarima, d_1857: 400/500 series completed.
arima_sarima, d_1857: 425/500 series completed.
arima_sarima, d_1857: 450/500 series completed.
arima_sarima, d_1857: 475/500 series completed.
arim

In [73]:
print(arima_checks)

print("\nConvergence results:")
print(
    arima_diagnostics["converged"]
    .value_counts(dropna=False)
)

print("\nSaved files:")
print(OUTPUT_DIR / "arima_sarima_forecasts.csv")
print(OUTPUT_DIR / "arima_sarima_fit_diagnostics.csv")
print(OUTPUT_DIR / "arima_sarima_protection_period_errors.csv")
print(OUTPUT_DIR / "arima_sarima_implementation_checks.csv")


                              check  passed
0  Expected number of forecast rows    True
1                Exactly 500 series    True
2      Exactly ten forecast origins    True
3              No missing forecasts    True
4                  Finite forecasts    True
5            Non-negative forecasts    True
6     28 rows per series and origin    True

Convergence results:
converged
True     4971
False      29
Name: count, dtype: int64

Saved files:
<M5_DATA_DIR>/chapter5_results/arima_sarima_forecasts.csv
<M5_DATA_DIR>/chapter5_results/arima_sarima_fit_diagnostics.csv
<M5_DATA_DIR>/chapter5_results/arima_sarima_protection_period_errors.csv
<M5_DATA_DIR>/chapter5_results/arima_sarima_implementation_checks.csv


In [75]:
arima_convergence_summary = pd.DataFrame({
    "measure": [
        "Total fits",
        "Converged fits",
        "Non-converged fits",
        "Fallback fits",
        "Convergence rate (%)",
        "Usable-fit rate (%)",
        "Fallback rate (%)"
    ],
    "value": [
        len(arima_diagnostics),
        int(arima_diagnostics["converged"].sum()),
        int((~arima_diagnostics["converged"]).sum()),
        int(arima_diagnostics["fallback_used"].sum()),
        100 * arima_diagnostics["converged"].mean(),
        100.0,
        100 * arima_diagnostics["fallback_used"].mean()
    ]
})

arima_convergence_summary.to_csv(
    OUTPUT_DIR / "arima_sarima_convergence_summary.csv",
    index=False
)

print(arima_convergence_summary)


                measure    value
0            Total fits  5000.00
1        Converged fits  4971.00
2    Non-converged fits    29.00
3         Fallback fits     0.00
4  Convergence rate (%)    99.42
5   Usable-fit rate (%)   100.00
6     Fallback rate (%)     0.00


Run XGBoost

In [28]:
import gc
import itertools
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor


RANDOM_SEED = 2026
FORECAST_HORIZON = 28
FINAL_TEST_ORIGIN = 1913

LAGS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 28]

VALIDATION_ORIGINS = [1829, 1857, 1885]


CATEGORICAL_FEATURES = [
    "item_id",
    "cat_id",
    "dept_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]


NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_sd_7",
    "zero_proportion_7",
    "rolling_mean_28",
    "rolling_sd_28",
    "zero_proportion_28",
    "weekday",
    "month",
    "snap",
    "price_level",
    "price_change_7",
    "price_pct_change_7"
]


In [34]:
from pathlib import Path
import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. File locations
# --------------------------------------------------

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))

SALES_FILE = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_FILE = DATA_DIR / "calendar.csv"
PRICES_FILE = DATA_DIR / "sell_prices.csv"
SAMPLE_FILE = DATA_DIR / "selected_sample_ids.csv"

OUTPUT_DIR = DATA_DIR / "chapter5_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


for file in [
    SALES_FILE,
    CALENDAR_FILE,
    PRICES_FILE,
    SAMPLE_FILE
]:
    if not file.exists():
        raise FileNotFoundError(
            f"File not found: {file}"
        )

print("All required files were found.")


# --------------------------------------------------
# 2. Load selected sales
# --------------------------------------------------

sales = pd.read_csv(SALES_FILE)
sample = pd.read_csv(SAMPLE_FILE)


if "demand_class" not in sample.columns:

    if "demand_pattern" in sample.columns:
        sample = sample.rename(
            columns={
                "demand_pattern": "demand_class"
            }
        )

    else:
        raise ValueError(
            "The sample requires demand_class "
            "or demand_pattern."
        )


class_names = {
    "smooth": "Smooth",
    "erratic": "Erratic",
    "intermittent": "Intermittent",
    "lumpy": "Lumpy"
}


sample["demand_class"] = (
    sample["demand_class"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(class_names)
)


sample_for_merge = (
    sample[
        ["id", "demand_class"]
    ]
    .sort_values("id")
    .reset_index(drop=True)
)


selected_sales = sample_for_merge.merge(
    sales,
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True
)


missing_ids = selected_sales.loc[
    selected_sales["_merge"] != "both",
    "id"
]


if len(missing_ids) > 0:
    raise ValueError(
        f"Selected IDs not found: "
        f"{missing_ids.head().tolist()}"
    )


selected_sales = selected_sales.drop(
    columns="_merge"
)


# --------------------------------------------------
# 3. Identify demand columns
# --------------------------------------------------

def day_number(column_name):
    return int(
        column_name.replace("d_", "")
    )


day_columns = sorted(
    [
        column
        for column in sales.columns
        if column.startswith("d_")
    ],
    key=day_number
)


if len(day_columns) < 1941:
    raise ValueError(
        "Demand data through d_1941 are required."
    )


demand_matrix = selected_sales[
    day_columns
].to_numpy(dtype=float)


# --------------------------------------------------
# 4. Prepare calendar
# --------------------------------------------------

calendar = pd.read_csv(CALENDAR_FILE)


calendar["day_number"] = (
    calendar["d"]
    .str.replace(
        "d_",
        "",
        regex=False
    )
    .astype(int)
)


calendar["date"] = pd.to_datetime(
    calendar["date"]
)

calendar["weekday"] = (
    calendar["date"].dt.dayofweek
)

calendar["month"] = (
    calendar["date"].dt.month
)


for column in [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]:
    calendar[column] = (
        calendar[column]
        .fillna("none")
        .astype(str)
    )


calendar = (
    calendar
    .sort_values("day_number")
    .reset_index(drop=True)
)


# --------------------------------------------------
# 5. Determine active periods
# --------------------------------------------------

prices = pd.read_csv(PRICES_FILE)


valid_prices = prices[
    prices["sell_price"].notna()
    & (prices["sell_price"] > 0)
].copy()


first_price_week = (
    valid_prices
    .groupby(
        ["store_id", "item_id"],
        as_index=False
    )["wm_yr_wk"]
    .min()
    .rename(
        columns={
            "wm_yr_wk": "first_price_week"
        }
    )
)


first_day_of_week = (
    calendar
    .groupby("wm_yr_wk")[
        "day_number"
    ]
    .min()
)


first_price_week["first_active_day"] = (
    first_price_week[
        "first_price_week"
    ].map(first_day_of_week)
)


selected_sales = selected_sales.merge(
    first_price_week[
        [
            "store_id",
            "item_id",
            "first_price_week",
            "first_active_day"
        ]
    ],
    on=["store_id", "item_id"],
    how="left",
    validate="one_to_one"
)


if selected_sales[
    "first_active_day"
].isna().any():
    raise ValueError(
        "Some selected series have no active period."
    )


selected_sales["first_active_day"] = (
    selected_sales[
        "first_active_day"
    ].astype(int)
)


# --------------------------------------------------
# 6. Create the daily price matrix
# --------------------------------------------------

number_of_series = len(selected_sales)
number_of_days = len(day_columns)


price_matrix = np.full(
    (
        number_of_series,
        number_of_days
    ),
    np.nan,
    dtype=float
)


weeks_by_day = calendar.loc[
    :number_of_days - 1,
    "wm_yr_wk"
].to_numpy()


price_groups = {
    key: group.set_index(
        "wm_yr_wk"
    )["sell_price"]

    for key, group in prices.groupby(
        ["store_id", "item_id"]
    )
}


for series_index, row in (
    selected_sales.iterrows()
):

    key = (
        row["store_id"],
        row["item_id"]
    )

    if key not in price_groups:
        continue

    daily_prices = (
        pd.Series(weeks_by_day)
        .map(price_groups[key])
        .astype(float)
        .ffill()
    )

    price_matrix[series_index] = (
        daily_prices.to_numpy()
    )


# --------------------------------------------------
# 7. Verification
# --------------------------------------------------

assert len(selected_sales) == 500
assert len(day_columns) == 1941
assert demand_matrix.shape == (500, 1941)
assert price_matrix.shape == (500, 1941)
assert selected_sales["first_active_day"].notna().all()


print("\nM5 data successfully restored.")
print("Selected series:", len(selected_sales))
print("Demand matrix:", demand_matrix.shape)
print("Price matrix:", price_matrix.shape)
print("Output folder:", OUTPUT_DIR)


All required files were found.


/var/folders/qg/nvrwk7_51bngv6xjz8w6p5f40000gn/T/ipykernel_30528/3987212543.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  selected_sales = sample_for_merge.merge(
/var/folders/qg/nvrwk7_51bngv6xjz8w6p5f40000gn/T/ipykernel_30528/3987212543.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  selected_sales = sample_for_merge.merge(



M5 data successfully restored.
Selected series: 500
Demand matrix: (500, 1941)
Price matrix: (500, 1941)
Output folder: <M5_DATA_DIR>/chapter5_results


In [36]:
required_objects = [
    "selected_sales",
    "day_columns",
    "calendar",
    "price_matrix",
    "demand_matrix",
    "OUTPUT_DIR"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

print("Missing objects:", missing_objects)


Missing objects: []


In [38]:
def safe_price_features(
    price_level,
    earlier_price
):

    price_change = price_level - earlier_price

    price_pct_change = np.divide(
        price_change,
        earlier_price,
        out=np.zeros_like(
            price_change,
            dtype=float
        ),
        where=(
            np.isfinite(earlier_price)
            & (earlier_price != 0)
        )
    )

    price_change = np.nan_to_num(
        price_change,
        nan=0.0
    )

    price_pct_change = np.nan_to_num(
        price_pct_change,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return price_change, price_pct_change


def build_historical_feature_panel(
    selected_sales,
    day_columns,
    calendar,
    price_matrix
):

    demand_values = selected_sales[
        day_columns
    ].to_numpy(dtype=float)

    calendar_by_day = calendar.set_index(
        "day_number"
    )

    panel_parts = []

    for series_index, row in (
        selected_sales
        .reset_index(drop=True)
        .iterrows()
    ):

        first_target_day = max(
            int(row["first_active_day"]) + 28,
            29
        )

        target_days = np.arange(
            first_target_day,
            FINAL_TEST_ORIGIN + 1
        )

        target_indices = target_days - 1
        series_demand = demand_values[series_index]

        series_panel = pd.DataFrame({
            "id": row["id"],
            "target_day": target_days,
            "target": series_demand[target_indices],
            "item_id": row["item_id"],
            "cat_id": row["cat_id"],
            "dept_id": row["dept_id"],
            "store_id": row["store_id"],
            "state_id": row["state_id"]
        })

        for lag in LAGS:
            series_panel[f"lag_{lag}"] = (
                series_demand[target_indices - lag]
            )

        for window in ROLLING_WINDOWS:

            rolling_values = np.vstack([
                series_demand[
                    index - window:index
                ]
                for index in target_indices
            ])

            series_panel[f"rolling_mean_{window}"] = (
                rolling_values.mean(axis=1)
            )

            series_panel[f"rolling_sd_{window}"] = (
                rolling_values.std(
                    axis=1,
                    ddof=0
                )
            )

            series_panel[
                f"zero_proportion_{window}"
            ] = (
                rolling_values == 0
            ).mean(axis=1)

        calendar_rows = calendar_by_day.loc[
            target_days
        ]

        series_panel["weekday"] = (
            calendar_rows["weekday"].to_numpy()
        )

        series_panel["month"] = (
            calendar_rows["month"].to_numpy()
        )

        for event_column in [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]:
            series_panel[event_column] = (
                calendar_rows[
                    event_column
                ].to_numpy()
            )

        snap_column = f"snap_{row['state_id']}"

        series_panel["snap"] = (
            calendar_rows[
                snap_column
            ].to_numpy(dtype=float)
        )

        # Price observed on the preceding day.
        price_level = price_matrix[
            series_index,
            target_indices - 1
        ]

        price_seven_days_earlier = price_matrix[
            series_index,
            target_indices - 8
        ]

        (
            price_change,
            price_pct_change
        ) = safe_price_features(
            price_level,
            price_seven_days_earlier
        )

        series_panel["price_level"] = (
            np.nan_to_num(
                price_level,
                nan=0.0
            )
        )

        series_panel["price_change_7"] = (
            price_change
        )

        series_panel["price_pct_change_7"] = (
            price_pct_change
        )

        panel_parts.append(series_panel)

    return pd.concat(
        panel_parts,
        ignore_index=True
    )


In [40]:
panel_file = (
    OUTPUT_DIR
    / "xgboost_historical_feature_panel.pkl"
)

if panel_file.exists():

    print("Loading the saved XGBoost panel...")

    xgboost_panel = pd.read_pickle(
        panel_file
    )

else:

    print("Constructing the XGBoost panel...")

    xgboost_panel = (
        build_historical_feature_panel(
            selected_sales=selected_sales,
            day_columns=day_columns,
            calendar=calendar,
            price_matrix=price_matrix
        )
    )

    xgboost_panel.to_pickle(
        panel_file
    )

print("Panel shape:", xgboost_panel.shape)
print("Panel series:", xgboost_panel["id"].nunique())
print("Maximum target day:", xgboost_panel["target_day"].max())
print(
    "Missing numeric values:",
    xgboost_panel[
        NUMERIC_FEATURES
    ].isna().sum().sum()
)


Loading the saved XGBoost panel...
Panel shape: (825229, 28)
Panel series: 500
Maximum target day: 1913
Missing numeric values: 0


In [42]:
def fit_feature_encoder(training_frame):

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32
    )

    encoder.fit(
        training_frame[
            CATEGORICAL_FEATURES
        ].astype(str)
    )

    return encoder


def transform_features(
    frame,
    encoder
):

    numeric = frame[
        NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    categorical = encoder.transform(
        frame[
            CATEGORICAL_FEATURES
        ].astype(str)
    ).astype(np.float32)

    features = np.column_stack([
        numeric,
        categorical
    ])

    if not np.isfinite(features).all():
        raise ValueError(
            "The XGBoost feature matrix contains invalid values."
        )

    return features


def fit_xgboost_at_origin(
    panel,
    origin,
    configuration
):

    training_frame = panel[
        panel["target_day"] <= origin
    ]

    encoder = fit_feature_encoder(
        training_frame
    )

    x_train = transform_features(
        training_frame,
        encoder
    )

    y_train = training_frame[
        "target"
    ].to_numpy(dtype=np.float32)

    model = XGBRegressor(
        n_estimators=int(
            configuration["n_estimators"]
        ),
        learning_rate=float(
            configuration["learning_rate"]
        ),
        max_depth=int(
            configuration["max_depth"]
        ),
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        x_train,
        y_train,
        verbose=False
    )

    return model, encoder, len(training_frame)


def recursive_xgboost_forecast(
    model,
    encoder,
    selected_sales,
    day_columns,
    calendar,
    price_matrix,
    origin
):

    demand_values = selected_sales[
        day_columns
    ].to_numpy(dtype=float)

    number_of_series = len(selected_sales)

    history = np.full(
        (
            number_of_series,
            origin + FORECAST_HORIZON
        ),
        np.nan,
        dtype=float
    )

    # Actual demand is available only through the origin.
    history[:, :origin] = demand_values[:, :origin]

    metadata = selected_sales.reset_index(
        drop=True
    )

    calendar_by_day = calendar.set_index(
        "day_number"
    )

    # Hold the last known price constant.
    origin_index = origin - 1

    known_price = price_matrix[
        :,
        origin_index
    ]

    earlier_price = price_matrix[
        :,
        origin_index - 7
    ]

    (
        price_change,
        price_pct_change
    ) = safe_price_features(
        known_price,
        earlier_price
    )

    for horizon in range(
        1,
        FORECAST_HORIZON + 1
    ):

        target_day = origin + horizon
        target_index = target_day - 1

        calendar_row = calendar_by_day.loc[
            target_day
        ]

        future_frame = metadata[
            [
                "item_id",
                "cat_id",
                "dept_id",
                "store_id",
                "state_id"
            ]
        ].copy()

        for lag in LAGS:
            future_frame[f"lag_{lag}"] = (
                history[
                    :,
                    target_index - lag
                ]
            )

        for window in ROLLING_WINDOWS:

            rolling_values = history[
                :,
                target_index - window:
                target_index
            ]

            future_frame[
                f"rolling_mean_{window}"
            ] = rolling_values.mean(axis=1)

            future_frame[
                f"rolling_sd_{window}"
            ] = rolling_values.std(
                axis=1,
                ddof=0
            )

            future_frame[
                f"zero_proportion_{window}"
            ] = (
                rolling_values == 0
            ).mean(axis=1)

        future_frame["weekday"] = float(
            calendar_row["weekday"]
        )

        future_frame["month"] = float(
            calendar_row["month"]
        )

        for event_column in [
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]:
            future_frame[event_column] = str(
                calendar_row[event_column]
            )

        future_frame["snap"] = [
            float(
                calendar_row[
                    f"snap_{state}"
                ]
            )
            for state in metadata["state_id"]
        ]

        future_frame["price_level"] = (
            np.nan_to_num(
                known_price,
                nan=0.0
            )
        )

        future_frame["price_change_7"] = (
            price_change
        )

        future_frame["price_pct_change_7"] = (
            price_pct_change
        )

        x_future = transform_features(
            future_frame,
            encoder
        )

        prediction = np.asarray(
            model.predict(x_future),
            dtype=float
        )

        prediction = np.maximum(
            prediction,
            0.0
        )

        if not np.isfinite(prediction).all():
            raise ValueError(
                f"Invalid prediction at horizon {horizon}."
            )

        # Earlier predictions become subsequent lags.
        history[:, target_index] = prediction

    return history[
        :,
        origin:origin + FORECAST_HORIZON
    ]


def mean_validation_rmsse(
    forecasts,
    actuals,
    origin
):

    rmsse_values = []

    for series_index, row in (
        selected_sales
        .reset_index(drop=True)
        .iterrows()
    ):

        active_start_index = (
            int(row["first_active_day"]) - 1
        )

        training_demand = demand_matrix[
            series_index,
            active_start_index:origin
        ]

        denominator = np.mean(
            np.diff(training_demand) ** 2
        )

        if (
            denominator <= 0
            or not np.isfinite(denominator)
        ):
            continue

        numerator = np.mean(
            (
                actuals[series_index]
                - forecasts[series_index]
            ) ** 2
        )

        rmsse_values.append(
            np.sqrt(
                numerator / denominator
            )
        )

    return (
        float(np.mean(rmsse_values)),
        len(rmsse_values)
    )


In [44]:
xgboost_configurations = []

for (
    n_estimators,
    learning_rate,
    max_depth
) in itertools.product(
    [300, 600],
    [0.03, 0.05],
    [6, 10]
):

    configuration_id = (
        f"n{n_estimators}_"
        f"lr{str(learning_rate).replace('.', 'p')}_"
        f"d{max_depth}"
    )

    xgboost_configurations.append({
        "configuration_id": configuration_id,
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth
    })


print(
    pd.DataFrame(xgboost_configurations)
)


  configuration_id  n_estimators  learning_rate  max_depth
0   n300_lr0p03_d6           300           0.03          6
1  n300_lr0p03_d10           300           0.03         10
2   n300_lr0p05_d6           300           0.05          6
3  n300_lr0p05_d10           300           0.05         10
4   n600_lr0p03_d6           600           0.03          6
5  n600_lr0p03_d10           600           0.03         10
6   n600_lr0p05_d6           600           0.05          6
7  n600_lr0p05_d10           600           0.05         10


In [46]:
tuning_file = (
    OUTPUT_DIR
    / "xgboost_tuning_rounds.csv"
)

if tuning_file.exists():

    tuning_results = pd.read_csv(
        tuning_file
    )

else:

    tuning_results = pd.DataFrame()


valid_configuration_ids = {
    configuration["configuration_id"]
    for configuration in xgboost_configurations
}

if len(tuning_results) > 0:

    tuning_results = tuning_results[
        tuning_results["configuration_id"].isin(
            valid_configuration_ids
        )
        & tuning_results["forecast_origin"].isin(
            VALIDATION_ORIGINS
        )
    ].copy()


for configuration in xgboost_configurations:

    configuration_id = (
        configuration["configuration_id"]
    )

    for origin in VALIDATION_ORIGINS:

        already_completed = False

        if len(tuning_results) > 0:

            already_completed = (
                (
                    tuning_results[
                        "configuration_id"
                    ] == configuration_id
                )
                & (
                    tuning_results[
                        "forecast_origin"
                    ] == origin
                )
            ).any()

        if already_completed:

            print(
                f"{configuration_id}, d_{origin} "
                "already completed. Skipping."
            )

            continue

        prediction_file = (
            OUTPUT_DIR
            / (
                f"xgboost_tuning_"
                f"{configuration_id}_"
                f"origin_{origin}.npy"
            )
        )

        started = time.perf_counter()

        if prediction_file.exists():

            predictions = np.load(
                prediction_file
            )

            training_rows = int(
                (
                    xgboost_panel["target_day"]
                    <= origin
                ).sum()
            )

            reused_prediction = True

        else:

            print(
                f"Fitting {configuration_id} "
                f"at d_{origin}..."
            )

            (
                xgb_model,
                xgb_encoder,
                training_rows
            ) = fit_xgboost_at_origin(
                panel=xgboost_panel,
                origin=origin,
                configuration=configuration
            )

            predictions = (
                recursive_xgboost_forecast(
                    model=xgb_model,
                    encoder=xgb_encoder,
                    selected_sales=selected_sales,
                    day_columns=day_columns,
                    calendar=calendar,
                    price_matrix=price_matrix,
                    origin=origin
                )
            )

            np.save(
                prediction_file,
                predictions
            )

            reused_prediction = False

            del xgb_model
            del xgb_encoder

            gc.collect()

        actuals = demand_matrix[
            :,
            origin:origin + 28
        ]

        (
            mean_rmsse,
            valid_rmsse_series
        ) = mean_validation_rmsse(
            forecasts=predictions,
            actuals=actuals,
            origin=origin
        )

        new_result = pd.DataFrame([{
            **configuration,
            "forecast_origin": origin,
            "mean_rmsse": mean_rmsse,
            "valid_rmsse_series":
                valid_rmsse_series,
            "training_rows": training_rows,
            "fit_and_forecast_seconds":
                time.perf_counter() - started,
            "reused_prediction":
                reused_prediction
        }])

        tuning_results = pd.concat(
            [
                tuning_results,
                new_result
            ],
            ignore_index=True
        )

        tuning_results = (
            tuning_results
            .drop_duplicates(
                subset=[
                    "configuration_id",
                    "forecast_origin"
                ],
                keep="last"
            )
        )

        tuning_results.to_csv(
            tuning_file,
            index=False
        )

        print(
            f"Completed {configuration_id} "
            f"at d_{origin}: "
            f"RMSSE = {mean_rmsse:.4f}"
        )


expected_tuning_fits = (
    len(xgboost_configurations)
    * len(VALIDATION_ORIGINS)
)

if len(tuning_results) != expected_tuning_fits:
    raise AssertionError(
        f"Expected {expected_tuning_fits} tuning fits, "
        f"but found {len(tuning_results)}."
    )

print("\nAll 24 XGBoost tuning fits are complete.")


Fitting n300_lr0p03_d6 at d_1829...
Completed n300_lr0p03_d6 at d_1829: RMSSE = 0.7485
Fitting n300_lr0p03_d6 at d_1857...
Completed n300_lr0p03_d6 at d_1857: RMSSE = 0.7465
Fitting n300_lr0p03_d6 at d_1885...
Completed n300_lr0p03_d6 at d_1885: RMSSE = 0.7448
Fitting n300_lr0p03_d10 at d_1829...
Completed n300_lr0p03_d10 at d_1829: RMSSE = 0.7442
Fitting n300_lr0p03_d10 at d_1857...
Completed n300_lr0p03_d10 at d_1857: RMSSE = 0.7458
Fitting n300_lr0p03_d10 at d_1885...
Completed n300_lr0p03_d10 at d_1885: RMSSE = 0.7273
Fitting n300_lr0p05_d6 at d_1829...
Completed n300_lr0p05_d6 at d_1829: RMSSE = 0.7456
Fitting n300_lr0p05_d6 at d_1857...
Completed n300_lr0p05_d6 at d_1857: RMSSE = 0.7469
Fitting n300_lr0p05_d6 at d_1885...
Completed n300_lr0p05_d6 at d_1885: RMSSE = 0.7311
Fitting n300_lr0p05_d10 at d_1829...
Completed n300_lr0p05_d10 at d_1829: RMSSE = 0.7455
Fitting n300_lr0p05_d10 at d_1857...
Completed n300_lr0p05_d10 at d_1857: RMSSE = 0.7514
Fitting n300_lr0p05_d10 at d_1885

In [48]:
xgboost_tuning_summary = (
    tuning_results
    .groupby(
        [
            "configuration_id",
            "n_estimators",
            "learning_rate",
            "max_depth"
        ],
        as_index=False
    )["mean_rmsse"]
    .mean()
    .rename(
        columns={
            "mean_rmsse":
                "mean_validation_rmsse"
        }
    )
    .sort_values(
        [
            "mean_validation_rmsse",
            "max_depth",
            "n_estimators",
            "learning_rate"
        ]
    )
    .reset_index(drop=True)
)


selected_row = (
    xgboost_tuning_summary.iloc[0]
)

selected_configuration = {
    "configuration_id":
        selected_row["configuration_id"],

    "n_estimators":
        int(selected_row["n_estimators"]),

    "learning_rate":
        float(selected_row["learning_rate"]),

    "max_depth":
        int(selected_row["max_depth"])
}


xgboost_tuning_summary.to_csv(
    OUTPUT_DIR
    / "xgboost_tuning_summary.csv",
    index=False
)

pd.DataFrame(
    [selected_configuration]
).assign(
    mean_validation_rmsse=(
        selected_row[
            "mean_validation_rmsse"
        ]
    )
).to_csv(
    OUTPUT_DIR
    / "xgboost_selected_configuration.csv",
    index=False
)


print(xgboost_tuning_summary)
print("\nSelected configuration:")
print(selected_configuration)


  configuration_id  n_estimators  learning_rate  max_depth  \
0  n300_lr0p03_d10           300           0.03         10   
1   n300_lr0p05_d6           300           0.05          6   
2  n600_lr0p03_d10           600           0.03         10   
3   n600_lr0p05_d6           600           0.05          6   
4  n300_lr0p05_d10           300           0.05         10   
5   n600_lr0p03_d6           600           0.03          6   
6   n300_lr0p03_d6           300           0.03          6   
7  n600_lr0p05_d10           600           0.05         10   

   mean_validation_rmsse  
0               0.739080  
1               0.741179  
2               0.742453  
3               0.742820  
4               0.744181  
5               0.745582  
6               0.746582  
7               0.753924  

Selected configuration:
{'configuration_id': 'n300_lr0p03_d10', 'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 10}


In [50]:
forecast_schedule = pd.DataFrame({
    "forecast_origin": [
        1661, 1689, 1717, 1745, 1773, 1801,
        1829, 1857, 1885,
        1913
    ],
    "purpose": [
        "calibration", "calibration", "calibration",
        "calibration", "calibration", "calibration",
        "validation", "validation", "validation",
        "final_test"
    ],
    "round": [
        1, 2, 3, 4, 5, 6,
        1, 2, 3,
        1
    ]
})


def create_xgboost_forecast_frame(
    predictions,
    origin,
    purpose,
    round_number
):

    actuals = demand_matrix[
        :,
        origin:origin + 28
    ]

    horizons = np.arange(1, 29)

    frame = pd.DataFrame({
        "id": np.repeat(
            selected_sales["id"].to_numpy(),
            28
        ),
        "demand_class": np.repeat(
            selected_sales[
                "demand_class"
            ].to_numpy(),
            28
        ),
        "model": "xgboost",
        "purpose": purpose,
        "round": round_number,
        "forecast_origin": origin,
        "forecast_origin_label": f"d_{origin}",
        "horizon": np.tile(
            horizons,
            len(selected_sales)
        ),
        "target_day": np.tile(
            origin + horizons,
            len(selected_sales)
        ),
        "actual": actuals.reshape(-1),
        "forecast": predictions.reshape(-1)
    })

    frame["target_day_label"] = (
        "d_"
        + frame["target_day"].astype(str)
    )

    return frame


In [52]:
selected_origin_forecasts = []
selected_diagnostics = []


for schedule_row in forecast_schedule.itertuples(
    index=False
):

    origin = int(
        schedule_row.forecast_origin
    )

    origin_forecast_file = (
        OUTPUT_DIR
        / f"xgboost_forecasts_origin_{origin}.csv"
    )

    selected_prediction_file = (
        OUTPUT_DIR
        / (
            f"xgboost_selected_predictions_"
            f"origin_{origin}.npy"
        )
    )

    if origin_forecast_file.exists():

        existing_forecasts = pd.read_csv(
            origin_forecast_file
        )

        if len(existing_forecasts) == 500 * 28:

            print(
                f"Selected XGBoost at d_{origin} "
                "already completed. Skipping."
            )

            selected_origin_forecasts.append(
                existing_forecasts
            )

            continue

    started = time.perf_counter()
    reused_tuning_prediction = False

    if selected_prediction_file.exists():

        predictions = np.load(
            selected_prediction_file
        )

        reused_saved_prediction = True

    else:

        tuning_prediction_file = (
            OUTPUT_DIR
            / (
                f"xgboost_tuning_"
                f"{selected_configuration['configuration_id']}_"
                f"origin_{origin}.npy"
            )
        )

        if tuning_prediction_file.exists():

            predictions = np.load(
                tuning_prediction_file
            )

            reused_tuning_prediction = True
            reused_saved_prediction = True

        else:

            print(
                f"Fitting selected XGBoost "
                f"at d_{origin}..."
            )

            (
                selected_model,
                selected_encoder,
                training_rows
            ) = fit_xgboost_at_origin(
                panel=xgboost_panel,
                origin=origin,
                configuration=selected_configuration
            )

            predictions = (
                recursive_xgboost_forecast(
                    model=selected_model,
                    encoder=selected_encoder,
                    selected_sales=selected_sales,
                    day_columns=day_columns,
                    calendar=calendar,
                    price_matrix=price_matrix,
                    origin=origin
                )
            )

            reused_saved_prediction = False

            del selected_model
            del selected_encoder

            gc.collect()

        np.save(
            selected_prediction_file,
            predictions
        )

    if predictions.shape != (500, 28):
        raise ValueError(
            f"Invalid prediction shape at d_{origin}: "
            f"{predictions.shape}"
        )

    if not np.isfinite(predictions).all():
        raise ValueError(
            f"Invalid forecasts at d_{origin}."
        )

    predictions = np.maximum(
        predictions,
        0.0
    )

    origin_forecasts = (
        create_xgboost_forecast_frame(
            predictions=predictions,
            origin=origin,
            purpose=str(schedule_row.purpose),
            round_number=int(schedule_row.round)
        )
    )

    origin_forecasts.to_csv(
        origin_forecast_file,
        index=False
    )

    selected_origin_forecasts.append(
        origin_forecasts
    )

    selected_diagnostics.append({
        "forecast_origin": origin,
        **selected_configuration,
        "reused_saved_prediction":
            reused_saved_prediction,
        "reused_tuning_prediction":
            reused_tuning_prediction,
        "fit_and_forecast_seconds":
            time.perf_counter() - started
    })

    print(
        f"Completed selected XGBoost "
        f"at d_{origin}."
    )


xgboost_forecasts = pd.concat(
    selected_origin_forecasts,
    ignore_index=True
)


Fitting selected XGBoost at d_1661...
Completed selected XGBoost at d_1661.
Fitting selected XGBoost at d_1689...
Completed selected XGBoost at d_1689.
Fitting selected XGBoost at d_1717...
Completed selected XGBoost at d_1717.
Fitting selected XGBoost at d_1745...
Completed selected XGBoost at d_1745.
Fitting selected XGBoost at d_1773...
Completed selected XGBoost at d_1773.
Fitting selected XGBoost at d_1801...
Completed selected XGBoost at d_1801.
Completed selected XGBoost at d_1829.
Completed selected XGBoost at d_1857.
Completed selected XGBoost at d_1885.
Fitting selected XGBoost at d_1913...
Completed selected XGBoost at d_1913.


In [54]:
xgboost_errors = (
    xgboost_forecasts
    .groupby(
        [
            "id",
            "demand_class",
            "model",
            "purpose",
            "round",
            "forecast_origin"
        ],
        as_index=False
    )
    .agg(
        actual_protection_demand=("actual", "sum"),
        forecast_protection_demand=("forecast", "sum"),
        forecast_days=("horizon", "count")
    )
)


xgboost_errors["protection_period_error"] = (
    xgboost_errors[
        "actual_protection_demand"
    ]
    - xgboost_errors[
        "forecast_protection_demand"
    ]
)


xgboost_checks = pd.DataFrame({
    "check": [
        "Expected number of forecast rows",
        "Exactly 500 series",
        "Exactly ten origins",
        "No missing forecasts",
        "Finite forecasts",
        "Non-negative forecasts",
        "28 rows per series and origin",
        "Eight tuning configurations",
        "Three validation rounds per configuration"
    ],
    "passed": [
        len(xgboost_forecasts)
        == 500 * 10 * 28,

        xgboost_forecasts[
            "id"
        ].nunique() == 500,

        xgboost_forecasts[
            "forecast_origin"
        ].nunique() == 10,

        xgboost_forecasts[
            "forecast"
        ].notna().all(),

        np.isfinite(
            xgboost_forecasts["forecast"]
        ).all(),

        (
            xgboost_forecasts["forecast"] >= 0
        ).all(),

        xgboost_forecasts.groupby(
            ["id", "forecast_origin"]
        ).size().eq(28).all(),

        xgboost_tuning_summary[
            "configuration_id"
        ].nunique() == 8,

        tuning_results.groupby(
            "configuration_id"
        ).size().eq(3).all()
    ]
})


if not xgboost_checks["passed"].all():
    raise AssertionError(
        "At least one XGBoost check failed."
    )


xgboost_forecasts.to_csv(
    OUTPUT_DIR / "xgboost_forecasts.csv",
    index=False
)

xgboost_errors.to_csv(
    OUTPUT_DIR
    / "xgboost_protection_period_errors.csv",
    index=False
)

xgboost_checks.to_csv(
    OUTPUT_DIR
    / "xgboost_implementation_checks.csv",
    index=False
)

pd.DataFrame(
    selected_diagnostics
).to_csv(
    OUTPUT_DIR
    / "xgboost_selected_fit_diagnostics.csv",
    index=False
)


print("\nXGBoost completed successfully.")
print("Forecast rows:", len(xgboost_forecasts))
print("Protection-period errors:", len(xgboost_errors))
print("\nImplementation checks:")
print(xgboost_checks)



XGBoost completed successfully.
Forecast rows: 140000
Protection-period errors: 5000

Implementation checks:
                                       check  passed
0           Expected number of forecast rows    True
1                         Exactly 500 series    True
2                        Exactly ten origins    True
3                       No missing forecasts    True
4                           Finite forecasts    True
5                     Non-negative forecasts    True
6              28 rows per series and origin    True
7                Eight tuning configurations    True
8  Three validation rounds per configuration    True


In [56]:
xgboost_selected = pd.read_csv(
    OUTPUT_DIR / "xgboost_selected_configuration.csv"
)

xgboost_ranking = pd.read_csv(
    OUTPUT_DIR / "xgboost_tuning_summary.csv"
)

print("Selected XGBoost configuration:")
print(xgboost_selected)

print("\nComplete tuning ranking:")
print(xgboost_ranking)


Selected XGBoost configuration:
  configuration_id  n_estimators  learning_rate  max_depth  \
0  n300_lr0p03_d10           300           0.03         10   

   mean_validation_rmsse  
0                0.73908  

Complete tuning ranking:
  configuration_id  n_estimators  learning_rate  max_depth  \
0  n300_lr0p03_d10           300           0.03         10   
1   n300_lr0p05_d6           300           0.05          6   
2  n600_lr0p03_d10           600           0.03         10   
3   n600_lr0p05_d6           600           0.05          6   
4  n300_lr0p05_d10           300           0.05         10   
5   n600_lr0p03_d6           600           0.03          6   
6   n300_lr0p03_d6           300           0.03          6   
7  n600_lr0p05_d10           600           0.05         10   

   mean_validation_rmsse  
0               0.739080  
1               0.741179  
2               0.742453  
3               0.742820  
4               0.744181  
5               0.745582  
6              

In [58]:
forecast_origins = [
    1661, 1689, 1717, 1745, 1773,
    1801, 1829, 1857, 1885, 1913
]

required_result_files = {
    "Seasonal naïve":
        OUTPUT_DIR / "seasonal_naive_forecasts.csv",

    "ETS":
        OUTPUT_DIR / "ets_forecasts.csv",

    "ARIMA/SARIMA":
        OUTPUT_DIR / "arima_sarima_forecasts.csv",

    "XGBoost":
        OUTPUT_DIR / "xgboost_forecasts.csv"
}

file_status = []

for model, file in required_result_files.items():
    file_status.append({
        "model": model,
        "file": file.name,
        "exists": file.exists()
    })

for origin in forecast_origins:

    tirex_file = (
        OUTPUT_DIR
        / f"tirex2_central_forecasts_origin_{origin}.csv"
    )

    file_status.append({
        "model": f"TiRex-2 d_{origin}",
        "file": tirex_file.name,
        "exists": tirex_file.exists()
    })

file_status = pd.DataFrame(file_status)

print(file_status)
print(
    "\nMissing files:",
    int((~file_status["exists"]).sum())
)


             model                                      file  exists
0   Seasonal naïve              seasonal_naive_forecasts.csv    True
1              ETS                         ets_forecasts.csv    True
2     ARIMA/SARIMA                arima_sarima_forecasts.csv    True
3          XGBoost                     xgboost_forecasts.csv    True
4   TiRex-2 d_1661  tirex2_central_forecasts_origin_1661.csv    True
5   TiRex-2 d_1689  tirex2_central_forecasts_origin_1689.csv    True
6   TiRex-2 d_1717  tirex2_central_forecasts_origin_1717.csv    True
7   TiRex-2 d_1745  tirex2_central_forecasts_origin_1745.csv    True
8   TiRex-2 d_1773  tirex2_central_forecasts_origin_1773.csv    True
9   TiRex-2 d_1801  tirex2_central_forecasts_origin_1801.csv    True
10  TiRex-2 d_1829  tirex2_central_forecasts_origin_1829.csv    True
11  TiRex-2 d_1857  tirex2_central_forecasts_origin_1857.csv    True
12  TiRex-2 d_1885  tirex2_central_forecasts_origin_1885.csv    True
13  TiRex-2 d_1913  tirex2_central

Consolidate the ten TiRex-2 origins

In [61]:
tirex_parts = []

for origin in forecast_origins:

    tirex_file = (
        OUTPUT_DIR
        / f"tirex2_central_forecasts_origin_{origin}.csv"
    )

    origin_data = pd.read_csv(
        tirex_file
    )

    origin_data["forecast_origin_label"] = (
        "d_"
        + origin_data["forecast_origin"].astype(str)
    )

    origin_data["target_day_label"] = (
        "d_"
        + origin_data["target_day"].astype(str)
    )

    tirex_parts.append(origin_data)


tirex2_central_forecasts = pd.concat(
    tirex_parts,
    ignore_index=True
)


tirex_checks = pd.DataFrame({
    "check": [
        "Expected number of rows",
        "Exactly 500 series",
        "Exactly ten origins",
        "No missing forecasts",
        "Finite forecasts",
        "Non-negative forecasts",
        "28 rows per series and origin"
    ],
    "passed": [
        len(tirex2_central_forecasts)
        == 500 * 10 * 28,

        tirex2_central_forecasts[
            "id"
        ].nunique() == 500,

        tirex2_central_forecasts[
            "forecast_origin"
        ].nunique() == 10,

        tirex2_central_forecasts[
            "forecast"
        ].notna().all(),

        np.isfinite(
            tirex2_central_forecasts["forecast"]
        ).all(),

        (
            tirex2_central_forecasts["forecast"]
            >= 0
        ).all(),

        tirex2_central_forecasts.groupby(
            ["id", "forecast_origin"]
        ).size().eq(28).all()
    ]
})


if not tirex_checks["passed"].all():
    raise AssertionError(
        "At least one TiRex-2 check failed."
    )


tirex2_central_forecasts.to_csv(
    OUTPUT_DIR
    / "tirex2_central_forecasts.csv",
    index=False
)

tirex_checks.to_csv(
    OUTPUT_DIR
    / "tirex2_central_forecast_checks.csv",
    index=False
)


print(tirex_checks)
print(
    "\nTiRex-2 rows:",
    len(tirex2_central_forecasts)
)


                           check  passed
0        Expected number of rows    True
1             Exactly 500 series    True
2            Exactly ten origins    True
3           No missing forecasts    True
4               Finite forecasts    True
5         Non-negative forecasts    True
6  28 rows per series and origin    True

TiRex-2 rows: 140000


Combine all five models

In [64]:
model_files = {
    "seasonal_naive":
        OUTPUT_DIR / "seasonal_naive_forecasts.csv",

    "ets":
        OUTPUT_DIR / "ets_forecasts.csv",

    "arima_sarima":
        OUTPUT_DIR / "arima_sarima_forecasts.csv",

    "xgboost":
        OUTPUT_DIR / "xgboost_forecasts.csv",

    "tirex2":
        OUTPUT_DIR / "tirex2_central_forecasts.csv"
}


common_columns = [
    "id",
    "demand_class",
    "model",
    "purpose",
    "round",
    "forecast_origin",
    "forecast_origin_label",
    "horizon",
    "target_day",
    "target_day_label",
    "actual",
    "forecast"
]


model_parts = []

for model_name, model_file in model_files.items():

    model_data = pd.read_csv(
        model_file
    )

    # Ensure a common, unambiguous model name.
    model_data["model"] = model_name

    if "forecast_origin_label" not in model_data.columns:
        model_data["forecast_origin_label"] = (
            "d_"
            + model_data[
                "forecast_origin"
            ].astype(str)
        )

    if "target_day_label" not in model_data.columns:
        model_data["target_day_label"] = (
            "d_"
            + model_data[
                "target_day"
            ].astype(str)
        )

    missing_columns = (
        set(common_columns)
        - set(model_data.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{model_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    model_parts.append(
        model_data[common_columns]
    )


all_models_forecasts = pd.concat(
    model_parts,
    ignore_index=True
)


all_models_forecasts = (
    all_models_forecasts
    .sort_values(
        [
            "model",
            "forecast_origin",
            "demand_class",
            "id",
            "horizon"
        ]
    )
    .reset_index(drop=True)
)


Validate the combined panel

In [67]:
forecast_key = [
    "model",
    "id",
    "forecast_origin",
    "horizon"
]


actual_key = [
    "id",
    "forecast_origin",
    "horizon"
]


duplicate_rows = (
    all_models_forecasts
    .duplicated(forecast_key)
    .sum()
)


maximum_actual_versions = (
    all_models_forecasts
    .groupby(actual_key)["actual"]
    .nunique(dropna=False)
    .max()
)


rows_per_model = (
    all_models_forecasts
    .groupby("model")
    .size()
)


combined_checks = pd.DataFrame({
    "check": [
        "Expected number of rows",
        "Exactly five models",
        "140000 rows per model",
        "No duplicate forecast keys",
        "Actual demand identical across models",
        "No missing forecasts",
        "Finite forecasts",
        "Non-negative forecasts",
        "Target day equals origin plus horizon"
    ],
    "passed": [
        len(all_models_forecasts)
        == 5 * 500 * 10 * 28,

        all_models_forecasts[
            "model"
        ].nunique() == 5,

        rows_per_model.eq(
            500 * 10 * 28
        ).all(),

        duplicate_rows == 0,

        maximum_actual_versions == 1,

        all_models_forecasts[
            "forecast"
        ].notna().all(),

        np.isfinite(
            all_models_forecasts["forecast"]
        ).all(),

        (
            all_models_forecasts["forecast"]
            >= 0
        ).all(),

        (
            all_models_forecasts["target_day"]
            == all_models_forecasts[
                "forecast_origin"
            ]
            + all_models_forecasts["horizon"]
        ).all()
    ]
})


if not combined_checks["passed"].all():
    raise AssertionError(
        "At least one combined-panel check failed."
    )


all_models_forecasts.to_csv(
    OUTPUT_DIR
    / "all_models_central_forecasts.csv",
    index=False
)

combined_checks.to_csv(
    OUTPUT_DIR
    / "all_models_central_forecast_checks.csv",
    index=False
)


print("Rows per model:")
print(rows_per_model)

print("\nCombined-panel checks:")
print(combined_checks)

print(
    "\nTotal rows:",
    len(all_models_forecasts)
)


Rows per model:
model
arima_sarima      140000
ets               140000
seasonal_naive    140000
tirex2            140000
xgboost           140000
dtype: int64

Combined-panel checks:
                                   check  passed
0                Expected number of rows    True
1                    Exactly five models    True
2                  140000 rows per model    True
3             No duplicate forecast keys    True
4  Actual demand identical across models    True
5                   No missing forecasts    True
6                       Finite forecasts    True
7                 Non-negative forecasts    True
8  Target day equals origin plus horizon    True

Total rows: 700000


In [71]:
evaluation_origins = [
    1829,
    1857,
    1885,
    1913
]

scale_rows = []

selected_metadata = (
    selected_sales
    .reset_index(drop=True)
)


for series_index, row in selected_metadata.iterrows():

    active_start_index = (
        int(row["first_active_day"]) - 1
    )

    for origin in evaluation_origins:

        training_demand = demand_matrix[
            series_index,
            active_start_index:origin
        ]

        rmsse_scale = np.mean(
            np.diff(training_demand) ** 2
        )

        scale_rows.append({
            "id": row["id"],
            "forecast_origin": origin,
            "rmsse_scale": rmsse_scale
        })


rmsse_scales = pd.DataFrame(
    scale_rows
)


if (
    (~np.isfinite(rmsse_scales["rmsse_scale"]))
    | (rmsse_scales["rmsse_scale"] <= 0)
).any():
    raise ValueError(
        "At least one RMSSE scaling factor is invalid."
    )


print("RMSSE scaling rows:", len(rmsse_scales))
print(
    "Minimum RMSSE scale:",
    rmsse_scales["rmsse_scale"].min()
)


RMSSE scaling rows: 2000
Minimum RMSSE scale: 0.04418103448275862


In [73]:
performance_forecasts = (
    all_models_forecasts[
        all_models_forecasts["purpose"].isin(
            ["validation", "final_test"]
        )
    ]
    .copy()
)


performance_forecasts = performance_forecasts.merge(
    rmsse_scales,
    on=["id", "forecast_origin"],
    how="left",
    validate="many_to_one"
)


performance_forecasts["error"] = (
    performance_forecasts["actual"]
    - performance_forecasts["forecast"]
)


performance_forecasts["absolute_error"] = (
    performance_forecasts["error"].abs()
)


performance_forecasts["squared_error"] = (
    performance_forecasts["error"] ** 2
)


if performance_forecasts[
    "rmsse_scale"
].isna().any():
    raise ValueError(
        "At least one forecast has no RMSSE scale."
    )


print(
    "Performance forecast rows:",
    len(performance_forecasts)
)


Performance forecast rows: 280000


In [75]:
series_origin_metrics = (
    performance_forecasts
    .groupby(
        [
            "model",
            "purpose",
            "round",
            "forecast_origin",
            "id",
            "demand_class"
        ],
        as_index=False
    )
    .agg(
        forecast_days=("horizon", "count"),
        actual_total=("actual", "sum"),
        forecast_total=("forecast", "sum"),
        absolute_error_sum=("absolute_error", "sum"),
        mae=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        rmsse_scale=("rmsse_scale", "first")
    )
)


series_origin_metrics["rmsse"] = np.sqrt(
    series_origin_metrics["mean_squared_error"]
    / series_origin_metrics["rmsse_scale"]
)


series_origin_metrics["wape"] = np.where(
    series_origin_metrics["actual_total"] > 0,

    100
    * series_origin_metrics["absolute_error_sum"]
    / series_origin_metrics["actual_total"],

    np.nan
)


series_origin_metrics.to_csv(
    OUTPUT_DIR
    / "point_forecast_series_origin_metrics.csv",
    index=False
)


print(
    "Series-origin metric rows:",
    len(series_origin_metrics)
)


Series-origin metric rows: 10000


In [77]:
def create_accuracy_summary(
    daily_forecasts,
    series_metrics,
    grouping_columns
):

    daily_summary = (
        daily_forecasts
        .groupby(
            grouping_columns,
            as_index=False
        )
        .agg(
            forecast_observations=(
                "forecast",
                "size"
            ),
            actual_demand=(
                "actual",
                "sum"
            ),
            absolute_error_sum=(
                "absolute_error",
                "sum"
            ),
            mae=(
                "absolute_error",
                "mean"
            )
        )
    )


    daily_summary["wape"] = (
        100
        * daily_summary["absolute_error_sum"]
        / daily_summary["actual_demand"]
    )


    rmsse_summary = (
        series_metrics
        .groupby(
            grouping_columns,
            as_index=False
        )
        .agg(
            rmsse=("rmsse", "mean"),
            series_origin_comparisons=(
                "rmsse",
                "count"
            )
        )
    )


    summary = daily_summary.merge(
        rmsse_summary,
        on=grouping_columns,
        how="left",
        validate="one_to_one"
    )


    return summary


In [79]:
accuracy_overall = create_accuracy_summary(
    daily_forecasts=performance_forecasts,
    series_metrics=series_origin_metrics,
    grouping_columns=[
        "model",
        "purpose"
    ]
)


accuracy_overall = (
    accuracy_overall
    .sort_values(
        ["purpose", "rmsse", "mae"]
    )
    .reset_index(drop=True)
)


accuracy_overall.to_csv(
    OUTPUT_DIR
    / "point_forecast_accuracy_overall.csv",
    index=False
)


print(accuracy_overall[
    [
        "model",
        "purpose",
        "mae",
        "rmsse",
        "wape",
        "series_origin_comparisons"
    ]
])


            model     purpose       mae     rmsse       wape  \
0             ets  final_test  2.035558  0.746648  55.193025   
1    arima_sarima  final_test  2.038335  0.752934  55.268330   
2         xgboost  final_test  2.016504  0.756746  54.676385   
3          tirex2  final_test  1.900029  0.765295  51.518223   
4  seasonal_naive  final_test  2.462500  0.953023  66.769314   
5             ets  validation  2.044017  0.717636  57.484254   
6    arima_sarima  validation  2.057076  0.720286  57.851523   
7          tirex2  validation  1.882409  0.724310  52.939315   
8         xgboost  validation  2.059100  0.739080  57.908430   
9  seasonal_naive  validation  2.453167  0.917988  68.990847   

   series_origin_comparisons  
0                        500  
1                        500  
2                        500  
3                        500  
4                        500  
5                       1500  
6                       1500  
7                       1500  
8               

In [81]:
accuracy_by_class = create_accuracy_summary(
    daily_forecasts=performance_forecasts,
    series_metrics=series_origin_metrics,
    grouping_columns=[
        "model",
        "purpose",
        "demand_class"
    ]
)


accuracy_by_class = (
    accuracy_by_class
    .sort_values(
        [
            "purpose",
            "demand_class",
            "rmsse",
            "mae"
        ]
    )
    .reset_index(drop=True)
)


accuracy_by_class.to_csv(
    OUTPUT_DIR
    / "point_forecast_accuracy_by_demand_class.csv",
    index=False
)


print(accuracy_by_class[
    [
        "model",
        "purpose",
        "demand_class",
        "mae",
        "rmsse",
        "wape"
    ]
])


             model     purpose  demand_class       mae     rmsse        wape
0          xgboost  final_test       Erratic  3.158389  0.703717   55.018728
1           tirex2  final_test       Erratic  3.015541  0.715487   52.530332
2     arima_sarima  final_test       Erratic  3.225758  0.730397   56.192287
3              ets  final_test       Erratic  3.312354  0.737450   57.700778
4   seasonal_naive  final_test       Erratic  3.901714  0.889464   67.967350
5              ets  final_test  Intermittent  0.701999  0.773078   89.248017
6     arima_sarima  final_test  Intermittent  0.708075  0.785844   90.020372
7           tirex2  final_test  Intermittent  0.623229  0.824492   79.233676
8          xgboost  final_test  Intermittent  0.775780  0.856649   98.627981
9   seasonal_naive  final_test  Intermittent  0.843714  1.052360  107.264802
10             ets  final_test         Lumpy  1.527215  0.764866   80.695213
11    arima_sarima  final_test         Lumpy  1.526886  0.765913   80.677849

In [83]:
accuracy_by_origin = create_accuracy_summary(
    daily_forecasts=performance_forecasts,
    series_metrics=series_origin_metrics,
    grouping_columns=[
        "model",
        "purpose",
        "round",
        "forecast_origin"
    ]
)


accuracy_by_origin = (
    accuracy_by_origin
    .sort_values(
        [
            "forecast_origin",
            "rmsse",
            "mae"
        ]
    )
    .reset_index(drop=True)
)


accuracy_by_origin.to_csv(
    OUTPUT_DIR
    / "point_forecast_accuracy_by_origin.csv",
    index=False
)


print(accuracy_by_origin[
    [
        "model",
        "purpose",
        "round",
        "forecast_origin",
        "mae",
        "rmsse",
        "wape"
    ]
])


             model     purpose  round  forecast_origin       mae     rmsse  \
0     arima_sarima  validation      1             1829  2.060060  0.717740   
1              ets  validation      1             1829  2.102617  0.719857   
2           tirex2  validation      1             1829  1.915506  0.723995   
3          xgboost  validation      1             1829  2.092686  0.744163   
4   seasonal_naive  validation      1             1829  2.447714  0.897000   
5              ets  validation      2             1857  1.998526  0.724115   
6     arima_sarima  validation      2             1857  2.031635  0.728286   
7           tirex2  validation      2             1857  1.882857  0.739380   
8          xgboost  validation      2             1857  2.041677  0.745822   
9   seasonal_naive  validation      2             1857  2.408429  0.917735   
10             ets  validation      3             1885  2.030907  0.708936   
11          tirex2  validation      3             1885  1.848862

In [88]:
accuracy_checks = pd.DataFrame({
    "check": [
        "Expected performance forecast rows",
        "Expected series-origin rows",
        "Exactly five models",
        "Only validation and final test used",
        "Exactly 28 days per comparison",
        "No missing MAE",
        "No missing RMSSE",
        "Finite accuracy measures",
        "Non-negative accuracy measures"
    ],
    "passed": [
        len(performance_forecasts)
        == 5 * 500 * 4 * 28,

        len(series_origin_metrics)
        == 5 * 500 * 4,

        series_origin_metrics[
            "model"
        ].nunique() == 5,

        set(
            performance_forecasts[
                "purpose"
            ].unique()
        ) == {"validation", "final_test"},

        series_origin_metrics[
            "forecast_days"
        ].eq(28).all(),

        series_origin_metrics[
            "mae"
        ].notna().all(),

        series_origin_metrics[
            "rmsse"
        ].notna().all(),

        np.isfinite(
            accuracy_overall[
                ["mae", "rmsse", "wape"]
            ].to_numpy()
        ).all(),

        (
            accuracy_overall[
                ["mae", "rmsse", "wape"]
            ].to_numpy() >= 0
        ).all()
    ]
})


if not accuracy_checks["passed"].all():
    failed_checks = accuracy_checks.loc[
        ~accuracy_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Accuracy checks failed: {failed_checks}"
    )


accuracy_checks.to_csv(
    OUTPUT_DIR
    / "point_forecast_accuracy_checks.csv",
    index=False
)


print(accuracy_checks)    


                                 check  passed
0   Expected performance forecast rows    True
1          Expected series-origin rows    True
2                  Exactly five models    True
3  Only validation and final test used    True
4       Exactly 28 days per comparison    True
5                       No missing MAE    True
6                     No missing RMSSE    True
7             Finite accuracy measures    True
8       Non-negative accuracy measures    True


Evaluate TiRex-2

In [91]:
import numpy as np
import pandas as pd


required_objects = [
    "selected_sales",
    "demand_matrix",
    "OUTPUT_DIR"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

print("Missing objects:", missing_objects)


Missing objects: []


In [93]:
quantile_levels = np.arange(
    0.1,
    1.0,
    0.1
)

evaluation_schedule = [
    {
        "origin": 1829,
        "purpose": "validation",
        "round": 1
    },
    {
        "origin": 1857,
        "purpose": "validation",
        "round": 2
    },
    {
        "origin": 1885,
        "purpose": "validation",
        "round": 3
    },
    {
        "origin": 1913,
        "purpose": "final_test",
        "round": 1
    }
]

quantile_evaluation_parts = []
total_quantile_crossings = 0


for schedule_row in evaluation_schedule:

    origin = schedule_row["origin"]

    quantile_file = (
        OUTPUT_DIR
        / f"tirex2_quantiles_origin_{origin}.npy"
    )

    if not quantile_file.exists():
        raise FileNotFoundError(
            f"Missing TiRex-2 file: {quantile_file}"
        )

    quantile_forecasts = np.load(
        quantile_file
    )

    expected_shape = (
        500,
        9,
        28
    )

    if quantile_forecasts.shape != expected_shape:
        raise ValueError(
            f"Unexpected shape at d_{origin}: "
            f"{quantile_forecasts.shape}"
        )

    total_quantile_crossings += int(
        np.sum(
            np.diff(
                quantile_forecasts,
                axis=1
            ) < -1e-8
        )
    )

    actuals = demand_matrix[
        :,
        origin:origin + 28
    ]

    actual_cube = np.broadcast_to(
        actuals[:, None, :],
        quantile_forecasts.shape
    )

    quantile_cube = np.broadcast_to(
        quantile_levels[
            None,
            :,
            None
        ],
        quantile_forecasts.shape
    )

    errors = (
        actual_cube
        - quantile_forecasts
    )

    pinball_loss = np.maximum(
        quantile_cube * errors,
        (quantile_cube - 1) * errors
    )

    covered = (
        actual_cube
        <= quantile_forecasts
    )

    number_of_series = len(selected_sales)

    origin_results = pd.DataFrame({
        "id": np.repeat(
            selected_sales["id"].to_numpy(),
            9 * 28
        ),
        "demand_class": np.repeat(
            selected_sales[
                "demand_class"
            ].to_numpy(),
            9 * 28
        ),
        "model": "tirex2",
        "purpose": schedule_row["purpose"],
        "round": schedule_row["round"],
        "forecast_origin": origin,
        "quantile_level": np.tile(
            np.repeat(
                quantile_levels,
                28
            ),
            number_of_series
        ),
        "horizon": np.tile(
            np.arange(1, 29),
            number_of_series * 9
        ),
        "target_day": np.tile(
            np.tile(
                origin + np.arange(1, 29),
                9
            ),
            number_of_series
        ),
        "actual": actual_cube.reshape(-1),
        "quantile_forecast":
            quantile_forecasts.reshape(-1),
        "pinball_loss":
            pinball_loss.reshape(-1),
        "covered":
            covered.reshape(-1)
    })

    quantile_evaluation_parts.append(
        origin_results
    )

    print(
        f"Completed TiRex-2 quantile evaluation "
        f"at d_{origin}."
    )


tirex2_quantile_evaluation = pd.concat(
    quantile_evaluation_parts,
    ignore_index=True
)

print(
    "Evaluation rows:",
    len(tirex2_quantile_evaluation)
)

print(
    "Quantile crossings:",
    total_quantile_crossings
)


Completed TiRex-2 quantile evaluation at d_1829.
Completed TiRex-2 quantile evaluation at d_1857.
Completed TiRex-2 quantile evaluation at d_1885.
Completed TiRex-2 quantile evaluation at d_1913.
Evaluation rows: 504000
Quantile crossings: 2920


In [95]:
tirex2_quantile_overall = (
    tirex2_quantile_evaluation
    .groupby(
        [
            "purpose",
            "quantile_level"
        ],
        as_index=False
    )
    .agg(
        observations=(
            "pinball_loss",
            "size"
        ),
        mean_pinball_loss=(
            "pinball_loss",
            "mean"
        ),
        empirical_coverage=(
            "covered",
            "mean"
        ),
        mean_quantile_forecast=(
            "quantile_forecast",
            "mean"
        )
    )
)


tirex2_quantile_overall[
    "calibration_gap"
] = (
    tirex2_quantile_overall[
        "empirical_coverage"
    ]
    - tirex2_quantile_overall[
        "quantile_level"
    ]
)


tirex2_quantile_overall[
    "absolute_calibration_gap"
] = tirex2_quantile_overall[
    "calibration_gap"
].abs()


tirex2_quantile_overall.to_csv(
    OUTPUT_DIR
    / "tirex2_daily_quantile_accuracy_overall.csv",
    index=False
)


print(tirex2_quantile_overall[
    [
        "purpose",
        "quantile_level",
        "mean_pinball_loss",
        "empirical_coverage",
        "calibration_gap"
    ]
])


       purpose  quantile_level  mean_pinball_loss  empirical_coverage  \
0   final_test             0.1           0.306618            0.364214   
1   final_test             0.2           0.545092            0.395857   
2   final_test             0.3           0.730399            0.441571   
3   final_test             0.4           0.865834            0.495643   
4   final_test             0.5           0.950014            0.556286   
5   final_test             0.6           0.982539            0.628429   
6   final_test             0.7           0.956103            0.705786   
7   final_test             0.8           0.851550            0.786357   
8   final_test             0.9           0.625004            0.880429   
9   validation             0.1           0.303816            0.393238   
10  validation             0.2           0.540439            0.420024   
11  validation             0.3           0.723716            0.461167   
12  validation             0.4           0.857823  

In [97]:
tirex2_quantile_by_class = (
    tirex2_quantile_evaluation
    .groupby(
        [
            "purpose",
            "demand_class",
            "quantile_level"
        ],
        as_index=False
    )
    .agg(
        observations=(
            "pinball_loss",
            "size"
        ),
        mean_pinball_loss=(
            "pinball_loss",
            "mean"
        ),
        empirical_coverage=(
            "covered",
            "mean"
        ),
        mean_quantile_forecast=(
            "quantile_forecast",
            "mean"
        )
    )
)


tirex2_quantile_by_class[
    "calibration_gap"
] = (
    tirex2_quantile_by_class[
        "empirical_coverage"
    ]
    - tirex2_quantile_by_class[
        "quantile_level"
    ]
)


tirex2_quantile_by_class[
    "absolute_calibration_gap"
] = tirex2_quantile_by_class[
    "calibration_gap"
].abs()


tirex2_quantile_by_class.to_csv(
    OUTPUT_DIR
    / "tirex2_daily_quantile_accuracy_by_class.csv",
    index=False
)


In [99]:
tirex2_quantile_by_origin = (
    tirex2_quantile_evaluation
    .groupby(
        [
            "purpose",
            "round",
            "forecast_origin",
            "quantile_level"
        ],
        as_index=False
    )
    .agg(
        observations=(
            "pinball_loss",
            "size"
        ),
        mean_pinball_loss=(
            "pinball_loss",
            "mean"
        ),
        empirical_coverage=(
            "covered",
            "mean"
        )
    )
)


tirex2_quantile_by_origin[
    "calibration_gap"
] = (
    tirex2_quantile_by_origin[
        "empirical_coverage"
    ]
    - tirex2_quantile_by_origin[
        "quantile_level"
    ]
)


tirex2_quantile_by_origin.to_csv(
    OUTPUT_DIR
    / "tirex2_daily_quantile_accuracy_by_origin.csv",
    index=False
)


In [101]:
tirex2_mean_quantile_score = (
    tirex2_quantile_overall
    .groupby(
        "purpose",
        as_index=False
    )
    .agg(
        mean_pinball_loss=(
            "mean_pinball_loss",
            "mean"
        ),
        mean_absolute_calibration_gap=(
            "absolute_calibration_gap",
            "mean"
        )
    )
)


tirex2_mean_quantile_score.to_csv(
    OUTPUT_DIR
    / "tirex2_mean_quantile_score.csv",
    index=False
)


print(tirex2_mean_quantile_score)


      purpose  mean_pinball_loss  mean_absolute_calibration_gap
0  final_test           0.757017                       0.091222
1  validation           0.743433                       0.103730


In [106]:
crossing_rows = []

for schedule_row in evaluation_schedule:

    origin = schedule_row["origin"]

    quantile_forecasts = np.load(
        OUTPUT_DIR
        / f"tirex2_quantiles_origin_{origin}.npy"
    )

    adjacent_differences = np.diff(
        quantile_forecasts,
        axis=1
    )

    crossing_mask = (
        adjacent_differences < -1e-8
    )

    crossing_magnitudes = (
        -adjacent_differences[crossing_mask]
    )

    crossing_count = int(
        crossing_mask.sum()
    )

    comparisons = int(
        crossing_mask.size
    )

    affected_series_days = int(
        crossing_mask.any(axis=1).sum()
    )

    affected_series = int(
        crossing_mask.any(
            axis=(1, 2)
        ).sum()
    )

    crossing_rows.append({
        "purpose": schedule_row["purpose"],
        "round": schedule_row["round"],
        "forecast_origin": origin,
        "adjacent_quantile_comparisons":
            comparisons,
        "crossing_count":
            crossing_count,
        "crossing_rate_percent":
            100 * crossing_count / comparisons,
        "affected_series_days":
            affected_series_days,
        "affected_series":
            affected_series,
        "mean_crossing_magnitude": (
            float(crossing_magnitudes.mean())
            if crossing_count > 0
            else 0.0
        ),
        "maximum_crossing_magnitude": (
            float(crossing_magnitudes.max())
            if crossing_count > 0
            else 0.0
        )
    })


crossing_diagnostics = pd.DataFrame(
    crossing_rows
)


total_comparisons = crossing_diagnostics[
    "adjacent_quantile_comparisons"
].sum()

total_crossings = crossing_diagnostics[
    "crossing_count"
].sum()


print(crossing_diagnostics)

print("\nTotal crossings:", total_crossings)

print(
    "Overall crossing rate (%):",
    100 * total_crossings / total_comparisons
)


crossing_diagnostics.to_csv(
    OUTPUT_DIR
    / "tirex2_quantile_crossing_diagnostics.csv",
    index=False
)


      purpose  round  forecast_origin  adjacent_quantile_comparisons  \
0  validation      1             1829                         112000   
1  validation      2             1857                         112000   
2  validation      3             1885                         112000   
3  final_test      1             1913                         112000   

   crossing_count  crossing_rate_percent  affected_series_days  \
0             779               0.695536                   720   
1             605               0.540179                   589   
2             749               0.668750                   720   
3             787               0.702679                   762   

   affected_series  mean_crossing_magnitude  maximum_crossing_magnitude  
0               81                 0.001936                    0.022988  
1               70                 0.002587                    0.070198  
2               78                 0.002117                    0.019332  
3           

In [108]:
quantile_checks = pd.DataFrame({
    "check": [
        "Expected number of rows",
        "Exactly nine quantiles",
        "Exactly four evaluation origins",
        "No missing forecasts",
        "Finite quantile forecasts",
        "Non-negative quantile forecasts",
        "Non-negative pinball loss",
        "Coverage lies between zero and one",
        "Q0.50 pinball equals one-half MAE"
    ],
    "passed": [
        len(tirex2_quantile_evaluation)
        == 500 * 4 * 9 * 28,

        tirex2_quantile_evaluation[
            "quantile_level"
        ].nunique() == 9,

        tirex2_quantile_evaluation[
            "forecast_origin"
        ].nunique() == 4,

        tirex2_quantile_evaluation[
            "quantile_forecast"
        ].notna().all(),

        np.isfinite(
            tirex2_quantile_evaluation[
                "quantile_forecast"
            ].to_numpy()
        ).all(),

        (
            tirex2_quantile_evaluation[
                "quantile_forecast"
            ] >= 0
        ).all(),

        (
            tirex2_quantile_evaluation[
                "pinball_loss"
            ] >= 0
        ).all(),

        tirex2_quantile_evaluation[
            "covered"
        ].isin([True, False]).all(),

        np.allclose(
            q50_check["mean_pinball_loss"],
            q50_check["expected_q50_pinball"],
            atol=1e-8
        )
    ]
})


if not quantile_checks["passed"].all():

    failed_checks = quantile_checks.loc[
        ~quantile_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Quantile checks failed: {failed_checks}"
    )


tirex2_quantile_evaluation.to_csv(
    OUTPUT_DIR
    / "tirex2_daily_quantile_evaluation_rows.csv",
    index=False
)


quantile_checks.to_csv(
    OUTPUT_DIR
    / "tirex2_daily_quantile_checks.csv",
    index=False
)


print(quantile_checks)


                                check  passed
0             Expected number of rows    True
1              Exactly nine quantiles    True
2     Exactly four evaluation origins    True
3                No missing forecasts    True
4           Finite quantile forecasts    True
5     Non-negative quantile forecasts    True
6           Non-negative pinball loss    True
7  Coverage lies between zero and one    True
8   Q0.50 pinball equals one-half MAE    True


Construction of the common residual-based 28-day quantiles at levels 0.90, 0.95 and 0.99

In [111]:
PROTECTION_PERIOD = 28

all_origins = sorted(
    all_models_forecasts[
        "forecast_origin"
    ].unique()
)

scale_rows = []


for series_index, row in (
    selected_sales
    .reset_index(drop=True)
    .iterrows()
):

    active_start_index = (
        int(row["first_active_day"]) - 1
    )

    for origin in all_origins:

        historical_demand = demand_matrix[
            series_index,
            active_start_index:origin
        ]

        rolling_totals = np.convolve(
            historical_demand,
            np.ones(PROTECTION_PERIOD),
            mode="valid"
        )

        if len(rolling_totals) < 2:
            historical_sd = 0.0
        else:
            historical_sd = np.std(
                rolling_totals,
                ddof=1
            )

        protection_scale = max(
            1.0,
            float(historical_sd)
        )

        scale_rows.append({
            "id": row["id"],
            "forecast_origin": origin,
            "protection_scale":
                protection_scale
        })


protection_scales = pd.DataFrame(
    scale_rows
)


print("Scale rows:", len(protection_scales))
print(
    "Minimum scale:",
    protection_scales[
        "protection_scale"
    ].min()
)


Scale rows: 5000
Minimum scale: 1.0


In [113]:
protection_period_errors = (
    all_models_forecasts
    .groupby(
        [
            "model",
            "id",
            "demand_class",
            "purpose",
            "round",
            "forecast_origin"
        ],
        as_index=False
    )
    .agg(
        actual_protection_demand=(
            "actual",
            "sum"
        ),
        forecast_protection_demand=(
            "forecast",
            "sum"
        ),
        forecast_days=(
            "horizon",
            "count"
        )
    )
)


protection_period_errors[
    "protection_period_error"
] = (
    protection_period_errors[
        "actual_protection_demand"
    ]
    - protection_period_errors[
        "forecast_protection_demand"
    ]
)


protection_period_errors = (
    protection_period_errors.merge(
        protection_scales,
        on=[
            "id",
            "forecast_origin"
        ],
        how="left",
        validate="many_to_one"
    )
)


protection_period_errors[
    "standardised_error"
] = (
    protection_period_errors[
        "protection_period_error"
    ]
    / protection_period_errors[
        "protection_scale"
    ]
)


protection_period_errors[
    "error_available_day"
] = (
    protection_period_errors[
        "forecast_origin"
    ]
    + PROTECTION_PERIOD
)


protection_period_errors.to_csv(
    OUTPUT_DIR
    / "protection_period_errors.csv",
    index=False
)


print(
    "Protection-period error rows:",
    len(protection_period_errors)
)


Protection-period error rows: 25000


In [115]:
RESIDUAL_QUANTILE_LEVELS = [
    0.90,
    0.95,
    0.99
]


residual_schedule = pd.DataFrame({
    "decision_origin": [
        1829,
        1857,
        1885,
        1913
    ],
    "purpose": [
        "validation",
        "validation",
        "validation",
        "final_test"
    ],
    "round": [
        1,
        2,
        3,
        1
    ]
})


pool_quantile_rows = []
pool_membership_parts = []


for schedule_row in residual_schedule.itertuples(
    index=False
):

    decision_origin = int(
        schedule_row.decision_origin
    )

    # Only errors whose entire 28-day period has ended
    # may enter the pool.
    available_errors = (
        protection_period_errors[
            protection_period_errors[
                "error_available_day"
            ] <= decision_origin
        ]
        .copy()
    )

    available_errors["decision_origin"] = (
        decision_origin
    )

    pool_membership_parts.append(
        available_errors[
            [
                "decision_origin",
                "model",
                "demand_class",
                "id",
                "forecast_origin",
                "error_available_day",
                "standardised_error"
            ]
        ]
    )

    for (
        model_name,
        demand_class
    ), group in available_errors.groupby(
        [
            "model",
            "demand_class"
        ]
    ):

        standardised_errors = group[
            "standardised_error"
        ].to_numpy(dtype=float)

        for quantile_level in (
            RESIDUAL_QUANTILE_LEVELS
        ):

            pooled_quantile = np.quantile(
                standardised_errors,
                quantile_level,
                method="linear"
            )

            pool_quantile_rows.append({
                "decision_origin":
                    decision_origin,

                "purpose":
                    schedule_row.purpose,

                "round":
                    int(schedule_row.round),

                "model":
                    model_name,

                "demand_class":
                    demand_class,

                "quantile_level":
                    quantile_level,

                "pool_size":
                    len(standardised_errors),

                "standardised_error_quantile":
                    pooled_quantile
            })


residual_pool_quantiles = pd.DataFrame(
    pool_quantile_rows
)


residual_pool_membership = pd.concat(
    pool_membership_parts,
    ignore_index=True
)


residual_pool_quantiles.to_csv(
    OUTPUT_DIR
    / "residual_pool_quantiles.csv",
    index=False
)


residual_pool_membership.to_csv(
    OUTPUT_DIR
    / "residual_pool_membership.csv",
    index=False
)


In [117]:
pool_sizes = (
    residual_pool_quantiles[
        [
            "decision_origin",
            "model",
            "demand_class",
            "pool_size"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "decision_origin",
            "model",
            "demand_class"
        ]
    )
)


print(
    pool_sizes.groupby(
        "decision_origin"
    )["pool_size"].unique()
)


decision_origin
1829     [750]
1857     [875]
1885    [1000]
1913    [1125]
Name: pool_size, dtype: object


In [119]:
constructed_quantile_parts = []


for schedule_row in residual_schedule.itertuples(
    index=False
):

    origin = int(
        schedule_row.decision_origin
    )

    current_forecasts = (
        protection_period_errors[
            protection_period_errors[
                "forecast_origin"
            ] == origin
        ][
            [
                "model",
                "id",
                "demand_class",
                "purpose",
                "round",
                "forecast_origin",
                "actual_protection_demand",
                "forecast_protection_demand",
                "protection_scale"
            ]
        ]
        .copy()
    )


    current_pool_quantiles = (
        residual_pool_quantiles[
            residual_pool_quantiles[
                "decision_origin"
            ] == origin
        ][
            [
                "model",
                "demand_class",
                "quantile_level",
                "pool_size",
                "standardised_error_quantile"
            ]
        ]
    )


    constructed = current_forecasts.merge(
        current_pool_quantiles,
        on=[
            "model",
            "demand_class"
        ],
        how="left",
        validate="many_to_many"
    )


    constructed["safety_stock"] = np.maximum(
        0.0,
        constructed["protection_scale"]
        * constructed[
            "standardised_error_quantile"
        ]
    )


    constructed[
        "protection_quantile_forecast"
    ] = np.maximum(
        0.0,
        constructed[
            "forecast_protection_demand"
        ]
        + constructed["safety_stock"]
    )


    cumulative_error = (
        constructed[
            "actual_protection_demand"
        ]
        - constructed[
            "protection_quantile_forecast"
        ]
    )


    alpha = constructed[
        "quantile_level"
    ]


    constructed[
        "cumulative_pinball_loss"
    ] = np.maximum(
        alpha * cumulative_error,
        (alpha - 1) * cumulative_error
    )


    constructed["covered"] = (
        constructed[
            "actual_protection_demand"
        ]
        <= constructed[
            "protection_quantile_forecast"
        ]
    )


    constructed_quantile_parts.append(
        constructed
    )


residual_based_quantiles = pd.concat(
    constructed_quantile_parts,
    ignore_index=True
)


residual_based_quantiles.to_csv(
    OUTPUT_DIR
    / "residual_based_protection_quantiles.csv",
    index=False
)


print(
    "Constructed quantile rows:",
    len(residual_based_quantiles)
)


Constructed quantile rows: 30000


In [121]:
protection_quantile_accuracy = (
    residual_based_quantiles
    .groupby(
        [
            "model",
            "purpose",
            "quantile_level"
        ],
        as_index=False
    )
    .agg(
        protection_periods=(
            "cumulative_pinball_loss",
            "size"
        ),
        mean_cumulative_pinball_loss=(
            "cumulative_pinball_loss",
            "mean"
        ),
        empirical_coverage=(
            "covered",
            "mean"
        ),
        mean_safety_stock=(
            "safety_stock",
            "mean"
        ),
        mean_inventory_target=(
            "protection_quantile_forecast",
            "mean"
        )
    )
)


protection_quantile_accuracy[
    "calibration_gap"
] = (
    protection_quantile_accuracy[
        "empirical_coverage"
    ]
    - protection_quantile_accuracy[
        "quantile_level"
    ]
)


protection_quantile_accuracy.to_csv(
    OUTPUT_DIR
    / "protection_quantile_accuracy_overall.csv",
    index=False
)


print(protection_quantile_accuracy[
    [
        "model",
        "purpose",
        "quantile_level",
        "mean_cumulative_pinball_loss",
        "empirical_coverage",
        "calibration_gap",
        "mean_safety_stock"
    ]
])


             model     purpose  quantile_level  mean_cumulative_pinball_loss  \
0     arima_sarima  final_test            0.90                      7.431023   
1     arima_sarima  final_test            0.95                      5.312972   
2     arima_sarima  final_test            0.99                      1.943645   
3     arima_sarima  validation            0.90                      6.684289   
4     arima_sarima  validation            0.95                      4.502351   
5     arima_sarima  validation            0.99                      1.556240   
6              ets  final_test            0.90                      7.160143   
7              ets  final_test            0.95                      4.945357   
8              ets  final_test            0.99                      1.645132   
9              ets  validation            0.90                      7.525343   
10             ets  validation            0.95                      5.012197   
11             ets  validation          

In [123]:
expected_pool_sizes = {
    1829: 750,
    1857: 875,
    1885: 1000,
    1913: 1125
}


pool_size_check = all(
    row.pool_size
    == expected_pool_sizes[
        row.decision_origin
    ]

    for row in pool_sizes.itertuples(
        index=False
    )
)


residual_checks = pd.DataFrame({
    "check": [
        "Expected number of error rows",
        "Exactly 28 days per error",
        "Positive finite scales",
        "Finite standardised errors",
        "Correct sequential pool sizes",
        "No future errors in any pool",
        "Expected number of quantile rows",
        "Exactly three quantile levels",
        "No missing constructed quantiles",
        "Finite constructed quantiles",
        "Non-negative safety stock",
        "Quantile target not below central forecast",
        "Non-negative cumulative pinball loss"
    ],
    "passed": [
        len(protection_period_errors)
        == 5 * 500 * 10,

        protection_period_errors[
            "forecast_days"
        ].eq(28).all(),

        (
            np.isfinite(
                protection_scales[
                    "protection_scale"
                ]
            ).all()
            and (
                protection_scales[
                    "protection_scale"
                ] >= 1
            ).all()
        ),

        np.isfinite(
            protection_period_errors[
                "standardised_error"
            ].to_numpy()
        ).all(),

        pool_size_check,

        (
            residual_pool_membership[
                "error_available_day"
            ]
            <= residual_pool_membership[
                "decision_origin"
            ]
        ).all(),

        len(residual_based_quantiles)
        == 5 * 500 * 4 * 3,

        residual_based_quantiles[
            "quantile_level"
        ].nunique() == 3,

        residual_based_quantiles[
            "protection_quantile_forecast"
        ].notna().all(),

        np.isfinite(
            residual_based_quantiles[
                "protection_quantile_forecast"
            ].to_numpy()
        ).all(),

        (
            residual_based_quantiles[
                "safety_stock"
            ] >= 0
        ).all(),

        (
            residual_based_quantiles[
                "protection_quantile_forecast"
            ]
            >= residual_based_quantiles[
                "forecast_protection_demand"
            ]
        ).all(),

        (
            residual_based_quantiles[
                "cumulative_pinball_loss"
            ] >= 0
        ).all()
    ]
})


if not residual_checks["passed"].all():

    failed_checks = residual_checks.loc[
        ~residual_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Residual checks failed: {failed_checks}"
    )


residual_checks.to_csv(
    OUTPUT_DIR
    / "residual_based_quantile_checks.csv",
    index=False
)


print(residual_checks)


                                         check  passed
0                Expected number of error rows    True
1                    Exactly 28 days per error    True
2                       Positive finite scales    True
3                   Finite standardised errors    True
4                Correct sequential pool sizes    True
5                 No future errors in any pool    True
6             Expected number of quantile rows    True
7                Exactly three quantile levels    True
8             No missing constructed quantiles    True
9                 Finite constructed quantiles    True
10                   Non-negative safety stock    True
11  Quantile target not below central forecast    True
12        Non-negative cumulative pinball loss    True


Generate the 12 weekly origins

In [126]:
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

DATA_DIR = Path(os.environ.get("M5_DATA_DIR", Path.home() / "Desktop" / "m5_data"))
OUTPUT_DIR = DATA_DIR / "chapter5_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SALES_FILE = DATA_DIR / "sales_train_evaluation.csv"
CALENDAR_FILE = DATA_DIR / "calendar.csv"
PRICES_FILE = DATA_DIR / "sell_prices.csv"

sample_candidates = list(DATA_DIR.rglob("selected_sample_ids.csv"))

if not sample_candidates:
    raise FileNotFoundError(
        "selected_sample_ids.csv was not found anywhere inside m5_data."
    )

SAMPLE_FILE = sample_candidates[0]

print("Sample file:", SAMPLE_FILE)


# ------------------------------------------------------------
# 2. Load the selected sample and M5 data
# ------------------------------------------------------------

sales = pd.read_csv(SALES_FILE)
sample = pd.read_csv(SAMPLE_FILE)
calendar = pd.read_csv(CALENDAR_FILE)
prices = pd.read_csv(PRICES_FILE)

if "demand_pattern" in sample.columns:
    sample = sample.rename(
        columns={"demand_pattern": "demand_class"}
    )

required_sample_columns = {"id", "demand_class"}

if not required_sample_columns.issubset(sample.columns):
    raise ValueError(
        "The sample file must contain id and demand_class."
    )

sample_columns = ["id", "demand_class"]

if "first_active_day" in sample.columns:
    sample_columns.append("first_active_day")

selected_sales = (
    sample[sample_columns]
    .merge(
        sales,
        on="id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("id")
    .reset_index(drop=True)
)

day_columns = sorted(
    [
        column
        for column in sales.columns
        if column.startswith("d_")
    ],
    key=lambda column: int(column.split("_")[1])
)

demand_matrix = selected_sales[day_columns].to_numpy(
    dtype=float
)

MAX_OBSERVED_DAY = len(day_columns)

print("Selected series:", len(selected_sales))
print("Observed demand ends at: d_" + str(MAX_OBSERVED_DAY))


# ------------------------------------------------------------
# 3. Define the weekly review origins
# ------------------------------------------------------------

WEEKLY_REVIEW_ORIGINS = [
    1829, 1836, 1843, 1850,
    1857, 1864, 1871, 1878,
    1885, 1892, 1899, 1906,
    1913, 1920, 1927, 1934
]

EXISTING_ORIGINS = [
    1829,
    1857,
    1885,
    1913
]

ADDITIONAL_WEEKLY_ORIGINS = [
    origin
    for origin in WEEKLY_REVIEW_ORIGINS
    if origin not in EXISTING_ORIGINS
]

FORECAST_HORIZON = 28
SEASONAL_PERIOD = 7

print("Additional origins:", ADDITIONAL_WEEKLY_ORIGINS)
print("Number of additional origins:",
      len(ADDITIONAL_WEEKLY_ORIGINS))


# ------------------------------------------------------------
# 4. Helper function for the common forecast format
# ------------------------------------------------------------

def build_weekly_forecast_frame(
    predictions,
    origin,
    model_name
):
    predictions = np.asarray(predictions, dtype=float)

    expected_shape = (
        len(selected_sales),
        FORECAST_HORIZON
    )

    if predictions.shape != expected_shape:
        raise ValueError(
            f"Forecast shape {predictions.shape}; "
            f"expected {expected_shape}."
        )

    horizons = np.arange(1, FORECAST_HORIZON + 1)
    target_days = origin + horizons

    # Actual demand is available only through d_1941.
    actuals = np.full(
        expected_shape,
        np.nan,
        dtype=float
    )

    available_days = min(
        FORECAST_HORIZON,
        MAX_OBSERVED_DAY - origin
    )

    if available_days > 0:
        actuals[:, :available_days] = demand_matrix[
            :,
            origin:origin + available_days
        ]

    purpose = (
        "inventory_warmup"
        if origin < 1913
        else "inventory_test"
    )

    review_number = (
        WEEKLY_REVIEW_ORIGINS.index(origin) + 1
    )

    frame = pd.DataFrame({
        "id": np.repeat(
            selected_sales["id"].to_numpy(),
            FORECAST_HORIZON
        ),
        "demand_class": np.repeat(
            selected_sales["demand_class"].to_numpy(),
            FORECAST_HORIZON
        ),
        "model": model_name,
        "purpose": purpose,
        "round": review_number,
        "forecast_origin": origin,
        "forecast_origin_label": f"d_{origin}",
        "horizon": np.tile(
            horizons,
            len(selected_sales)
        ),
        "target_day": np.tile(
            target_days,
            len(selected_sales)
        ),
        "actual": actuals.reshape(-1),
        "forecast": np.maximum(
            predictions,
            0.0
        ).reshape(-1)
    })

    frame["target_day_label"] = (
        "d_" + frame["target_day"].astype(str)
    )

    return frame[
        [
            "id",
            "demand_class",
            "model",
            "purpose",
            "round",
            "forecast_origin",
            "forecast_origin_label",
            "horizon",
            "target_day",
            "target_day_label",
            "actual",
            "forecast"
        ]
    ]


# ------------------------------------------------------------
# 5. Generate additional seasonal-naïve forecasts
# ------------------------------------------------------------

seasonal_naive_parts = []

for origin in ADDITIONAL_WEEKLY_ORIGINS:

    origin_file = (
        OUTPUT_DIR
        / f"inventory_weekly_seasonal_naive_origin_{origin}.csv"
    )

    # Reuse a valid checkpoint if it already exists.
    if origin_file.exists():
        origin_forecasts = pd.read_csv(origin_file)

        valid_checkpoint = (
            len(origin_forecasts)
            == len(selected_sales) * FORECAST_HORIZON
            and origin_forecasts["id"].nunique()
            == len(selected_sales)
            and origin_forecasts["forecast_origin"].eq(origin).all()
            and origin_forecasts["forecast"].notna().all()
        )

        if valid_checkpoint:
            seasonal_naive_parts.append(origin_forecasts)

            print(
                f"Seasonal naïve, d_{origin}: "
                "existing checkpoint loaded."
            )
            continue

    training_values = demand_matrix[:, :origin]

    last_week = training_values[
        :,
        -SEASONAL_PERIOD:
    ]

    seasonal_naive_predictions = np.tile(
        last_week,
        (
            1,
            int(
                np.ceil(
                    FORECAST_HORIZON
                    / SEASONAL_PERIOD
                )
            )
        )
    )[:, :FORECAST_HORIZON]

    seasonal_naive_predictions = np.maximum(
        seasonal_naive_predictions,
        0.0
    )

    origin_forecasts = build_weekly_forecast_frame(
        predictions=seasonal_naive_predictions,
        origin=origin,
        model_name="seasonal_naive"
    )

    origin_forecasts.to_csv(
        origin_file,
        index=False
    )

    seasonal_naive_parts.append(origin_forecasts)

    print(
        f"Seasonal naïve: completed origin d_{origin}."
    )


# ------------------------------------------------------------
# 6. Combine and check the additional forecasts
# ------------------------------------------------------------

seasonal_naive_weekly = pd.concat(
    seasonal_naive_parts,
    ignore_index=True
)

expected_rows = (
    len(selected_sales)
    * len(ADDITIONAL_WEEKLY_ORIGINS)
    * FORECAST_HORIZON
)

group_sizes = seasonal_naive_weekly.groupby(
    ["id", "forecast_origin"]
).size()

seasonal_naive_weekly_checks = pd.DataFrame({
    "check": [
        "Expected number of rows",
        "Exactly 500 series",
        "Exactly twelve additional origins",
        "28 rows per series and origin",
        "No missing forecasts",
        "Finite forecasts",
        "Non-negative forecasts",
        "Target day equals origin plus horizon",
        "Missing actuals occur only after d_1941"
    ],
    "passed": [
        len(seasonal_naive_weekly) == expected_rows,
        seasonal_naive_weekly["id"].nunique()
        == len(selected_sales),
        seasonal_naive_weekly["forecast_origin"].nunique()
        == len(ADDITIONAL_WEEKLY_ORIGINS),
        group_sizes.eq(FORECAST_HORIZON).all(),
        seasonal_naive_weekly["forecast"].notna().all(),
        np.isfinite(
            seasonal_naive_weekly["forecast"]
        ).all(),
        seasonal_naive_weekly["forecast"].ge(0).all(),
        (
            seasonal_naive_weekly["target_day"]
            == seasonal_naive_weekly["forecast_origin"]
            + seasonal_naive_weekly["horizon"]
        ).all(),
        (
            seasonal_naive_weekly["actual"].isna()
            == seasonal_naive_weekly["target_day"].gt(
                MAX_OBSERVED_DAY
            )
        ).all()
    ]
})

if not seasonal_naive_weekly_checks["passed"].all():
    failed_checks = seasonal_naive_weekly_checks.loc[
        ~seasonal_naive_weekly_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Weekly seasonal-naïve checks failed: "
        f"{failed_checks}"
    )

combined_file = (
    OUTPUT_DIR
    / "inventory_weekly_seasonal_naive_additional_forecasts.csv"
)

checks_file = (
    OUTPUT_DIR
    / "inventory_weekly_seasonal_naive_checks.csv"
)

seasonal_naive_weekly.to_csv(
    combined_file,
    index=False
)

seasonal_naive_weekly_checks.to_csv(
    checks_file,
    index=False
)

print("\nSeasonal-naïve weekly forecasts completed.")
print("Forecast rows:", len(seasonal_naive_weekly))
print("\nChecks:")
print(seasonal_naive_weekly_checks)
print("\nSaved file:")
print(combined_file)


Sample file: <M5_DATA_DIR>/selected_sample_ids.csv
Selected series: 500
Observed demand ends at: d_1941
Additional origins: [1836, 1843, 1850, 1864, 1871, 1878, 1892, 1899, 1906, 1920, 1927, 1934]
Number of additional origins: 12
Seasonal naïve: completed origin d_1836.
Seasonal naïve: completed origin d_1843.
Seasonal naïve: completed origin d_1850.
Seasonal naïve: completed origin d_1864.
Seasonal naïve: completed origin d_1871.
Seasonal naïve: completed origin d_1878.
Seasonal naïve: completed origin d_1892.
Seasonal naïve: completed origin d_1899.
Seasonal naïve: completed origin d_1906.
Seasonal naïve: completed origin d_1920.
Seasonal naïve: completed origin d_1927.
Seasonal naïve: completed origin d_1934.

Seasonal-naïve weekly forecasts completed.
Forecast rows: 168000

Checks:
                                     check  passed
0                  Expected number of rows    True
1                       Exactly 500 series    True
2        Exactly twelve additional origins    True

In [130]:
# ------------------------------------------------------------
# Reconstruct first_active_day from the M5 price history
# ------------------------------------------------------------

calendar_for_active_period = calendar.copy()

calendar_for_active_period["day_number"] = (
    calendar_for_active_period["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

# First M5 day belonging to each wm_yr_wk.
first_day_by_week = (
    calendar_for_active_period
    .groupby("wm_yr_wk")["day_number"]
    .min()
)

# Retain only valid, strictly positive prices.
valid_prices = prices.loc[
    prices["sell_price"].notna()
    & np.isfinite(prices["sell_price"])
    & prices["sell_price"].gt(0)
].copy()

# Find the first valid-price week for every SKU–store series.
first_active_periods = (
    valid_prices
    .groupby(
        ["store_id", "item_id"],
        as_index=False
    )["wm_yr_wk"]
    .min()
    .rename(
        columns={
            "wm_yr_wk": "first_price_week"
        }
    )
)

first_active_periods["first_active_day"] = (
    first_active_periods["first_price_week"]
    .map(first_day_by_week)
)

# Prevent duplicate columns if this correction cell is rerun.
columns_to_remove = [
    column
    for column in [
        "first_price_week",
        "first_active_day"
    ]
    if column in selected_sales.columns
]

if columns_to_remove:
    selected_sales = selected_sales.drop(
        columns=columns_to_remove
    )

selected_sales = selected_sales.merge(
    first_active_periods[
        [
            "store_id",
            "item_id",
            "first_price_week",
            "first_active_day"
        ]
    ],
    on=["store_id", "item_id"],
    how="left",
    validate="one_to_one"
)

missing_active_days = (
    selected_sales["first_active_day"].isna().sum()
)

if missing_active_days > 0:
    missing_ids = selected_sales.loc[
        selected_sales["first_active_day"].isna(),
        "id"
    ].head(10).tolist()

    raise ValueError(
        f"{missing_active_days} selected series have no "
        f"valid first active day. Examples: {missing_ids}"
    )

selected_sales["first_active_day"] = (
    selected_sales["first_active_day"].astype(int)
)

# Recreate the demand matrix to preserve row alignment.
demand_matrix = selected_sales[
    day_columns
].to_numpy(dtype=float)

print("first_active_day successfully reconstructed.")
print(
    "Earliest active day:",
    selected_sales["first_active_day"].min()
)
print(
    "Latest active day:",
    selected_sales["first_active_day"].max()
)
print(
    "Missing active days:",
    selected_sales["first_active_day"].isna().sum()
)
print(
    "Demand matrix shape:",
    demand_matrix.shape
)


first_active_day successfully reconstructed.
Earliest active day: 1
Latest active day: 1093
Missing active days: 0
Demand matrix shape: (500, 1941)


In [132]:
import time
import warnings

import numpy as np
import pandas as pd

from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.exponential_smoothing.ets import ETSModel


# ------------------------------------------------------------
# 1. Automatic ETS fitting
# ------------------------------------------------------------

def fit_weekly_automatic_ets(training_values, horizon=28):

    candidates = [
        {
            "trend": None,
            "damped_trend": False,
            "name": "no_trend"
        },
        {
            "trend": "add",
            "damped_trend": False,
            "name": "additive_trend"
        },
        {
            "trend": "add",
            "damped_trend": True,
            "name": "damped_additive_trend"
        }
    ]

    successful_models = []
    failure_messages = []

    for candidate in candidates:

        try:
            with warnings.catch_warnings():
                warnings.simplefilter(
                    "ignore",
                    ConvergenceWarning
                )

                model = ETSModel(
                    training_values,
                    error="add",
                    trend=candidate["trend"],
                    damped_trend=candidate["damped_trend"],
                    seasonal="add",
                    seasonal_periods=7,
                    initialization_method="estimated"
                )

                result = model.fit(
                    maxiter=1000,
                    disp=False
                )

            converged = bool(
                result.mle_retvals.get(
                    "converged",
                    False
                )
            )

            aicc = float(result.aicc)

            if np.isfinite(aicc):
                successful_models.append({
                    "result": result,
                    "specification": candidate["name"],
                    "trend": candidate["trend"],
                    "damped_trend":
                        candidate["damped_trend"],
                    "aicc": aicc,
                    "converged": converged
                })

        except Exception as error:
            failure_messages.append(
                f"{candidate['name']}: "
                f"{type(error).__name__}: {error}"
            )

    if not successful_models:
        raise RuntimeError(
            "All ETS specifications failed. "
            + " | ".join(failure_messages)
        )

    converged_models = [
        candidate
        for candidate in successful_models
        if candidate["converged"]
    ]

    eligible_models = (
        converged_models
        if converged_models
        else successful_models
    )

    selected_model = min(
        eligible_models,
        key=lambda candidate: candidate["aicc"]
    )

    forecast = np.asarray(
        selected_model["result"].forecast(horizon),
        dtype=float
    )

    forecast = np.maximum(forecast, 0.0)

    diagnostics = {
        "selected_specification":
            selected_model["specification"],
        "trend": str(selected_model["trend"]),
        "damped_trend":
            selected_model["damped_trend"],
        "aicc": selected_model["aicc"],
        "converged": selected_model["converged"],
        "used_nonconverged_candidate":
            len(converged_models) == 0,
        "candidate_failures":
            " | ".join(failure_messages)
    }

    return forecast, diagnostics


# ------------------------------------------------------------
# 2. Construct the output rows for one series
# ------------------------------------------------------------

def build_single_series_weekly_frame(
    series_index,
    origin,
    predictions,
    model_name
):

    predictions = np.asarray(
        predictions,
        dtype=float
    )

    if predictions.shape != (FORECAST_HORIZON,):
        raise ValueError(
            f"Forecast has shape {predictions.shape}; "
            f"expected ({FORECAST_HORIZON},)."
        )

    row = selected_sales.iloc[series_index]
    horizons = np.arange(1, FORECAST_HORIZON + 1)
    target_days = origin + horizons

    actuals = np.full(
        FORECAST_HORIZON,
        np.nan
    )

    available_days = min(
        FORECAST_HORIZON,
        MAX_OBSERVED_DAY - origin
    )

    if available_days > 0:
        actuals[:available_days] = demand_matrix[
            series_index,
            origin:origin + available_days
        ]

    purpose = (
        "inventory_warmup"
        if origin < 1913
        else "inventory_test"
    )

    review_number = (
        WEEKLY_REVIEW_ORIGINS.index(origin) + 1
    )

    frame = pd.DataFrame({
        "id": row["id"],
        "demand_class": row["demand_class"],
        "model": model_name,
        "purpose": purpose,
        "round": review_number,
        "forecast_origin": origin,
        "forecast_origin_label": f"d_{origin}",
        "horizon": horizons,
        "target_day": target_days,
        "target_day_label": [
            f"d_{day}" for day in target_days
        ],
        "actual": actuals,
        "forecast": np.maximum(
            predictions,
            0.0
        )
    })

    return frame


# ------------------------------------------------------------
# 3. Run ETS with resumable checkpoints
# ------------------------------------------------------------

def run_additional_weekly_ets(
    checkpoint_interval=25
):

    all_origin_forecasts = []
    all_origin_diagnostics = []

    total_start_time = time.perf_counter()

    for origin in ADDITIONAL_WEEKLY_ORIGINS:

        forecast_file = (
            OUTPUT_DIR
            / f"inventory_weekly_ets_origin_{origin}.csv"
        )

        diagnostics_file = (
            OUTPUT_DIR
            / f"inventory_weekly_ets_diagnostics_origin_{origin}.csv"
        )

        # Load any existing checkpoint.
        if forecast_file.exists():
            existing_forecasts = pd.read_csv(
                forecast_file
            )

            valid_groups = (
                existing_forecasts
                .groupby("id")
                .agg(
                    rows=("horizon", "size"),
                    valid_forecast=(
                        "forecast",
                        lambda values:
                            values.notna().all()
                    )
                )
            )

            completed_ids = set(
                valid_groups.loc[
                    (valid_groups["rows"] == 28)
                    & valid_groups["valid_forecast"],
                ].index
            )

            existing_forecasts = (
                existing_forecasts.loc[
                    existing_forecasts["id"].isin(
                        completed_ids
                    )
                ]
                .drop_duplicates(
                    subset=[
                        "id",
                        "forecast_origin",
                        "horizon"
                    ],
                    keep="last"
                )
            )

        else:
            existing_forecasts = pd.DataFrame()
            completed_ids = set()

        if diagnostics_file.exists():
            existing_diagnostics = pd.read_csv(
                diagnostics_file
            )

            existing_diagnostics = (
                existing_diagnostics.loc[
                    existing_diagnostics["id"].isin(
                        completed_ids
                    )
                ]
                .drop_duplicates(
                    subset=[
                        "id",
                        "forecast_origin"
                    ],
                    keep="last"
                )
            )

        else:
            existing_diagnostics = pd.DataFrame()

        if len(completed_ids) == len(selected_sales):
            print(
                f"ETS, d_{origin}: "
                "complete checkpoint loaded."
            )

            all_origin_forecasts.append(
                existing_forecasts
            )

            all_origin_diagnostics.append(
                existing_diagnostics
            )

            continue

        new_forecast_parts = []
        new_diagnostic_rows = []

        for series_index, row in (
            selected_sales.iterrows()
        ):

            series_id = row["id"]

            if series_id in completed_ids:
                continue

            fitting_start = time.perf_counter()

            active_start = int(
                row["first_active_day"]
            )

            training_values = demand_matrix[
                series_index,
                active_start - 1:origin
            ]

            fallback_used = False
            error_message = ""

            try:
                predictions, diagnostics = (
                    fit_weekly_automatic_ets(
                        training_values,
                        horizon=FORECAST_HORIZON
                    )
                )

            except Exception as error:
                # Transparent seasonal-naïve fallback.
                last_week = training_values[-7:]

                predictions = np.tile(
                    last_week,
                    int(
                        np.ceil(
                            FORECAST_HORIZON / 7
                        )
                    )
                )[:FORECAST_HORIZON]

                predictions = np.maximum(
                    predictions,
                    0.0
                )

                diagnostics = {}
                fallback_used = True

                error_message = (
                    f"{type(error).__name__}: {error}"
                )

            forecast_part = (
                build_single_series_weekly_frame(
                    series_index=series_index,
                    origin=origin,
                    predictions=predictions,
                    model_name="ets"
                )
            )

            new_forecast_parts.append(
                forecast_part
            )

            new_diagnostic_rows.append({
                "id": series_id,
                "demand_class":
                    row["demand_class"],
                "model": "ets",
                "forecast_origin": origin,
                "active_training_start":
                    active_start,
                "training_observations":
                    len(training_values),
                "fit_seconds":
                    time.perf_counter()
                    - fitting_start,
                "fallback_used":
                    fallback_used,
                "error_message":
                    error_message,
                **diagnostics
            })

            completed_count = (
                len(completed_ids)
                + len(new_diagnostic_rows)
            )

            # Save after every 25 newly completed series.
            if (
                completed_count
                % checkpoint_interval == 0
            ):
                current_forecasts = pd.concat(
                    [
                        existing_forecasts,
                        *new_forecast_parts
                    ],
                    ignore_index=True
                )

                current_forecasts = (
                    current_forecasts
                    .drop_duplicates(
                        subset=[
                            "id",
                            "forecast_origin",
                            "horizon"
                        ],
                        keep="last"
                    )
                    .sort_values(
                        ["id", "horizon"]
                    )
                )

                current_diagnostics = pd.concat(
                    [
                        existing_diagnostics,
                        pd.DataFrame(
                            new_diagnostic_rows
                        )
                    ],
                    ignore_index=True
                )

                current_diagnostics = (
                    current_diagnostics
                    .drop_duplicates(
                        subset=[
                            "id",
                            "forecast_origin"
                        ],
                        keep="last"
                    )
                    .sort_values("id")
                )

                current_forecasts.to_csv(
                    forecast_file,
                    index=False
                )

                current_diagnostics.to_csv(
                    diagnostics_file,
                    index=False
                )

                print(
                    f"ETS, d_{origin}: "
                    f"{completed_count}/500 "
                    "series completed."
                )

        # Final save for the origin.
        origin_forecasts = pd.concat(
            [
                existing_forecasts,
                *new_forecast_parts
            ],
            ignore_index=True
        )

        origin_forecasts = (
            origin_forecasts
            .drop_duplicates(
                subset=[
                    "id",
                    "forecast_origin",
                    "horizon"
                ],
                keep="last"
            )
            .sort_values(
                ["id", "horizon"]
            )
            .reset_index(drop=True)
        )

        origin_diagnostics = pd.concat(
            [
                existing_diagnostics,
                pd.DataFrame(new_diagnostic_rows)
            ],
            ignore_index=True
        )

        origin_diagnostics = (
            origin_diagnostics
            .drop_duplicates(
                subset=[
                    "id",
                    "forecast_origin"
                ],
                keep="last"
            )
            .sort_values("id")
            .reset_index(drop=True)
        )

        expected_origin_rows = (
            len(selected_sales)
            * FORECAST_HORIZON
        )

        if len(origin_forecasts) != expected_origin_rows:
            raise AssertionError(
                f"ETS d_{origin} has "
                f"{len(origin_forecasts)} rows; "
                f"expected {expected_origin_rows}."
            )

        origin_forecasts.to_csv(
            forecast_file,
            index=False
        )

        origin_diagnostics.to_csv(
            diagnostics_file,
            index=False
        )

        all_origin_forecasts.append(
            origin_forecasts
        )

        all_origin_diagnostics.append(
            origin_diagnostics
        )

        print(
            f"ETS: completed origin d_{origin}."
        )

    weekly_forecasts = pd.concat(
        all_origin_forecasts,
        ignore_index=True
    )

    weekly_diagnostics = pd.concat(
        all_origin_diagnostics,
        ignore_index=True
    )

    expected_rows = (
        len(selected_sales)
        * len(ADDITIONAL_WEEKLY_ORIGINS)
        * FORECAST_HORIZON
    )

    group_sizes = weekly_forecasts.groupby(
        ["id", "forecast_origin"]
    ).size()

    checks = pd.DataFrame({
        "check": [
            "Expected number of forecast rows",
            "Exactly 500 series",
            "Exactly twelve additional origins",
            "No missing forecasts",
            "Finite forecasts",
            "Non-negative forecasts",
            "28 rows per series and origin",
            "Missing actuals only after d_1941"
        ],
        "passed": [
            len(weekly_forecasts)
            == expected_rows,
            weekly_forecasts["id"].nunique()
            == len(selected_sales),
            weekly_forecasts[
                "forecast_origin"
            ].nunique()
            == len(ADDITIONAL_WEEKLY_ORIGINS),
            weekly_forecasts[
                "forecast"
            ].notna().all(),
            np.isfinite(
                weekly_forecasts["forecast"]
            ).all(),
            weekly_forecasts[
                "forecast"
            ].ge(0).all(),
            group_sizes.eq(
                FORECAST_HORIZON
            ).all(),
            (
                weekly_forecasts["actual"].isna()
                == weekly_forecasts[
                    "target_day"
                ].gt(MAX_OBSERVED_DAY)
            ).all()
        ]
    })

    if not checks["passed"].all():
        failed_checks = checks.loc[
            ~checks["passed"],
            "check"
        ].tolist()

        raise AssertionError(
            f"Additional ETS checks failed: "
            f"{failed_checks}"
        )

    forecast_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_ets_additional_forecasts.csv"
    )

    diagnostics_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_ets_additional_diagnostics.csv"
    )

    checks_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_ets_checks.csv"
    )

    weekly_forecasts.to_csv(
        forecast_output_file,
        index=False
    )

    weekly_diagnostics.to_csv(
        diagnostics_output_file,
        index=False
    )

    checks.to_csv(
        checks_output_file,
        index=False
    )

    print("\nETS weekly forecasts completed successfully.")
    print("Forecast rows:", len(weekly_forecasts))
    print(
        "Fallback fits:",
        int(
            weekly_diagnostics[
                "fallback_used"
            ].sum()
        )
    )
    print(
        "Elapsed minutes:",
        round(
            (
                time.perf_counter()
                - total_start_time
            ) / 60,
            2
        )
    )
    print("\nChecks:")
    print(checks)
    print("\nSaved file:")
    print(forecast_output_file)

    return (
        weekly_forecasts,
        weekly_diagnostics,
        checks
    )


# ------------------------------------------------------------
# 4. Start the ETS calculation
# ------------------------------------------------------------

ets_weekly_forecasts, ets_weekly_diagnostics, ets_weekly_checks = (
    run_additional_weekly_ets(
        checkpoint_interval=25
    )
)


ETS, d_1836: 25/500 series completed.
ETS, d_1836: 50/500 series completed.
ETS, d_1836: 75/500 series completed.
ETS, d_1836: 100/500 series completed.
ETS, d_1836: 125/500 series completed.
ETS, d_1836: 150/500 series completed.
ETS, d_1836: 175/500 series completed.
ETS, d_1836: 200/500 series completed.
ETS, d_1836: 225/500 series completed.
ETS, d_1836: 250/500 series completed.
ETS, d_1836: 275/500 series completed.
ETS, d_1836: 300/500 series completed.
ETS, d_1836: 325/500 series completed.
ETS, d_1836: 350/500 series completed.
ETS, d_1836: 375/500 series completed.
ETS, d_1836: 400/500 series completed.
ETS, d_1836: 425/500 series completed.
ETS, d_1836: 450/500 series completed.
ETS, d_1836: 475/500 series completed.
ETS, d_1836: 500/500 series completed.
ETS: completed origin d_1836.
ETS, d_1843: 25/500 series completed.
ETS, d_1843: 50/500 series completed.
ETS, d_1843: 75/500 series completed.
ETS, d_1843: 100/500 series completed.
ETS, d_1843: 125/500 series completed.
E

ARIMA/SARIMA

In [135]:
import json
import time
import warnings

import numpy as np
import pandas as pd

from pmdarima.arima import auto_arima


# ------------------------------------------------------------
# 1. Automatic ARIMA/SARIMA fitting
# ------------------------------------------------------------

def fit_weekly_automatic_arima(
    training_values,
    horizon=28
):

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        model = auto_arima(
            training_values,

            # Non-seasonal orders
            start_p=1,
            start_q=1,
            max_p=3,
            max_q=3,
            d=None,
            max_d=2,

            # Weekly seasonal orders
            start_P=0,
            start_Q=0,
            max_P=1,
            max_Q=1,
            D=None,
            max_D=1,
            m=7,
            seasonal=True,

            # Model selection
            information_criterion="aicc",
            test="kpss",
            seasonal_test="ocsb",
            stepwise=True,
            max_order=None,

            # Exact likelihood estimation
            method="lbfgs",
            maxiter=200,
            suppress_warnings=True,
            error_action="raise",
            trace=False,
            random=False,
            with_intercept="auto",
            n_jobs=1,

            sarimax_kwargs={
                "simple_differencing": False,
                "enforce_stationarity": False,
                "enforce_invertibility": False
            }
        )

    predictions = np.asarray(
        model.predict(n_periods=horizon),
        dtype=float
    )

    predictions = np.maximum(
        predictions,
        0.0
    )

    arima_result = model.arima_res_

    mle_information = (
        getattr(
            arima_result,
            "mle_retvals",
            {}
        )
        or {}
    )

    diagnostics = {
        "order": json.dumps(
            list(model.order)
        ),
        "seasonal_order": json.dumps(
            list(model.seasonal_order)
        ),
        "aicc": float(model.aicc()),
        "converged": bool(
            mle_information.get(
                "converged",
                False
            )
        ),
        "stepwise_search": True,
        "likelihood_approximation": False
    }

    return predictions, diagnostics


# ------------------------------------------------------------
# 2. ARIMA/SARIMA checkpoint runner
# ------------------------------------------------------------

def run_additional_weekly_arima(
    checkpoint_interval=25
):

    all_origin_forecasts = []
    all_origin_diagnostics = []

    calculation_start = time.perf_counter()

    for origin in ADDITIONAL_WEEKLY_ORIGINS:

        forecast_file = (
            OUTPUT_DIR
            / f"inventory_weekly_arima_sarima_origin_{origin}.csv"
        )

        diagnostics_file = (
            OUTPUT_DIR
            / f"inventory_weekly_arima_sarima_diagnostics_origin_{origin}.csv"
        )

        # ----------------------------------------------------
        # Load an existing forecast checkpoint
        # ----------------------------------------------------

        if forecast_file.exists():
            existing_forecasts = pd.read_csv(
                forecast_file
            )

            forecast_groups = (
                existing_forecasts
                .groupby("id")
                .agg(
                    rows=("horizon", "size"),
                    complete=(
                        "forecast",
                        lambda values:
                            values.notna().all()
                            and np.isfinite(values).all()
                    )
                )
            )

            valid_forecast_ids = set(
                forecast_groups.loc[
                    forecast_groups["rows"].eq(
                        FORECAST_HORIZON
                    )
                    & forecast_groups["complete"],
                ].index
            )

        else:
            existing_forecasts = pd.DataFrame()
            valid_forecast_ids = set()

        # ----------------------------------------------------
        # Load an existing diagnostics checkpoint
        # ----------------------------------------------------

        if diagnostics_file.exists():
            existing_diagnostics = pd.read_csv(
                diagnostics_file
            )

            valid_diagnostic_ids = set(
                existing_diagnostics["id"]
            )

        else:
            existing_diagnostics = pd.DataFrame()
            valid_diagnostic_ids = set()

        # A series counts as complete only if both files exist.
        completed_ids = (
            valid_forecast_ids
            & valid_diagnostic_ids
        )

        if not existing_forecasts.empty:
            existing_forecasts = (
                existing_forecasts.loc[
                    existing_forecasts["id"].isin(
                        completed_ids
                    )
                ]
                .drop_duplicates(
                    subset=[
                        "id",
                        "forecast_origin",
                        "horizon"
                    ],
                    keep="last"
                )
            )

        if not existing_diagnostics.empty:
            existing_diagnostics = (
                existing_diagnostics.loc[
                    existing_diagnostics["id"].isin(
                        completed_ids
                    )
                ]
                .drop_duplicates(
                    subset=[
                        "id",
                        "forecast_origin"
                    ],
                    keep="last"
                )
            )

        # Skip an already completed origin.
        if len(completed_ids) == len(selected_sales):

            print(
                f"ARIMA/SARIMA, d_{origin}: "
                "complete checkpoint loaded."
            )

            all_origin_forecasts.append(
                existing_forecasts
            )

            all_origin_diagnostics.append(
                existing_diagnostics
            )

            continue

        new_forecast_parts = []
        new_diagnostic_rows = []

        # ----------------------------------------------------
        # Fit the 500 individual series
        # ----------------------------------------------------

        for series_index, row in (
            selected_sales.iterrows()
        ):

            series_id = row["id"]

            if series_id in completed_ids:
                continue

            fitting_start = time.perf_counter()

            active_start = int(
                row["first_active_day"]
            )

            training_values = demand_matrix[
                series_index,
                active_start - 1:origin
            ]

            fallback_used = False
            error_message = ""

            try:
                predictions, diagnostics = (
                    fit_weekly_automatic_arima(
                        training_values,
                        horizon=FORECAST_HORIZON
                    )
                )

            except Exception as error:

                # Transparent seasonal-naïve fallback
                last_week = training_values[-7:]

                predictions = np.tile(
                    last_week,
                    int(
                        np.ceil(
                            FORECAST_HORIZON / 7
                        )
                    )
                )[:FORECAST_HORIZON]

                predictions = np.maximum(
                    predictions,
                    0.0
                )

                diagnostics = {}
                fallback_used = True

                error_message = (
                    f"{type(error).__name__}: {error}"
                )

            forecast_part = (
                build_single_series_weekly_frame(
                    series_index=series_index,
                    origin=origin,
                    predictions=predictions,
                    model_name="arima_sarima"
                )
            )

            new_forecast_parts.append(
                forecast_part
            )

            new_diagnostic_rows.append({
                "id": series_id,
                "demand_class":
                    row["demand_class"],
                "model": "arima_sarima",
                "forecast_origin": origin,
                "active_training_start":
                    active_start,
                "training_observations":
                    len(training_values),
                "fit_seconds":
                    time.perf_counter()
                    - fitting_start,
                "fallback_used":
                    fallback_used,
                "error_message":
                    error_message,
                **diagnostics
            })

            completed_count = (
                len(completed_ids)
                + len(new_diagnostic_rows)
            )

            # ------------------------------------------------
            # Save every 25 series
            # ------------------------------------------------

            if (
                completed_count
                % checkpoint_interval == 0
            ):

                current_forecasts = pd.concat(
                    [
                        existing_forecasts,
                        *new_forecast_parts
                    ],
                    ignore_index=True
                )

                current_forecasts = (
                    current_forecasts
                    .drop_duplicates(
                        subset=[
                            "id",
                            "forecast_origin",
                            "horizon"
                        ],
                        keep="last"
                    )
                    .sort_values(
                        ["id", "horizon"]
                    )
                )

                current_diagnostics = pd.concat(
                    [
                        existing_diagnostics,
                        pd.DataFrame(
                            new_diagnostic_rows
                        )
                    ],
                    ignore_index=True
                )

                current_diagnostics = (
                    current_diagnostics
                    .drop_duplicates(
                        subset=[
                            "id",
                            "forecast_origin"
                        ],
                        keep="last"
                    )
                    .sort_values("id")
                )

                current_forecasts.to_csv(
                    forecast_file,
                    index=False
                )

                current_diagnostics.to_csv(
                    diagnostics_file,
                    index=False
                )

                print(
                    f"ARIMA/SARIMA, d_{origin}: "
                    f"{completed_count}/500 "
                    "series completed."
                )

        # ----------------------------------------------------
        # Final save for the origin
        # ----------------------------------------------------

        origin_forecasts = pd.concat(
            [
                existing_forecasts,
                *new_forecast_parts
            ],
            ignore_index=True
        )

        origin_forecasts = (
            origin_forecasts
            .drop_duplicates(
                subset=[
                    "id",
                    "forecast_origin",
                    "horizon"
                ],
                keep="last"
            )
            .sort_values(
                ["id", "horizon"]
            )
            .reset_index(drop=True)
        )

        origin_diagnostics = pd.concat(
            [
                existing_diagnostics,
                pd.DataFrame(
                    new_diagnostic_rows
                )
            ],
            ignore_index=True
        )

        origin_diagnostics = (
            origin_diagnostics
            .drop_duplicates(
                subset=[
                    "id",
                    "forecast_origin"
                ],
                keep="last"
            )
            .sort_values("id")
            .reset_index(drop=True)
        )

        expected_origin_rows = (
            len(selected_sales)
            * FORECAST_HORIZON
        )

        if len(origin_forecasts) != expected_origin_rows:
            raise AssertionError(
                f"ARIMA/SARIMA d_{origin} contains "
                f"{len(origin_forecasts)} rows; "
                f"expected {expected_origin_rows}."
            )

        origin_forecasts.to_csv(
            forecast_file,
            index=False
        )

        origin_diagnostics.to_csv(
            diagnostics_file,
            index=False
        )

        all_origin_forecasts.append(
            origin_forecasts
        )

        all_origin_diagnostics.append(
            origin_diagnostics
        )

        print(
            f"ARIMA/SARIMA: completed origin d_{origin}."
        )

    # --------------------------------------------------------
    # 3. Combine all twelve additional origins
    # --------------------------------------------------------

    weekly_forecasts = pd.concat(
        all_origin_forecasts,
        ignore_index=True
    )

    weekly_diagnostics = pd.concat(
        all_origin_diagnostics,
        ignore_index=True
    )

    expected_rows = (
        len(selected_sales)
        * len(ADDITIONAL_WEEKLY_ORIGINS)
        * FORECAST_HORIZON
    )

    group_sizes = weekly_forecasts.groupby(
        ["id", "forecast_origin"]
    ).size()

    checks = pd.DataFrame({
        "check": [
            "Expected number of forecast rows",
            "Exactly 500 series",
            "Exactly twelve additional origins",
            "No missing forecasts",
            "Finite forecasts",
            "Non-negative forecasts",
            "28 rows per series and origin",
            "Missing actuals only after d_1941"
        ],
        "passed": [
            len(weekly_forecasts)
            == expected_rows,
            weekly_forecasts["id"].nunique()
            == len(selected_sales),
            weekly_forecasts[
                "forecast_origin"
            ].nunique()
            == len(ADDITIONAL_WEEKLY_ORIGINS),
            weekly_forecasts[
                "forecast"
            ].notna().all(),
            np.isfinite(
                weekly_forecasts["forecast"]
            ).all(),
            weekly_forecasts[
                "forecast"
            ].ge(0).all(),
            group_sizes.eq(
                FORECAST_HORIZON
            ).all(),
            (
                weekly_forecasts["actual"].isna()
                == weekly_forecasts[
                    "target_day"
                ].gt(MAX_OBSERVED_DAY)
            ).all()
        ]
    })

    if not checks["passed"].all():
        failed_checks = checks.loc[
            ~checks["passed"],
            "check"
        ].tolist()

        raise AssertionError(
            f"Additional ARIMA/SARIMA checks failed: "
            f"{failed_checks}"
        )

    # --------------------------------------------------------
    # 4. Save combined outputs
    # --------------------------------------------------------

    forecast_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_arima_sarima_additional_forecasts.csv"
    )

    diagnostics_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_arima_sarima_additional_diagnostics.csv"
    )

    checks_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_arima_sarima_checks.csv"
    )

    weekly_forecasts.to_csv(
        forecast_output_file,
        index=False
    )

    weekly_diagnostics.to_csv(
        diagnostics_output_file,
        index=False
    )

    checks.to_csv(
        checks_output_file,
        index=False
    )

    print(
        "\nARIMA/SARIMA weekly forecasts "
        "completed successfully."
    )

    print(
        "Forecast rows:",
        len(weekly_forecasts)
    )

    print(
        "Fallback fits:",
        int(
            weekly_diagnostics[
                "fallback_used"
            ].sum()
        )
    )

    print("\nConvergence results:")
    print(
        weekly_diagnostics[
            "converged"
        ].value_counts(
            dropna=False
        )
    )

    print(
        "\nElapsed hours:",
        round(
            (
                time.perf_counter()
                - calculation_start
            ) / 3600,
            2
        )
    )

    print("\nChecks:")
    print(checks)

    print("\nSaved file:")
    print(forecast_output_file)

    return (
        weekly_forecasts,
        weekly_diagnostics,
        checks
    )


# ------------------------------------------------------------
# 5. Start ARIMA/SARIMA
# ------------------------------------------------------------

arima_weekly_forecasts, arima_weekly_diagnostics, arima_weekly_checks = (
    run_additional_weekly_arima(
        checkpoint_interval=25
    )
)


ARIMA/SARIMA, d_1836: 25/500 series completed.
ARIMA/SARIMA, d_1836: 50/500 series completed.
ARIMA/SARIMA, d_1836: 75/500 series completed.
ARIMA/SARIMA, d_1836: 100/500 series completed.
ARIMA/SARIMA, d_1836: 125/500 series completed.
ARIMA/SARIMA, d_1836: 150/500 series completed.
ARIMA/SARIMA, d_1836: 175/500 series completed.
ARIMA/SARIMA, d_1836: 200/500 series completed.
ARIMA/SARIMA, d_1836: 225/500 series completed.
ARIMA/SARIMA, d_1836: 250/500 series completed.
ARIMA/SARIMA, d_1836: 275/500 series completed.
ARIMA/SARIMA, d_1836: 300/500 series completed.
ARIMA/SARIMA, d_1836: 325/500 series completed.
ARIMA/SARIMA, d_1836: 350/500 series completed.
ARIMA/SARIMA, d_1836: 375/500 series completed.
ARIMA/SARIMA, d_1836: 400/500 series completed.
ARIMA/SARIMA, d_1836: 425/500 series completed.
ARIMA/SARIMA, d_1836: 450/500 series completed.
ARIMA/SARIMA, d_1836: 475/500 series completed.
ARIMA/SARIMA, d_1836: 500/500 series completed.
ARIMA/SARIMA: completed origin d_1836.
ARIM

Run XGBoost

In [138]:
import gc
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor


# ------------------------------------------------------------
# 1. Selected XGBoost configuration
# ------------------------------------------------------------

XGBOOST_CONFIGURATION = {
    "configuration_id": "n300_lr0p03_d10",
    "n_estimators": 300,
    "learning_rate": 0.03,
    "max_depth": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8
}

RANDOM_SEED = 2026

XGB_LAGS = [1, 7, 14, 28]
XGB_ROLLING_WINDOWS = [7, 28]

XGB_CATEGORICAL_FEATURES = [
    "item_id",
    "cat_id",
    "dept_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

XGB_NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_sd_7",
    "zero_proportion_7",
    "rolling_mean_28",
    "rolling_sd_28",
    "zero_proportion_28",
    "weekday",
    "month",
    "snap",
    "price_level",
    "price_change_7",
    "price_pct_change_7"
]

print("Selected configuration:")
print(XGBOOST_CONFIGURATION)


# ------------------------------------------------------------
# 2. Prepare calendar information
# ------------------------------------------------------------

calendar_xgb = pd.read_csv(CALENDAR_FILE)

calendar_xgb["day_number"] = (
    calendar_xgb["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

calendar_xgb["date"] = pd.to_datetime(
    calendar_xgb["date"]
)

calendar_xgb["weekday"] = (
    calendar_xgb["date"].dt.dayofweek
)

calendar_xgb["month"] = (
    calendar_xgb["date"].dt.month
)

event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for column in event_columns:
    calendar_xgb[column] = (
        calendar_xgb[column]
        .fillna("none")
        .astype(str)
    )

calendar_xgb = (
    calendar_xgb
    .sort_values("day_number")
    .reset_index(drop=True)
)

required_last_calendar_day = (
    max(ADDITIONAL_WEEKLY_ORIGINS)
    + FORECAST_HORIZON
)

if calendar_xgb["day_number"].max() < required_last_calendar_day:
    raise ValueError(
        "The calendar does not cover all required "
        "future forecast days."
    )

print(
    "Calendar covers forecasts through:",
    f"d_{required_last_calendar_day}"
)


# ------------------------------------------------------------
# 3. Reconstruct the daily price matrix
# ------------------------------------------------------------

prices_xgb = pd.read_csv(PRICES_FILE)

observed_calendar = calendar_xgb.loc[
    calendar_xgb["day_number"].between(
        1,
        MAX_OBSERVED_DAY
    )
].copy()

week_by_day = (
    observed_calendar["wm_yr_wk"]
    .to_numpy()
)

price_groups = {
    key: (
        group
        .drop_duplicates("wm_yr_wk")
        .set_index("wm_yr_wk")["sell_price"]
    )
    for key, group in prices_xgb.groupby(
        ["store_id", "item_id"]
    )
}

price_matrix_xgb = np.full(
    (
        len(selected_sales),
        MAX_OBSERVED_DAY
    ),
    np.nan,
    dtype=float
)

for series_index, row in selected_sales.iterrows():

    series_key = (
        row["store_id"],
        row["item_id"]
    )

    if series_key not in price_groups:
        continue

    daily_prices = (
        pd.Series(week_by_day)
        .map(price_groups[series_key])
        .astype(float)
        .ffill()
    )

    price_matrix_xgb[series_index] = (
        daily_prices.to_numpy()
    )

origin_indices = (
    np.array(ADDITIONAL_WEEKLY_ORIGINS) - 1
)

if not np.isfinite(
    price_matrix_xgb[:, origin_indices]
).all():
    raise ValueError(
        "At least one series has no valid price "
        "at a weekly forecast origin."
    )

print(
    "Price matrix shape:",
    price_matrix_xgb.shape
)


# ------------------------------------------------------------
# 4. Price-feature helper
# ------------------------------------------------------------

def calculate_safe_price_features(
    current_price,
    earlier_price
):

    price_change = (
        current_price - earlier_price
    )

    percentage_change = np.divide(
        price_change,
        earlier_price,
        out=np.zeros_like(
            price_change,
            dtype=float
        ),
        where=(
            np.isfinite(earlier_price)
            & (earlier_price != 0)
        )
    )

    price_change = np.nan_to_num(
        price_change,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    percentage_change = np.nan_to_num(
        percentage_change,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return price_change, percentage_change


# ------------------------------------------------------------
# 5. Build the extended historical training panel
# ------------------------------------------------------------

def build_extended_xgboost_panel():

    calendar_indexed = (
        calendar_xgb.set_index("day_number")
    )

    panel_parts = []

    last_training_day = max(
        ADDITIONAL_WEEKLY_ORIGINS
    )

    for series_index, row in (
        selected_sales.iterrows()
    ):

        first_target_day = max(
            int(row["first_active_day"]) + 28,
            29
        )

        target_days = np.arange(
            first_target_day,
            last_training_day + 1
        )

        target_indices = target_days - 1

        series_values = demand_matrix[
            series_index
        ]

        part = pd.DataFrame({
            "id": row["id"],
            "target_day": target_days,
            "target": series_values[
                target_indices
            ],
            "item_id": row["item_id"],
            "cat_id": row["cat_id"],
            "dept_id": row["dept_id"],
            "store_id": row["store_id"],
            "state_id": row["state_id"]
        })

        for lag in XGB_LAGS:
            part[f"lag_{lag}"] = (
                series_values[
                    target_indices - lag
                ]
            )

        for window in XGB_ROLLING_WINDOWS:

            rolling_values = np.vstack([
                series_values[
                    index - window:index
                ]
                for index in target_indices
            ])

            part[f"rolling_mean_{window}"] = (
                rolling_values.mean(axis=1)
            )

            part[f"rolling_sd_{window}"] = (
                rolling_values.std(
                    axis=1,
                    ddof=0
                )
            )

            part[
                f"zero_proportion_{window}"
            ] = (
                rolling_values == 0
            ).mean(axis=1)

        calendar_rows = calendar_indexed.loc[
            target_days
        ]

        part["weekday"] = (
            calendar_rows["weekday"].to_numpy()
        )

        part["month"] = (
            calendar_rows["month"].to_numpy()
        )

        for column in event_columns:
            part[column] = (
                calendar_rows[column].to_numpy()
            )

        snap_column = (
            f"snap_{row['state_id']}"
        )

        part["snap"] = (
            calendar_rows[
                snap_column
            ].to_numpy(dtype=float)
        )

        current_price = price_matrix_xgb[
            series_index,
            target_indices - 1
        ]

        earlier_price = price_matrix_xgb[
            series_index,
            target_indices - 8
        ]

        (
            price_change,
            price_percentage_change
        ) = calculate_safe_price_features(
            current_price,
            earlier_price
        )

        part["price_level"] = np.nan_to_num(
            current_price,
            nan=0.0
        )

        part["price_change_7"] = (
            price_change
        )

        part["price_pct_change_7"] = (
            price_percentage_change
        )

        panel_parts.append(part)

    panel = pd.concat(
        panel_parts,
        ignore_index=True
    )

    return panel


print("Building extended XGBoost training panel...")

xgboost_weekly_panel = (
    build_extended_xgboost_panel()
)

print(
    "XGBoost panel rows:",
    len(xgboost_weekly_panel)
)

print(
    "Last historical target day:",
    xgboost_weekly_panel[
        "target_day"
    ].max()
)


# ------------------------------------------------------------
# 6. Transform XGBoost features
# ------------------------------------------------------------

def transform_weekly_xgboost_features(
    frame,
    encoder
):

    numeric_features = frame[
        XGB_NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    categorical_features = encoder.transform(
        frame[
            XGB_CATEGORICAL_FEATURES
        ].astype(str)
    ).astype(np.float32)

    feature_matrix = np.column_stack([
        numeric_features,
        categorical_features
    ])

    if not np.isfinite(
        feature_matrix
    ).all():
        raise ValueError(
            "XGBoost feature matrix contains "
            "non-finite values."
        )

    return feature_matrix


# ------------------------------------------------------------
# 7. Fit XGBoost at one weekly origin
# ------------------------------------------------------------

def fit_weekly_xgboost(origin):

    training_frame = (
        xgboost_weekly_panel.loc[
            xgboost_weekly_panel[
                "target_day"
            ].le(origin)
        ]
    )

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32
    )

    encoder.fit(
        training_frame[
            XGB_CATEGORICAL_FEATURES
        ].astype(str)
    )

    feature_matrix = (
        transform_weekly_xgboost_features(
            training_frame,
            encoder
        )
    )

    target = training_frame[
        "target"
    ].to_numpy(dtype=np.float32)

    model = XGBRegressor(
        n_estimators=(
            XGBOOST_CONFIGURATION[
                "n_estimators"
            ]
        ),
        learning_rate=(
            XGBOOST_CONFIGURATION[
                "learning_rate"
            ]
        ),
        max_depth=(
            XGBOOST_CONFIGURATION[
                "max_depth"
            ]
        ),
        subsample=(
            XGBOOST_CONFIGURATION[
                "subsample"
            ]
        ),
        colsample_bytree=(
            XGBOOST_CONFIGURATION[
                "colsample_bytree"
            ]
        ),
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        feature_matrix,
        target,
        verbose=False
    )

    return model, encoder, len(training_frame)


# ------------------------------------------------------------
# 8. Recursive 28-day XGBoost forecast
# ------------------------------------------------------------

def recursive_weekly_xgboost_forecast(
    model,
    encoder,
    origin
):

    number_of_series = len(
        selected_sales
    )

    history = np.full(
        (
            number_of_series,
            origin + FORECAST_HORIZON
        ),
        np.nan,
        dtype=float
    )

    # Demand through the origin is known.
    history[:, :origin] = (
        demand_matrix[:, :origin]
    )

    metadata = selected_sales.reset_index(
        drop=True
    )

    calendar_indexed = (
        calendar_xgb.set_index("day_number")
    )

    origin_index = origin - 1

    known_price = price_matrix_xgb[
        :,
        origin_index
    ]

    earlier_price = price_matrix_xgb[
        :,
        origin_index - 7
    ]

    (
        fixed_price_change,
        fixed_price_percentage_change
    ) = calculate_safe_price_features(
        known_price,
        earlier_price
    )

    for horizon in range(
        1,
        FORECAST_HORIZON + 1
    ):

        target_day = origin + horizon
        target_index = target_day - 1

        calendar_row = (
            calendar_indexed.loc[target_day]
        )

        feature_frame = metadata[
            [
                "item_id",
                "cat_id",
                "dept_id",
                "store_id",
                "state_id"
            ]
        ].copy()

        for lag in XGB_LAGS:
            feature_frame[
                f"lag_{lag}"
            ] = history[
                :,
                target_index - lag
            ]

        for window in XGB_ROLLING_WINDOWS:

            rolling_values = history[
                :,
                target_index - window:
                target_index
            ]

            feature_frame[
                f"rolling_mean_{window}"
            ] = rolling_values.mean(axis=1)

            feature_frame[
                f"rolling_sd_{window}"
            ] = rolling_values.std(
                axis=1,
                ddof=0
            )

            feature_frame[
                f"zero_proportion_{window}"
            ] = (
                rolling_values == 0
            ).mean(axis=1)

        feature_frame["weekday"] = float(
            calendar_row["weekday"]
        )

        feature_frame["month"] = float(
            calendar_row["month"]
        )

        for column in event_columns:
            feature_frame[column] = str(
                calendar_row[column]
            )

        feature_frame["snap"] = [
            float(
                calendar_row[
                    f"snap_{state}"
                ]
            )
            for state in metadata["state_id"]
        ]

        # Future selling prices are not used.
        feature_frame["price_level"] = (
            np.nan_to_num(
                known_price,
                nan=0.0
            )
        )

        feature_frame["price_change_7"] = (
            fixed_price_change
        )

        feature_frame[
            "price_pct_change_7"
        ] = fixed_price_percentage_change

        features = (
            transform_weekly_xgboost_features(
                feature_frame,
                encoder
            )
        )

        predictions = np.asarray(
            model.predict(features),
            dtype=float
        )

        predictions = np.maximum(
            predictions,
            0.0
        )

        if not np.isfinite(
            predictions
        ).all():
            raise ValueError(
                "XGBoost produced a non-finite "
                f"forecast at horizon {horizon}."
            )

        history[:, target_index] = predictions

    return history[
        :,
        origin:
        origin + FORECAST_HORIZON
    ]


# ------------------------------------------------------------
# 9. Run the twelve additional origins
# ------------------------------------------------------------

def run_additional_weekly_xgboost():

    forecast_parts = []
    diagnostic_rows = []

    total_start = time.perf_counter()

    for origin in ADDITIONAL_WEEKLY_ORIGINS:

        forecast_file = (
            OUTPUT_DIR
            / f"inventory_weekly_xgboost_origin_{origin}.csv"
        )

        diagnostics_file = (
            OUTPUT_DIR
            / f"inventory_weekly_xgboost_diagnostics_origin_{origin}.csv"
        )

        if (
            forecast_file.exists()
            and diagnostics_file.exists()
        ):
            saved_forecasts = pd.read_csv(
                forecast_file
            )

            saved_diagnostics = pd.read_csv(
                diagnostics_file
            )

            valid_checkpoint = (
                len(saved_forecasts)
                == len(selected_sales)
                * FORECAST_HORIZON
                and saved_forecasts[
                    "id"
                ].nunique()
                == len(selected_sales)
                and saved_forecasts[
                    "forecast_origin"
                ].eq(origin).all()
                and saved_forecasts[
                    "forecast"
                ].notna().all()
                and np.isfinite(
                    saved_forecasts[
                        "forecast"
                    ]
                ).all()
            )

            if valid_checkpoint:
                forecast_parts.append(
                    saved_forecasts
                )

                diagnostic_rows.append(
                    saved_diagnostics
                )

                print(
                    f"XGBoost, d_{origin}: "
                    "complete checkpoint loaded."
                )

                continue

        fit_start = time.perf_counter()

        model, encoder, training_rows = (
            fit_weekly_xgboost(origin)
        )

        fit_seconds = (
            time.perf_counter() - fit_start
        )

        forecast_start = time.perf_counter()

        predictions = (
            recursive_weekly_xgboost_forecast(
                model=model,
                encoder=encoder,
                origin=origin
            )
        )

        forecast_seconds = (
            time.perf_counter()
            - forecast_start
        )

        origin_forecasts = (
            build_weekly_forecast_frame(
                predictions=predictions,
                origin=origin,
                model_name="xgboost"
            )
        )

        origin_diagnostics = pd.DataFrame([{
            "model": "xgboost",
            "forecast_origin": origin,
            "configuration_id":
                XGBOOST_CONFIGURATION[
                    "configuration_id"
                ],
            "n_estimators":
                XGBOOST_CONFIGURATION[
                    "n_estimators"
                ],
            "learning_rate":
                XGBOOST_CONFIGURATION[
                    "learning_rate"
                ],
            "max_depth":
                XGBOOST_CONFIGURATION[
                    "max_depth"
                ],
            "subsample":
                XGBOOST_CONFIGURATION[
                    "subsample"
                ],
            "colsample_bytree":
                XGBOOST_CONFIGURATION[
                    "colsample_bytree"
                ],
            "training_rows": training_rows,
            "fit_seconds": fit_seconds,
            "forecast_seconds":
                forecast_seconds,
            "future_prices_used": False
        }])

        origin_forecasts.to_csv(
            forecast_file,
            index=False
        )

        origin_diagnostics.to_csv(
            diagnostics_file,
            index=False
        )

        forecast_parts.append(
            origin_forecasts
        )

        diagnostic_rows.append(
            origin_diagnostics
        )

        del model
        del encoder
        gc.collect()

        print(
            f"XGBoost: completed origin d_{origin}."
        )

    weekly_forecasts = pd.concat(
        forecast_parts,
        ignore_index=True
    )

    weekly_diagnostics = pd.concat(
        diagnostic_rows,
        ignore_index=True
    )

    expected_rows = (
        len(selected_sales)
        * len(ADDITIONAL_WEEKLY_ORIGINS)
        * FORECAST_HORIZON
    )

    group_sizes = weekly_forecasts.groupby(
        ["id", "forecast_origin"]
    ).size()

    checks = pd.DataFrame({
        "check": [
            "Expected number of forecast rows",
            "Exactly 500 series",
            "Exactly twelve additional origins",
            "No missing forecasts",
            "Finite forecasts",
            "Non-negative forecasts",
            "28 rows per series and origin",
            "Selected configuration retained",
            "Future prices excluded",
            "Missing actuals only after d_1941"
        ],
        "passed": [
            len(weekly_forecasts)
            == expected_rows,
            weekly_forecasts["id"].nunique()
            == len(selected_sales),
            weekly_forecasts[
                "forecast_origin"
            ].nunique()
            == len(ADDITIONAL_WEEKLY_ORIGINS),
            weekly_forecasts[
                "forecast"
            ].notna().all(),
            np.isfinite(
                weekly_forecasts["forecast"]
            ).all(),
            weekly_forecasts[
                "forecast"
            ].ge(0).all(),
            group_sizes.eq(
                FORECAST_HORIZON
            ).all(),
            weekly_diagnostics[
                "configuration_id"
            ].eq(
                "n300_lr0p03_d10"
            ).all(),
            weekly_diagnostics[
                "future_prices_used"
            ].eq(False).all(),
            (
                weekly_forecasts["actual"].isna()
                == weekly_forecasts[
                    "target_day"
                ].gt(MAX_OBSERVED_DAY)
            ).all()
        ]
    })

    if not checks["passed"].all():
        failed_checks = checks.loc[
            ~checks["passed"],
            "check"
        ].tolist()

        raise AssertionError(
            f"Additional XGBoost checks failed: "
            f"{failed_checks}"
        )

    forecast_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_xgboost_additional_forecasts.csv"
    )

    diagnostics_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_xgboost_additional_diagnostics.csv"
    )

    checks_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_xgboost_checks.csv"
    )

    weekly_forecasts.to_csv(
        forecast_output_file,
        index=False
    )

    weekly_diagnostics.to_csv(
        diagnostics_output_file,
        index=False
    )

    checks.to_csv(
        checks_output_file,
        index=False
    )

    print(
        "\nXGBoost weekly forecasts "
        "completed successfully."
    )

    print(
        "Forecast rows:",
        len(weekly_forecasts)
    )

    print(
        "Elapsed minutes:",
        round(
            (
                time.perf_counter()
                - total_start
            ) / 60,
            2
        )
    )

    print("\nChecks:")
    print(checks)

    print("\nSaved file:")
    print(forecast_output_file)

    return (
        weekly_forecasts,
        weekly_diagnostics,
        checks
    )


# ------------------------------------------------------------
# 10. Start XGBoost
# ------------------------------------------------------------

xgboost_weekly_forecasts, xgboost_weekly_diagnostics, xgboost_weekly_checks = (
    run_additional_weekly_xgboost()
)


Selected configuration:
{'configuration_id': 'n300_lr0p03_d10', 'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 10, 'subsample': 0.8, 'colsample_bytree': 0.8}
Calendar covers forecasts through: d_1962
Price matrix shape: (500, 1941)
Building extended XGBoost training panel...
XGBoost panel rows: 835729
Last historical target day: 1934
XGBoost: completed origin d_1836.
XGBoost: completed origin d_1843.
XGBoost: completed origin d_1850.
XGBoost: completed origin d_1864.
XGBoost: completed origin d_1871.
XGBoost: completed origin d_1878.
XGBoost: completed origin d_1892.
XGBoost: completed origin d_1899.
XGBoost: completed origin d_1906.
XGBoost: completed origin d_1920.
XGBoost: completed origin d_1927.
XGBoost: completed origin d_1934.

XGBoost weekly forecasts completed successfully.
Forecast rows: 168000
Elapsed minutes: 1.11

Checks:
                               check  passed
0   Expected number of forecast rows    True
1                 Exactly 500 series    True
2  Exact

Run TiRex-2

In [141]:
import gc
import time

import numpy as np
import pandas as pd
import torch

from tirex2 import load_model, TimeseriesType


# ------------------------------------------------------------
# 1. TiRex-2 settings and device
# ------------------------------------------------------------

TIREX_CHECKPOINT = "NX-AI/TiRex-2"
TIREX_BATCH_SIZE = 16

if torch.cuda.is_available():
    TIREX_DEVICE = "cuda"
elif torch.backends.mps.is_available():
    TIREX_DEVICE = "mps"
else:
    TIREX_DEVICE = "cpu"

print("Selected device:", TIREX_DEVICE)


# ------------------------------------------------------------
# 2. Construct the known calendar covariates
# ------------------------------------------------------------

def build_tirex_calendar_covariates(
    calendar_slice,
    state_id
):

    weekday = calendar_slice[
        "weekday"
    ].to_numpy(dtype=float)

    weekday_angle = (
        2 * np.pi * weekday / 7
    )

    covariate_columns = [
        np.sin(weekday_angle),
        np.cos(weekday_angle)
    ]

    event_name_1 = calendar_slice[
        "event_name_1"
    ].astype(str)

    event_name_2 = calendar_slice[
        "event_name_2"
    ].astype(str)

    # Indicators for the presence of named events.
    covariate_columns.append(
        (event_name_1 != "none")
        .to_numpy(dtype=float)
    )

    covariate_columns.append(
        (event_name_2 != "none")
        .to_numpy(dtype=float)
    )

    # Binary indicators for each event type.
    event_types = sorted(
        (
            set(
                calendar_slice[
                    "event_type_1"
                ].astype(str)
            )
            |
            set(
                calendar_slice[
                    "event_type_2"
                ].astype(str)
            )
        )
        - {"none"}
    )

    for event_type in event_types:
        event_indicator = (
            (
                calendar_slice[
                    "event_type_1"
                ].astype(str)
                == event_type
            )
            |
            (
                calendar_slice[
                    "event_type_2"
                ].astype(str)
                == event_type
            )
        )

        covariate_columns.append(
            event_indicator.to_numpy(
                dtype=float
            )
        )

    snap_column = f"snap_{state_id}"

    if snap_column not in calendar_slice.columns:
        raise ValueError(
            f"Calendar is missing {snap_column}."
        )

    covariate_columns.append(
        calendar_slice[
            snap_column
        ].to_numpy(dtype=float)
    )

    return np.vstack(
        covariate_columns
    ).astype(np.float32)


# ------------------------------------------------------------
# 3. Build TiRex-2 inputs for one weekly origin
# ------------------------------------------------------------

def build_weekly_tirex_inputs(origin):

    timeseries_inputs = []

    for series_index, row in (
        selected_sales.iterrows()
    ):

        active_start_index = (
            int(row["first_active_day"]) - 1
        )

        target_history = demand_matrix[
            series_index,
            active_start_index:origin
        ]

        past_price = pd.Series(
            price_matrix_xgb[
                series_index,
                active_start_index:origin
            ]
        ).ffill().bfill()

        if past_price.isna().any():
            raise ValueError(
                "Missing active-period prices for "
                f"{row['id']}."
            )

        # Historical calendar covariates followed by
        # the 28 known future calendar observations.
        calendar_slice = calendar_xgb.iloc[
            active_start_index:
            origin + FORECAST_HORIZON
        ]

        future_known_covariates = (
            build_tirex_calendar_covariates(
                calendar_slice=calendar_slice,
                state_id=str(row["state_id"])
            )
        )

        timeseries_input = TimeseriesType(
            target=torch.tensor(
                target_history,
                dtype=torch.float32
            ).unsqueeze(0),

            # Prices are included only through the origin.
            past_covariates=torch.tensor(
                past_price.to_numpy(
                    dtype=np.float32
                ),
                dtype=torch.float32
            ).unsqueeze(0),

            # These contain only calendar, event and
            # SNAP information known in advance.
            future_covariates=torch.tensor(
                future_known_covariates,
                dtype=torch.float32
            )
        )

        timeseries_inputs.append(
            timeseries_input
        )

    return timeseries_inputs


# ------------------------------------------------------------
# 4. Load TiRex-2 once
# ------------------------------------------------------------

print("Loading TiRex-2...")

tirex_weekly_model = load_model(
    TIREX_CHECKPOINT,
    device=TIREX_DEVICE
)

tirex_quantile_levels = np.asarray(
    tirex_weekly_model._quantile_levels(),
    dtype=float
)

expected_quantile_levels = np.arange(
    0.1,
    1.0,
    0.1
)

if not np.allclose(
    tirex_quantile_levels,
    expected_quantile_levels
):
    raise ValueError(
        "Unexpected TiRex-2 quantile levels: "
        f"{tirex_quantile_levels.tolist()}"
    )

median_index = int(
    np.flatnonzero(
        np.isclose(
            tirex_quantile_levels,
            0.50
        )
    )[0]
)

print(
    "TiRex-2 quantiles:",
    tirex_quantile_levels
)

print(
    "Central forecast quantile:",
    tirex_quantile_levels[median_index]
)


# ------------------------------------------------------------
# 5. Run the twelve additional weekly origins
# ------------------------------------------------------------

def run_additional_weekly_tirex():

    forecast_parts = []
    diagnostic_parts = []

    total_start = time.perf_counter()

    for origin in ADDITIONAL_WEEKLY_ORIGINS:

        forecast_file = (
            OUTPUT_DIR
            / f"inventory_weekly_tirex2_origin_{origin}.csv"
        )

        diagnostics_file = (
            OUTPUT_DIR
            / f"inventory_weekly_tirex2_diagnostics_origin_{origin}.csv"
        )

        # Load a completed origin if the cell is rerun.
        if (
            forecast_file.exists()
            and diagnostics_file.exists()
        ):
            saved_forecasts = pd.read_csv(
                forecast_file
            )

            saved_diagnostics = pd.read_csv(
                diagnostics_file
            )

            valid_checkpoint = (
                len(saved_forecasts)
                == len(selected_sales)
                * FORECAST_HORIZON
                and saved_forecasts[
                    "id"
                ].nunique()
                == len(selected_sales)
                and saved_forecasts[
                    "forecast_origin"
                ].eq(origin).all()
                and saved_forecasts[
                    "forecast"
                ].notna().all()
                and np.isfinite(
                    saved_forecasts["forecast"]
                ).all()
            )

            if valid_checkpoint:
                forecast_parts.append(
                    saved_forecasts
                )

                diagnostic_parts.append(
                    saved_diagnostics
                )

                print(
                    f"TiRex-2, d_{origin}: "
                    "complete checkpoint loaded."
                )

                continue

        origin_start = time.perf_counter()

        tirex_inputs = (
            build_weekly_tirex_inputs(origin)
        )

        inference_start = time.perf_counter()

        tirex_outputs = (
            tirex_weekly_model.forecast(
                tirex_inputs,
                prediction_length=(
                    FORECAST_HORIZON
                ),
                output_type="numpy",
                batch_size=TIREX_BATCH_SIZE
            )
        )

        inference_seconds = (
            time.perf_counter()
            - inference_start
        )

        if len(tirex_outputs) != len(
            selected_sales
        ):
            raise ValueError(
                "TiRex-2 returned the wrong "
                "number of series."
            )

        all_quantiles = np.stack(
            [
                np.asarray(
                    output,
                    dtype=float
                )[0]
                for output in tirex_outputs
            ],
            axis=0
        )

        expected_shape = (
            len(selected_sales),
            len(tirex_quantile_levels),
            FORECAST_HORIZON
        )

        if all_quantiles.shape != expected_shape:
            raise ValueError(
                f"TiRex-2 output shape "
                f"{all_quantiles.shape}; "
                f"expected {expected_shape}."
            )

        all_quantiles = np.maximum(
            all_quantiles,
            0.0
        )

        # Record native quantile crossings without
        # modifying the forecasts.
        quantile_differences = np.diff(
            all_quantiles,
            axis=1
        )

        crossing_count = int(
            np.sum(
                quantile_differences < -1e-8
            )
        )

        affected_series_days = int(
            np.sum(
                np.any(
                    quantile_differences < -1e-8,
                    axis=1
                )
            )
        )

        central_predictions = (
            all_quantiles[
                :,
                median_index,
                :
            ]
        )

        origin_forecasts = (
            build_weekly_forecast_frame(
                predictions=central_predictions,
                origin=origin,
                model_name="tirex2"
            )
        )

        origin_diagnostics = pd.DataFrame([{
            "model": "tirex2",
            "forecast_origin": origin,
            "series": len(selected_sales),
            "prediction_length":
                FORECAST_HORIZON,
            "quantile_count":
                len(tirex_quantile_levels),
            "central_quantile":
                float(
                    tirex_quantile_levels[
                        median_index
                    ]
                ),
            "device": TIREX_DEVICE,
            "batch_size": TIREX_BATCH_SIZE,
            "quantile_crossings":
                crossing_count,
            "affected_series_days":
                affected_series_days,
            "future_prices_used": False,
            "input_seconds":
                inference_start - origin_start,
            "inference_seconds":
                inference_seconds
        }])

        origin_forecasts.to_csv(
            forecast_file,
            index=False
        )

        origin_diagnostics.to_csv(
            diagnostics_file,
            index=False
        )

        forecast_parts.append(
            origin_forecasts
        )

        diagnostic_parts.append(
            origin_diagnostics
        )

        del tirex_inputs
        del tirex_outputs
        del all_quantiles
        del central_predictions

        gc.collect()

        if (
            TIREX_DEVICE == "mps"
            and hasattr(
                torch.mps,
                "empty_cache"
            )
        ):
            torch.mps.empty_cache()

        print(
            f"TiRex-2: completed origin d_{origin}."
        )

    weekly_forecasts = pd.concat(
        forecast_parts,
        ignore_index=True
    )

    weekly_diagnostics = pd.concat(
        diagnostic_parts,
        ignore_index=True
    )

    expected_rows = (
        len(selected_sales)
        * len(ADDITIONAL_WEEKLY_ORIGINS)
        * FORECAST_HORIZON
    )

    group_sizes = weekly_forecasts.groupby(
        ["id", "forecast_origin"]
    ).size()

    checks = pd.DataFrame({
        "check": [
            "Expected number of forecast rows",
            "Exactly 500 series",
            "Exactly twelve additional origins",
            "No missing forecasts",
            "Finite forecasts",
            "Non-negative forecasts",
            "28 rows per series and origin",
            "Nine native quantiles returned",
            "Q0.50 retained as central forecast",
            "Future prices excluded",
            "Missing actuals only after d_1941"
        ],
        "passed": [
            len(weekly_forecasts)
            == expected_rows,
            weekly_forecasts["id"].nunique()
            == len(selected_sales),
            weekly_forecasts[
                "forecast_origin"
            ].nunique()
            == len(ADDITIONAL_WEEKLY_ORIGINS),
            weekly_forecasts[
                "forecast"
            ].notna().all(),
            np.isfinite(
                weekly_forecasts["forecast"]
            ).all(),
            weekly_forecasts[
                "forecast"
            ].ge(0).all(),
            group_sizes.eq(
                FORECAST_HORIZON
            ).all(),
            weekly_diagnostics[
                "quantile_count"
            ].eq(9).all(),
            weekly_diagnostics[
                "central_quantile"
            ].eq(0.50).all(),
            weekly_diagnostics[
                "future_prices_used"
            ].eq(False).all(),
            (
                weekly_forecasts["actual"].isna()
                == weekly_forecasts[
                    "target_day"
                ].gt(MAX_OBSERVED_DAY)
            ).all()
        ]
    })

    if not checks["passed"].all():
        failed_checks = checks.loc[
            ~checks["passed"],
            "check"
        ].tolist()

        raise AssertionError(
            f"Additional TiRex-2 checks failed: "
            f"{failed_checks}"
        )

    forecast_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_tirex2_additional_forecasts.csv"
    )

    diagnostics_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_tirex2_additional_diagnostics.csv"
    )

    checks_output_file = (
        OUTPUT_DIR
        / "inventory_weekly_tirex2_checks.csv"
    )

    weekly_forecasts.to_csv(
        forecast_output_file,
        index=False
    )

    weekly_diagnostics.to_csv(
        diagnostics_output_file,
        index=False
    )

    checks.to_csv(
        checks_output_file,
        index=False
    )

    print(
        "\nTiRex-2 weekly forecasts "
        "completed successfully."
    )

    print(
        "Forecast rows:",
        len(weekly_forecasts)
    )

    print(
        "Native quantile crossings:",
        int(
            weekly_diagnostics[
                "quantile_crossings"
            ].sum()
        )
    )

    print(
        "Elapsed minutes:",
        round(
            (
                time.perf_counter()
                - total_start
            ) / 60,
            2
        )
    )

    print("\nChecks:")
    print(checks)

    print("\nSaved file:")
    print(forecast_output_file)

    return (
        weekly_forecasts,
        weekly_diagnostics,
        checks
    )


# ------------------------------------------------------------
# 6. Start TiRex-2
# ------------------------------------------------------------

tirex_weekly_forecasts, tirex_weekly_diagnostics, tirex_weekly_checks = (
    run_additional_weekly_tirex()
)


Selected device: mps
Loading TiRex-2...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TiRex-2 quantiles: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
Central forecast quantile: 0.5


W0830 20:42:07.693000 30528 site-packages/torch/_inductor/utils.py:1953] [0/0] Not enough SMs to use max_autotune_gemm mode


TiRex-2: completed origin d_1836.
TiRex-2: completed origin d_1843.
TiRex-2: completed origin d_1850.
TiRex-2: completed origin d_1864.
TiRex-2: completed origin d_1871.
TiRex-2: completed origin d_1878.
TiRex-2: completed origin d_1892.
TiRex-2: completed origin d_1899.
TiRex-2: completed origin d_1906.
TiRex-2: completed origin d_1920.
TiRex-2: completed origin d_1927.
TiRex-2: completed origin d_1934.

TiRex-2 weekly forecasts completed successfully.
Forecast rows: 168000
Native quantile crossings: 8973
Elapsed minutes: 37.67

Checks:
                                 check  passed
0     Expected number of forecast rows    True
1                   Exactly 500 series    True
2    Exactly twelve additional origins    True
3                 No missing forecasts    True
4                     Finite forecasts    True
5               Non-negative forecasts    True
6        28 rows per series and origin    True
7       Nine native quantiles returned    True
8   Q0.50 retained as central for

Forecast-combination code

In [144]:
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Weekly inventory origins
# ------------------------------------------------------------

BASE_INVENTORY_ORIGINS = [
    1829,
    1857,
    1885,
    1913
]

ALL_INVENTORY_ORIGINS = [
    1829, 1836, 1843, 1850,
    1857, 1864, 1871, 1878,
    1885, 1892, 1899, 1906,
    1913, 1920, 1927, 1934
]

MODEL_NAMES = [
    "seasonal_naive",
    "ets",
    "arima_sarima",
    "xgboost",
    "tirex2"
]

FORECAST_HORIZON = 28


# ------------------------------------------------------------
# 2. Standardise a forecast file
# ------------------------------------------------------------

def standardise_inventory_forecasts(
    frame,
    model_name
):

    frame = frame.copy()

    # Allow for alternative forecast-column names.
    if "forecast" not in frame.columns:

        alternative_columns = [
            "central_forecast",
            "prediction",
            "point_forecast"
        ]

        for column in alternative_columns:
            if column in frame.columns:
                frame = frame.rename(
                    columns={column: "forecast"}
                )
                break

    # Add the demand class if necessary.
    if "demand_class" not in frame.columns:
        frame = frame.merge(
            selected_sales[
                ["id", "demand_class"]
            ],
            on="id",
            how="left",
            validate="many_to_one"
        )

    required_columns = [
        "id",
        "demand_class",
        "forecast_origin",
        "horizon",
        "target_day",
        "actual",
        "forecast"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{model_name} is missing columns: "
            f"{missing_columns}"
        )

    frame["model"] = model_name

    frame["forecast_origin"] = (
        frame["forecast_origin"].astype(int)
    )

    frame["horizon"] = (
        frame["horizon"].astype(int)
    )

    frame["target_day"] = (
        frame["target_day"].astype(int)
    )

    frame["purpose"] = np.where(
        frame["forecast_origin"] < 1913,
        "inventory_warmup",
        "inventory_test"
    )

    origin_to_round = {
        origin: number
        for number, origin in enumerate(
            ALL_INVENTORY_ORIGINS,
            start=1
        )
    }

    frame["round"] = (
        frame["forecast_origin"]
        .map(origin_to_round)
        .astype(int)
    )

    frame["forecast_origin_label"] = (
        "d_"
        + frame["forecast_origin"].astype(str)
    )

    frame["target_day_label"] = (
        "d_"
        + frame["target_day"].astype(str)
    )

    return frame[
        [
            "id",
            "demand_class",
            "model",
            "purpose",
            "round",
            "forecast_origin",
            "forecast_origin_label",
            "horizon",
            "target_day",
            "target_day_label",
            "actual",
            "forecast"
        ]
    ]


# ------------------------------------------------------------
# 3. Load the four original origins
# ------------------------------------------------------------

original_files = {
    "seasonal_naive":
        OUTPUT_DIR / "seasonal_naive_forecasts.csv",

    "ets":
        OUTPUT_DIR / "ets_forecasts.csv",

    "arima_sarima":
        OUTPUT_DIR / "arima_sarima_forecasts.csv",

    "xgboost":
        OUTPUT_DIR / "xgboost_forecasts.csv"
}

all_forecast_parts = []

for model_name, file_path in original_files.items():

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing file: {file_path}"
        )

    frame = pd.read_csv(file_path)

    frame = frame.loc[
        frame["forecast_origin"].isin(
            BASE_INVENTORY_ORIGINS
        )
    ].copy()

    frame = standardise_inventory_forecasts(
        frame,
        model_name
    )

    all_forecast_parts.append(frame)

    print(
        f"Loaded original {model_name}:",
        len(frame)
    )


# ------------------------------------------------------------
# 4. Load the four original TiRex-2 origins
# ------------------------------------------------------------

tirex_original_parts = []

for origin in BASE_INVENTORY_ORIGINS:

    tirex_file = (
        OUTPUT_DIR
        / f"tirex2_central_forecasts_origin_{origin}.csv"
    )

    if not tirex_file.exists():
        raise FileNotFoundError(
            f"Missing TiRex-2 file: {tirex_file}"
        )

    origin_frame = pd.read_csv(
        tirex_file
    )

    tirex_original_parts.append(
        origin_frame
    )

tirex_original = pd.concat(
    tirex_original_parts,
    ignore_index=True
)

tirex_original = standardise_inventory_forecasts(
    tirex_original,
    "tirex2"
)

all_forecast_parts.append(
    tirex_original
)

print(
    "Loaded original tirex2:",
    len(tirex_original)
)


# ------------------------------------------------------------
# 5. Load the twelve additional weekly origins
# ------------------------------------------------------------

additional_files = {
    "seasonal_naive":
        OUTPUT_DIR
        / "inventory_weekly_seasonal_naive_additional_forecasts.csv",

    "ets":
        OUTPUT_DIR
        / "inventory_weekly_ets_additional_forecasts.csv",

    "arima_sarima":
        OUTPUT_DIR
        / "inventory_weekly_arima_sarima_additional_forecasts.csv",

    "xgboost":
        OUTPUT_DIR
        / "inventory_weekly_xgboost_additional_forecasts.csv",

    "tirex2":
        OUTPUT_DIR
        / "inventory_weekly_tirex2_additional_forecasts.csv"
}

for model_name, file_path in additional_files.items():

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing file: {file_path}"
        )

    frame = pd.read_csv(file_path)

    frame = standardise_inventory_forecasts(
        frame,
        model_name
    )

    all_forecast_parts.append(frame)

    print(
        f"Loaded additional {model_name}:",
        len(frame)
    )


# ------------------------------------------------------------
# 6. Combine the complete weekly panel
# ------------------------------------------------------------

inventory_weekly_forecasts = pd.concat(
    all_forecast_parts,
    ignore_index=True
)

inventory_weekly_forecasts = (
    inventory_weekly_forecasts
    .sort_values(
        [
            "model",
            "forecast_origin",
            "id",
            "horizon"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Validate the combined panel
# ------------------------------------------------------------

expected_rows_per_model = (
    500
    * len(ALL_INVENTORY_ORIGINS)
    * FORECAST_HORIZON
)

expected_total_rows = (
    len(MODEL_NAMES)
    * expected_rows_per_model
)

rows_per_model = (
    inventory_weekly_forecasts
    .groupby("model")
    .size()
)

origins_per_model = (
    inventory_weekly_forecasts
    .groupby("model")[
        "forecast_origin"
    ]
    .nunique()
)

series_per_model = (
    inventory_weekly_forecasts
    .groupby("model")["id"]
    .nunique()
)

rows_per_series_origin = (
    inventory_weekly_forecasts
    .groupby(
        [
            "model",
            "id",
            "forecast_origin"
        ]
    )
    .size()
)

duplicate_keys = [
    "model",
    "id",
    "forecast_origin",
    "horizon"
]

observed_rows = inventory_weekly_forecasts.loc[
    inventory_weekly_forecasts[
        "target_day"
    ].le(MAX_OBSERVED_DAY)
]

actual_demand_consistent = (
    observed_rows
    .groupby(
        [
            "id",
            "forecast_origin",
            "horizon"
        ]
    )["actual"]
    .nunique(dropna=False)
    .eq(1)
    .all()
)

combination_checks = pd.DataFrame({
    "check": [
        "Expected total number of rows",
        "Exactly five models",
        "Expected rows per model",
        "Exactly sixteen origins per model",
        "Exactly 500 series per model",
        "28 rows per series and origin",
        "No duplicate forecast keys",
        "No missing forecasts",
        "Finite forecasts",
        "Non-negative forecasts",
        "Target day equals origin plus horizon",
        "Actual demand identical across models",
        "Missing actuals only after d_1941"
    ],
    "passed": [
        len(inventory_weekly_forecasts)
        == expected_total_rows,

        inventory_weekly_forecasts[
            "model"
        ].nunique() == 5,

        rows_per_model.eq(
            expected_rows_per_model
        ).all(),

        origins_per_model.eq(16).all(),

        series_per_model.eq(500).all(),

        rows_per_series_origin.eq(
            FORECAST_HORIZON
        ).all(),

        not inventory_weekly_forecasts
        .duplicated(duplicate_keys)
        .any(),

        inventory_weekly_forecasts[
            "forecast"
        ].notna().all(),

        np.isfinite(
            inventory_weekly_forecasts[
                "forecast"
            ]
        ).all(),

        inventory_weekly_forecasts[
            "forecast"
        ].ge(0).all(),

        (
            inventory_weekly_forecasts[
                "target_day"
            ]
            ==
            inventory_weekly_forecasts[
                "forecast_origin"
            ]
            +
            inventory_weekly_forecasts[
                "horizon"
            ]
        ).all(),

        actual_demand_consistent,

        (
            inventory_weekly_forecasts[
                "actual"
            ].isna()
            ==
            inventory_weekly_forecasts[
                "target_day"
            ].gt(MAX_OBSERVED_DAY)
        ).all()
    ]
})

if not combination_checks["passed"].all():

    failed_checks = combination_checks.loc[
        ~combination_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Combination checks failed: "
        f"{failed_checks}"
    )


# ------------------------------------------------------------
# 8. Save the combined forecasts
# ------------------------------------------------------------

combined_output_file = (
    OUTPUT_DIR
    / "inventory_weekly_all_model_forecasts.csv"
)

checks_output_file = (
    OUTPUT_DIR
    / "inventory_weekly_combination_checks.csv"
)

schedule_output_file = (
    OUTPUT_DIR
    / "inventory_weekly_schedule_summary.csv"
)

inventory_weekly_forecasts.to_csv(
    combined_output_file,
    index=False
)

combination_checks.to_csv(
    checks_output_file,
    index=False
)

schedule_summary = (
    inventory_weekly_forecasts
    .groupby(
        [
            "model",
            "forecast_origin"
        ],
        as_index=False
    )
    .agg(
        series=("id", "nunique"),
        forecast_rows=("forecast", "size"),
        forecast_start=("target_day", "min"),
        forecast_end=("target_day", "max")
    )
)

schedule_summary.to_csv(
    schedule_output_file,
    index=False
)


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\nRows per model:")
print(rows_per_model)

print("\nCombination checks:")
print(combination_checks)

print(
    "\nTotal combined rows:",
    len(inventory_weekly_forecasts)
)

print("\nSaved file:")
print(combined_output_file)


Loaded original seasonal_naive: 56000
Loaded original ets: 56000
Loaded original arima_sarima: 56000
Loaded original xgboost: 56000
Loaded original tirex2: 56000
Loaded additional seasonal_naive: 168000
Loaded additional ets: 168000
Loaded additional arima_sarima: 168000
Loaded additional xgboost: 168000
Loaded additional tirex2: 168000

Rows per model:
model
arima_sarima      224000
ets               224000
seasonal_naive    224000
tirex2            224000
xgboost           224000
dtype: int64

Combination checks:
                                    check  passed
0           Expected total number of rows    True
1                     Exactly five models    True
2                 Expected rows per model    True
3       Exactly sixteen origins per model    True
4            Exactly 500 series per model    True
5           28 rows per series and origin    True
6              No duplicate forecast keys    True
7                    No missing forecasts    True
8                        Fini

We now construct the residual-based 28-day order-up-to targets for the three policy quantiles: 0.90, 0.95, and 0.99.

In [147]:
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Residual-estimation origins and quantiles
# ------------------------------------------------------------

CALIBRATION_ORIGINS = [
    1661,
    1689,
    1717,
    1745,
    1773,
    1801
]

VALIDATION_ORIGINS = [
    1829,
    1857,
    1885
]

RESIDUAL_ORIGINS = (
    CALIBRATION_ORIGINS
    + VALIDATION_ORIGINS
)

INVENTORY_QUANTILES = [
    0.90,
    0.95,
    0.99
]

PROTECTION_PERIOD = 28


# ------------------------------------------------------------
# 2. Prepare an original forecast file
# ------------------------------------------------------------

def prepare_residual_forecasts(
    frame,
    model_name
):

    frame = frame.copy()

    if "forecast" not in frame.columns:

        alternative_columns = [
            "central_forecast",
            "prediction",
            "point_forecast"
        ]

        for column in alternative_columns:
            if column in frame.columns:
                frame = frame.rename(
                    columns={column: "forecast"}
                )
                break

    if "demand_class" not in frame.columns:
        frame = frame.merge(
            selected_sales[
                ["id", "demand_class"]
            ],
            on="id",
            how="left",
            validate="many_to_one"
        )

    required_columns = [
        "id",
        "demand_class",
        "forecast_origin",
        "horizon",
        "target_day",
        "actual",
        "forecast"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{model_name} is missing columns: "
            f"{missing_columns}"
        )

    frame = frame.loc[
        frame["forecast_origin"].isin(
            RESIDUAL_ORIGINS
        )
    ].copy()

    frame["model"] = model_name

    return frame[
        [
            "id",
            "demand_class",
            "model",
            "forecast_origin",
            "horizon",
            "target_day",
            "actual",
            "forecast"
        ]
    ]


# ------------------------------------------------------------
# 3. Load calibration and validation forecasts
# ------------------------------------------------------------

residual_forecast_parts = []

original_model_files = {
    "seasonal_naive":
        OUTPUT_DIR / "seasonal_naive_forecasts.csv",

    "ets":
        OUTPUT_DIR / "ets_forecasts.csv",

    "arima_sarima":
        OUTPUT_DIR / "arima_sarima_forecasts.csv",

    "xgboost":
        OUTPUT_DIR / "xgboost_forecasts.csv"
}

for model_name, file_path in (
    original_model_files.items()
):

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing file: {file_path}"
        )

    frame = pd.read_csv(file_path)

    frame = prepare_residual_forecasts(
        frame,
        model_name
    )

    residual_forecast_parts.append(frame)

    print(
        f"Loaded residual forecasts for "
        f"{model_name}:",
        len(frame)
    )


# TiRex-2 files are stored separately by origin.
tirex_residual_parts = []

for origin in RESIDUAL_ORIGINS:

    tirex_file = (
        OUTPUT_DIR
        / f"tirex2_central_forecasts_origin_{origin}.csv"
    )

    if not tirex_file.exists():
        raise FileNotFoundError(
            f"Missing TiRex-2 file: "
            f"{tirex_file}"
        )

    frame = pd.read_csv(tirex_file)

    frame = prepare_residual_forecasts(
        frame,
        "tirex2"
    )

    tirex_residual_parts.append(frame)

tirex_residual_forecasts = pd.concat(
    tirex_residual_parts,
    ignore_index=True
)

residual_forecast_parts.append(
    tirex_residual_forecasts
)

print(
    "Loaded residual forecasts for tirex2:",
    len(tirex_residual_forecasts)
)

residual_forecasts = pd.concat(
    residual_forecast_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Calculate historical 28-day demand scales
# ------------------------------------------------------------

scale_origins = sorted(
    set(
        RESIDUAL_ORIGINS
        + ALL_INVENTORY_ORIGINS
    )
)

scale_rows = []

for origin in scale_origins:

    for series_index, row in (
        selected_sales.iterrows()
    ):

        active_start_index = (
            int(row["first_active_day"]) - 1
        )

        historical_demand = demand_matrix[
            series_index,
            active_start_index:origin
        ]

        if len(historical_demand) < PROTECTION_PERIOD:
            raise ValueError(
                f"Insufficient history for "
                f"{row['id']} at d_{origin}."
            )

        rolling_totals = np.convolve(
            historical_demand,
            np.ones(PROTECTION_PERIOD),
            mode="valid"
        )

        if len(rolling_totals) >= 2:
            rolling_standard_deviation = (
                np.std(
                    rolling_totals,
                    ddof=1
                )
            )
        else:
            rolling_standard_deviation = 0.0

        protection_scale = max(
            1.0,
            float(
                rolling_standard_deviation
            )
        )

        scale_rows.append({
            "id": row["id"],
            "demand_class":
                row["demand_class"],
            "scale_origin": origin,
            "protection_scale":
                protection_scale,
            "rolling_totals_available":
                len(rolling_totals)
        })

    print(
        f"Completed protection scales "
        f"at d_{origin}."
    )

protection_scales = pd.DataFrame(
    scale_rows
)


# ------------------------------------------------------------
# 5. Calculate one 28-day error per series and origin
# ------------------------------------------------------------

protection_errors = (
    residual_forecasts
    .groupby(
        [
            "id",
            "demand_class",
            "model",
            "forecast_origin"
        ],
        as_index=False
    )
    .agg(
        actual_protection_demand=(
            "actual",
            "sum"
        ),
        forecast_protection_demand=(
            "forecast",
            "sum"
        ),
        forecast_days=(
            "horizon",
            "size"
        )
    )
)

protection_errors[
    "protection_period_error"
] = (
    protection_errors[
        "actual_protection_demand"
    ]
    -
    protection_errors[
        "forecast_protection_demand"
    ]
)

error_scales = protection_scales.rename(
    columns={
        "scale_origin":
            "forecast_origin"
    }
)

protection_errors = (
    protection_errors
    .merge(
        error_scales[
            [
                "id",
                "forecast_origin",
                "protection_scale"
            ]
        ],
        on=[
            "id",
            "forecast_origin"
        ],
        how="left",
        validate="many_to_one"
    )
)

protection_errors[
    "standardised_error"
] = (
    protection_errors[
        "protection_period_error"
    ]
    /
    protection_errors[
        "protection_scale"
    ]
)

protection_errors[
    "error_observed_day"
] = (
    protection_errors[
        "forecast_origin"
    ]
    + PROTECTION_PERIOD
)


# ------------------------------------------------------------
# 6. Construct sequential residual pools
# ------------------------------------------------------------

residual_quantile_rows = []

for decision_origin in ALL_INVENTORY_ORIGINS:

    # Only errors whose full 28-day periods have
    # finished by the decision origin are available.
    available_errors = (
        protection_errors.loc[
            protection_errors[
                "error_observed_day"
            ].le(decision_origin)
        ]
    )

    grouped_errors = available_errors.groupby(
        [
            "model",
            "demand_class"
        ]
    )

    for (
        model_name,
        demand_class
    ), group in grouped_errors:

        standardised_errors = group[
            "standardised_error"
        ].to_numpy(dtype=float)

        for quantile_level in (
            INVENTORY_QUANTILES
        ):

            residual_quantile = float(
                np.quantile(
                    standardised_errors,
                    quantile_level,
                    method="linear"
                )
            )

            residual_quantile_rows.append({
                "forecast_origin":
                    decision_origin,
                "model":
                    model_name,
                "demand_class":
                    demand_class,
                "quantile_level":
                    quantile_level,
                "standardised_error_quantile":
                    residual_quantile,
                "pool_size":
                    len(standardised_errors),
                "earliest_error_origin":
                    int(
                        group[
                            "forecast_origin"
                        ].min()
                    ),
                "latest_error_origin":
                    int(
                        group[
                            "forecast_origin"
                        ].max()
                    ),
                "latest_error_observed_day":
                    int(
                        group[
                            "error_observed_day"
                        ].max()
                    )
            })

sequential_residual_quantiles = (
    pd.DataFrame(
        residual_quantile_rows
    )
)


# ------------------------------------------------------------
# 7. Aggregate weekly central forecasts to 28-day totals
# ------------------------------------------------------------

weekly_forecast_totals = (
    inventory_weekly_forecasts
    .groupby(
        [
            "id",
            "demand_class",
            "model",
            "forecast_origin"
        ],
        as_index=False
    )
    .agg(
        forecast_protection_demand=(
            "forecast",
            "sum"
        ),
        forecast_days=(
            "horizon",
            "size"
        )
    )
)

current_scales = (
    protection_scales
    .rename(
        columns={
            "scale_origin":
                "forecast_origin"
        }
    )
)

weekly_forecast_totals = (
    weekly_forecast_totals
    .merge(
        current_scales[
            [
                "id",
                "forecast_origin",
                "protection_scale"
            ]
        ],
        on=[
            "id",
            "forecast_origin"
        ],
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# 8. Construct safety stocks and order-up-to targets
# ------------------------------------------------------------

inventory_targets = (
    weekly_forecast_totals
    .merge(
        sequential_residual_quantiles[
            [
                "forecast_origin",
                "model",
                "demand_class",
                "quantile_level",
                "standardised_error_quantile",
                "pool_size"
            ]
        ],
        on=[
            "forecast_origin",
            "model",
            "demand_class"
        ],
        how="left",
        validate="many_to_many"
    )
)

inventory_targets["safety_stock"] = (
    np.maximum(
        0.0,
        inventory_targets[
            "protection_scale"
        ]
        *
        inventory_targets[
            "standardised_error_quantile"
        ]
    )
)

inventory_targets["order_up_to_target"] = (
    np.maximum(
        0.0,
        inventory_targets[
            "forecast_protection_demand"
        ]
        +
        inventory_targets[
            "safety_stock"
        ]
    )
)

inventory_targets = (
    inventory_targets
    .sort_values(
        [
            "model",
            "quantile_level",
            "forecast_origin",
            "id"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Validation checks
# ------------------------------------------------------------

expected_error_rows = (
    5
    * 500
    * len(RESIDUAL_ORIGINS)
)

expected_quantile_rows = (
    5
    * 4
    * len(ALL_INVENTORY_ORIGINS)
    * len(INVENTORY_QUANTILES)
)

expected_target_rows = (
    5
    * 500
    * len(ALL_INVENTORY_ORIGINS)
    * len(INVENTORY_QUANTILES)
)

expected_pool_sizes = {}

for origin in ALL_INVENTORY_ORIGINS:

    available_origin_count = sum(
        residual_origin
        + PROTECTION_PERIOD
        <= origin
        for residual_origin
        in RESIDUAL_ORIGINS
    )

    expected_pool_sizes[origin] = (
        125 * available_origin_count
    )

pool_sizes_correct = (
    sequential_residual_quantiles.apply(
        lambda row:
            row["pool_size"]
            ==
            expected_pool_sizes[
                int(row["forecast_origin"])
            ],
        axis=1
    ).all()
)

target_formula_correct = np.allclose(
    inventory_targets[
        "order_up_to_target"
    ],
    np.maximum(
        0.0,
        inventory_targets[
            "forecast_protection_demand"
        ]
        +
        inventory_targets[
            "safety_stock"
        ]
    )
)

target_checks = pd.DataFrame({
    "check": [
        "Expected number of protection errors",
        "Exactly 28 days per protection error",
        "Positive finite protection scales",
        "Finite standardised errors",
        "Correct sequential pool sizes",
        "No future errors in residual pools",
        "Expected residual-quantile rows",
        "Exactly three quantile levels",
        "Expected order-up-to target rows",
        "No missing order-up-to targets",
        "Finite order-up-to targets",
        "Non-negative safety stock",
        "Target not below central forecast",
        "Order-up-to formula correct"
    ],
    "passed": [
        len(protection_errors)
        == expected_error_rows,

        protection_errors[
            "forecast_days"
        ].eq(PROTECTION_PERIOD).all(),

        (
            protection_errors[
                "protection_scale"
            ].gt(0).all()
            and np.isfinite(
                protection_errors[
                    "protection_scale"
                ]
            ).all()
        ),

        np.isfinite(
            protection_errors[
                "standardised_error"
            ]
        ).all(),

        pool_sizes_correct,

        (
            sequential_residual_quantiles[
                "latest_error_observed_day"
            ]
            <=
            sequential_residual_quantiles[
                "forecast_origin"
            ]
        ).all(),

        len(
            sequential_residual_quantiles
        ) == expected_quantile_rows,

        set(
            sequential_residual_quantiles[
                "quantile_level"
            ].unique()
        )
        == set(INVENTORY_QUANTILES),

        len(inventory_targets)
        == expected_target_rows,

        inventory_targets[
            "order_up_to_target"
        ].notna().all(),

        np.isfinite(
            inventory_targets[
                "order_up_to_target"
            ]
        ).all(),

        inventory_targets[
            "safety_stock"
        ].ge(0).all(),

        (
            inventory_targets[
                "order_up_to_target"
            ]
            + 1e-10
            >=
            inventory_targets[
                "forecast_protection_demand"
            ]
        ).all(),

        target_formula_correct
    ]
})

if not target_checks["passed"].all():

    failed_checks = target_checks.loc[
        ~target_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Inventory-target checks failed: "
        f"{failed_checks}"
    )


# ------------------------------------------------------------
# 10. Save the results
# ------------------------------------------------------------

protection_scales.to_csv(
    OUTPUT_DIR
    / "inventory_protection_period_scales.csv",
    index=False
)

protection_errors.to_csv(
    OUTPUT_DIR
    / "inventory_standardised_protection_errors.csv",
    index=False
)

sequential_residual_quantiles.to_csv(
    OUTPUT_DIR
    / "inventory_sequential_residual_quantiles.csv",
    index=False
)

inventory_targets.to_csv(
    OUTPUT_DIR
    / "inventory_weekly_order_up_to_targets.csv",
    index=False
)

target_checks.to_csv(
    OUTPUT_DIR
    / "inventory_order_up_to_target_checks.csv",
    index=False
)


# ------------------------------------------------------------
# 11. Display results
# ------------------------------------------------------------

pool_size_summary = (
    sequential_residual_quantiles
    .groupby(
        "forecast_origin",
        as_index=False
    )
    .agg(
        minimum_pool_size=(
            "pool_size",
            "min"
        ),
        maximum_pool_size=(
            "pool_size",
            "max"
        )
    )
)

print(
    "Protection-error rows:",
    len(protection_errors)
)

print(
    "Order-up-to target rows:",
    len(inventory_targets)
)

print("\nSequential pool sizes:")
print(pool_size_summary)

print("\nTarget checks:")
print(target_checks)

print("\nSaved target file:")
print(
    OUTPUT_DIR
    / "inventory_weekly_order_up_to_targets.csv"
)


Loaded residual forecasts for seasonal_naive: 126000
Loaded residual forecasts for ets: 126000
Loaded residual forecasts for arima_sarima: 126000
Loaded residual forecasts for xgboost: 126000
Loaded residual forecasts for tirex2: 126000
Completed protection scales at d_1661.
Completed protection scales at d_1689.
Completed protection scales at d_1717.
Completed protection scales at d_1745.
Completed protection scales at d_1773.
Completed protection scales at d_1801.
Completed protection scales at d_1829.
Completed protection scales at d_1836.
Completed protection scales at d_1843.
Completed protection scales at d_1850.
Completed protection scales at d_1857.
Completed protection scales at d_1864.
Completed protection scales at d_1871.
Completed protection scales at d_1878.
Completed protection scales at d_1885.
Completed protection scales at d_1892.
Completed protection scales at d_1899.
Completed protection scales at d_1906.
Completed protection scales at d_1913.
Completed protection s

The following code runs the daily lost-sales inventory simulation

In [152]:
# ------------------------------------------------------------
# Corrected Step 12: Fill-rate eligibility and ranking
# ------------------------------------------------------------

MINIMUM_FILL_RATE = 0.95

inventory_summary[
    "meets_minimum_fill_rate"
] = (
    inventory_summary[
        "fill_rate"
    ].ge(MINIMUM_FILL_RATE)
)

inventory_summary[
    "eligible_cost_rank"
] = np.nan

eligible_mask = inventory_summary[
    "meets_minimum_fill_rate"
]

eligible_ranks = (
    inventory_summary.loc[
        eligible_mask
    ]
    .groupby(
        [
            "scenario",
            "demand_class",
            "quantile_level"
        ]
    )["total_cost"]
    .rank(
        method="min",
        ascending=True
    )
)

inventory_summary.loc[
    eligible_ranks.index,
    "eligible_cost_rank"
] = eligible_ranks


# ------------------------------------------------------------
# 13. Pareto-efficient alternatives
# ------------------------------------------------------------

inventory_summary[
    "pareto_efficient"
] = True

for (
    scenario_name,
    demand_group
), group in inventory_summary.groupby(
    [
        "scenario",
        "demand_class"
    ]
):

    group_indices = group.index.to_numpy()

    costs = group[
        "total_cost"
    ].to_numpy(dtype=float)

    fill_rates = group[
        "fill_rate"
    ].to_numpy(dtype=float)

    dominated = np.zeros(
        len(group),
        dtype=bool
    )

    for candidate_index in range(
        len(group)
    ):

        better_or_equal_cost = (
            costs
            <= costs[candidate_index]
        )

        better_or_equal_service = (
            fill_rates
            >= fill_rates[candidate_index]
        )

        strictly_better = (
            (
                costs
                < costs[candidate_index]
            )
            |
            (
                fill_rates
                > fill_rates[candidate_index]
            )
        )

        dominated[candidate_index] = np.any(
            better_or_equal_cost
            &
            better_or_equal_service
            &
            strictly_better
        )

    inventory_summary.loc[
        group_indices,
        "pareto_efficient"
    ] = ~dominated


# ------------------------------------------------------------
# 14. Final implementation checks
# ------------------------------------------------------------

expected_daily_rows = (
    500
    * 5
    * 3
    * 112
)

expected_test_rows = (
    500
    * 5
    * 3
    * 28
)

demand_identical = (
    inventory_daily
    .groupby(
        ["id", "day"]
    )["demand"]
    .nunique(dropna=False)
    .eq(1)
    .all()
)

simulation_checks = pd.DataFrame({
    "check": [
        "Expected number of daily rows",
        "Exactly 84 warm-up days",
        "Exactly 28 final-test days",
        "Expected number of final-test rows",
        "Demand identical across methods",
        "No negative ending inventory",
        "No negative lost sales",
        "No negative orders",
        "No negative pipeline",
        "Demand equals sales plus lost sales",
        "Orders occur only on review days",
        "Finite positive final-test prices",
        "Expected number of initial states",
        "Expected number of cost-scenario rows",
        "Non-negative costs",
        "Fill rates between zero and one",
        "Cycle service levels between zero and one"
    ],
    "passed": [
        len(inventory_daily)
        == expected_daily_rows,

        inventory_daily.loc[
            inventory_daily[
                "simulation_period"
            ].eq("validation_warmup"),
            "day"
        ].nunique() == 84,

        inventory_final_test[
            "day"
        ].nunique() == 28,

        len(inventory_final_test)
        == expected_test_rows,

        demand_identical,

        inventory_daily[
            "ending_inventory"
        ].ge(-accounting_tolerance).all(),

        inventory_daily[
            "lost_sales"
        ].ge(-accounting_tolerance).all(),

        inventory_daily[
            "order_quantity"
        ].ge(-accounting_tolerance).all(),

        inventory_daily[
            "ending_pipeline"
        ].ge(-accounting_tolerance).all(),

        np.allclose(
            inventory_daily["demand"],
            inventory_daily[
                "fulfilled_sales"
            ]
            +
            inventory_daily[
                "lost_sales"
            ],
            atol=accounting_tolerance
        ),

        inventory_daily.loc[
            inventory_daily[
                "review_day"
            ].eq(0),
            "order_quantity"
        ].le(accounting_tolerance).all(),

        (
            np.isfinite(
                inventory_final_test[
                    "unit_price"
                ]
            ).all()
            and inventory_final_test[
                "unit_price"
            ].gt(0).all()
        ),

        len(inventory_initial_states)
        == 500 * 5 * 3,

        len(inventory_series_results)
        == (
            500
            * 5
            * 3
            * len(COST_SCENARIOS)
        ),

        inventory_series_results[
            [
                "holding_cost",
                "ordering_cost",
                "lost_sales_cost",
                "total_cost"
            ]
        ].ge(-accounting_tolerance)
        .all().all(),

        inventory_summary[
            "fill_rate"
        ].between(0, 1).all(),

        inventory_summary[
            "cycle_service_level"
        ].between(0, 1).all()
    ]
})

if not simulation_checks[
    "passed"
].all():

    failed_checks = simulation_checks.loc[
        ~simulation_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Inventory simulation checks failed: "
        f"{failed_checks}"
    )


# ------------------------------------------------------------
# 15. Save all inventory outputs
# ------------------------------------------------------------

inventory_initial_states.to_csv(
    OUTPUT_DIR
    / "inventory_initial_states.csv",
    index=False
)

inventory_daily.to_csv(
    OUTPUT_DIR
    / "inventory_daily_trajectories.csv",
    index=False
)

cycle_results.to_csv(
    OUTPUT_DIR
    / "inventory_cycle_results.csv",
    index=False
)

series_physical_results.to_csv(
    OUTPUT_DIR
    / "inventory_series_physical_results.csv",
    index=False
)

inventory_series_results.to_csv(
    OUTPUT_DIR
    / "inventory_series_cost_scenarios.csv",
    index=False
)

inventory_summary.to_csv(
    OUTPUT_DIR
    / "inventory_summary.csv",
    index=False
)

simulation_checks.to_csv(
    OUTPUT_DIR
    / "inventory_simulation_checks.csv",
    index=False
)


# ------------------------------------------------------------
# 16. Display baseline overall results
# ------------------------------------------------------------

baseline_overall_results = (
    inventory_summary.loc[
        (
            inventory_summary[
                "scenario"
            ].eq("baseline")
        )
        &
        (
            inventory_summary[
                "demand_class"
            ].eq("Overall")
        )
    ]
    [
        [
            "model",
            "quantile_level",
            "holding_cost",
            "ordering_cost",
            "lost_sales_cost",
            "total_cost",
            "fill_rate",
            "cycle_service_level",
            "average_inventory",
            "order_count",
            "meets_minimum_fill_rate",
            "eligible_cost_rank",
            "pareto_efficient"
        ]
    ]
    .sort_values(
        [
            "quantile_level",
            "total_cost"
        ]
    )
)

print(
    "Inventory simulation completed successfully."
)

print(
    "\nDaily trajectory rows:",
    len(inventory_daily)
)

print(
    "\nFinal-test rows:",
    len(inventory_final_test)
)

print("\nSimulation checks:")
print(simulation_checks)

print("\nBaseline overall results:")
print(
    baseline_overall_results.to_string(
        index=False
    )
)

print("\nSaved summary file:")
print(
    OUTPUT_DIR
    / "inventory_summary.csv"
)


Inventory simulation completed successfully.

Daily trajectory rows: 840000

Final-test rows: 210000

Simulation checks:
                                        check  passed
0               Expected number of daily rows    True
1                     Exactly 84 warm-up days    True
2                  Exactly 28 final-test days    True
3          Expected number of final-test rows    True
4             Demand identical across methods    True
5                No negative ending inventory    True
6                      No negative lost sales    True
7                          No negative orders    True
8                        No negative pipeline    True
9         Demand equals sales plus lost sales    True
10           Orders occur only on review days    True
11          Finite positive final-test prices    True
12          Expected number of initial states    True
13      Expected number of cost-scenario rows    True
14                         Non-negative costs    True
15            F

In [154]:
import itertools

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Bootstrap settings
# ------------------------------------------------------------

BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2026
CONFIDENCE_LEVEL = 0.95

LOWER_PERCENTILE = (
    100 * (1 - CONFIDENCE_LEVEL) / 2
)

UPPER_PERCENTILE = (
    100 - LOWER_PERCENTILE
)

DEMAND_GROUPS = [
    "Overall",
    "Smooth",
    "Erratic",
    "Intermittent",
    "Lumpy"
]

MODEL_PAIRS = list(
    itertools.combinations(
        MODEL_NAMES,
        2
    )
)

bootstrap_random_generator = (
    np.random.default_rng(
        BOOTSTRAP_SEED
    )
)


# ------------------------------------------------------------
# 2. Pivot helper
# ------------------------------------------------------------

def make_model_matrix(
    frame,
    value_column
):

    matrix = (
        frame
        .pivot(
            index=[
                "demand_class",
                "id"
            ],
            columns="model",
            values=value_column
        )
        .reindex(
            columns=MODEL_NAMES
        )
        .sort_index()
    )

    if matrix.isna().any().any():
        raise ValueError(
            f"Missing paired values in "
            f"{value_column}."
        )

    return matrix


# ------------------------------------------------------------
# 3. Construct paired bootstrap samples
# ------------------------------------------------------------

def construct_bootstrap_indices(
    index,
    demand_group
):

    number_of_rows = len(index)

    if demand_group != "Overall":

        return (
            bootstrap_random_generator
            .integers(
                low=0,
                high=number_of_rows,
                size=(
                    BOOTSTRAP_RESAMPLES,
                    number_of_rows
                )
            )
        )

    # For the overall result, resample separately
    # within each demand class.
    class_values = (
        index.get_level_values(
            "demand_class"
        ).to_numpy()
    )

    sampled_class_parts = []

    for demand_class in [
        "Smooth",
        "Erratic",
        "Intermittent",
        "Lumpy"
    ]:

        class_positions = np.flatnonzero(
            class_values == demand_class
        )

        sampled_positions = (
            bootstrap_random_generator.choice(
                class_positions,
                size=(
                    BOOTSTRAP_RESAMPLES,
                    len(class_positions)
                ),
                replace=True
            )
        )

        sampled_class_parts.append(
            sampled_positions
        )

    return np.concatenate(
        sampled_class_parts,
        axis=1
    )


# ------------------------------------------------------------
# 4. Run the paired bootstrap
# ------------------------------------------------------------

bootstrap_rows = []

for quantile_level in INVENTORY_QUANTILES:

    physical_quantile = (
        series_physical_results.loc[
            np.isclose(
                series_physical_results[
                    "quantile_level"
                ],
                quantile_level
            )
        ]
    )

    for demand_group in DEMAND_GROUPS:

        if demand_group == "Overall":
            physical_group = (
                physical_quantile.copy()
            )
        else:
            physical_group = (
                physical_quantile.loc[
                    physical_quantile[
                        "demand_class"
                    ].eq(demand_group)
                ]
            )

        fulfilled_matrix_frame = (
            make_model_matrix(
                physical_group,
                "fulfilled_units"
            )
        )

        demand_matrix_frame = (
            make_model_matrix(
                physical_group,
                "total_demand"
            )
        )

        if not np.allclose(
            demand_matrix_frame.to_numpy(),
            demand_matrix_frame[
                MODEL_NAMES[0]
            ].to_numpy()[:, None]
        ):
            raise AssertionError(
                "Demand is not identical "
                "across models."
            )

        paired_index = (
            fulfilled_matrix_frame.index
        )

        bootstrap_indices = (
            construct_bootstrap_indices(
                paired_index,
                demand_group
            )
        )

        fulfilled_matrix_bootstrap = (
            fulfilled_matrix_frame
            .to_numpy(dtype=float)
        )

        demand_vector = (
            demand_matrix_frame[
                MODEL_NAMES[0]
            ].to_numpy(dtype=float)
        )

        fulfilled_draws = np.column_stack([
            fulfilled_matrix_bootstrap[
                bootstrap_indices,
                model_index
            ].sum(axis=1)

            for model_index in range(
                len(MODEL_NAMES)
            )
        ])

        demand_draws = demand_vector[
            bootstrap_indices
        ].sum(axis=1)

        if np.any(demand_draws <= 0):
            raise ValueError(
                "A bootstrap sample contains "
                "zero total demand."
            )

        fill_rate_draws = (
            fulfilled_draws
            / demand_draws[:, None]
        )

        point_fill_rates = (
            fulfilled_matrix_bootstrap.sum(
                axis=0
            )
            / demand_vector.sum()
        )

        for scenario_name in (
            COST_SCENARIOS
        ):

            cost_group = (
                inventory_series_results.loc[
                    (
                        inventory_series_results[
                            "scenario"
                        ].eq(scenario_name)
                    )
                    &
                    (
                        np.isclose(
                            inventory_series_results[
                                "quantile_level"
                            ],
                            quantile_level
                        )
                    )
                ]
            )

            if demand_group != "Overall":
                cost_group = cost_group.loc[
                    cost_group[
                        "demand_class"
                    ].eq(demand_group)
                ]

            cost_matrix_frame = (
                make_model_matrix(
                    cost_group,
                    "total_cost"
                )
                .reindex(paired_index)
            )

            if cost_matrix_frame.isna().any().any():
                raise ValueError(
                    "Cost data are not completely "
                    "paired."
                )

            cost_matrix = (
                cost_matrix_frame.to_numpy(
                    dtype=float
                )
            )

            cost_draws = np.column_stack([
                cost_matrix[
                    bootstrap_indices,
                    model_index
                ].sum(axis=1)

                for model_index in range(
                    len(MODEL_NAMES)
                )
            ])

            point_total_costs = (
                cost_matrix.sum(axis=0)
            )

            for (
                model_a,
                model_b
            ) in MODEL_PAIRS:

                model_a_index = (
                    MODEL_NAMES.index(model_a)
                )

                model_b_index = (
                    MODEL_NAMES.index(model_b)
                )

                # Difference is model A minus model B.
                cost_difference_draws = (
                    cost_draws[
                        :,
                        model_a_index
                    ]
                    -
                    cost_draws[
                        :,
                        model_b_index
                    ]
                )

                fill_difference_draws = (
                    (
                        fill_rate_draws[
                            :,
                            model_a_index
                        ]
                        -
                        fill_rate_draws[
                            :,
                            model_b_index
                        ]
                    )
                    * 100
                )

                point_cost_difference = (
                    point_total_costs[
                        model_a_index
                    ]
                    -
                    point_total_costs[
                        model_b_index
                    ]
                )

                point_fill_difference = (
                    (
                        point_fill_rates[
                            model_a_index
                        ]
                        -
                        point_fill_rates[
                            model_b_index
                        ]
                    )
                    * 100
                )

                cost_ci_lower = float(
                    np.percentile(
                        cost_difference_draws,
                        LOWER_PERCENTILE
                    )
                )

                cost_ci_upper = float(
                    np.percentile(
                        cost_difference_draws,
                        UPPER_PERCENTILE
                    )
                )

                fill_ci_lower = float(
                    np.percentile(
                        fill_difference_draws,
                        LOWER_PERCENTILE
                    )
                )

                fill_ci_upper = float(
                    np.percentile(
                        fill_difference_draws,
                        UPPER_PERCENTILE
                    )
                )

                cost_significant = (
                    cost_ci_lower > 0
                    or cost_ci_upper < 0
                )

                fill_significant = (
                    fill_ci_lower > 0
                    or fill_ci_upper < 0
                )

                if cost_ci_upper < 0:
                    lower_cost_model = model_a
                elif cost_ci_lower > 0:
                    lower_cost_model = model_b
                else:
                    lower_cost_model = (
                        "no_clear_difference"
                    )

                if fill_ci_lower > 0:
                    higher_fill_model = model_a
                elif fill_ci_upper < 0:
                    higher_fill_model = model_b
                else:
                    higher_fill_model = (
                        "no_clear_difference"
                    )

                bootstrap_rows.append({
                    "scenario":
                        scenario_name,
                    "demand_group":
                        demand_group,
                    "quantile_level":
                        quantile_level,
                    "model_a":
                        model_a,
                    "model_b":
                        model_b,
                    "series_count":
                        len(paired_index),
                    "bootstrap_resamples":
                        BOOTSTRAP_RESAMPLES,

                    "point_total_cost_difference_a_minus_b":
                        point_cost_difference,
                    "bootstrap_mean_cost_difference":
                        float(
                            cost_difference_draws.mean()
                        ),
                    "bootstrap_se_cost_difference":
                        float(
                            cost_difference_draws.std(
                                ddof=1
                            )
                        ),
                    "cost_ci_lower":
                        cost_ci_lower,
                    "cost_ci_upper":
                        cost_ci_upper,
                    "cost_ci_excludes_zero":
                        cost_significant,
                    "lower_cost_model":
                        lower_cost_model,

                    "point_fill_rate_difference_pp_a_minus_b":
                        point_fill_difference,
                    "bootstrap_mean_fill_difference_pp":
                        float(
                            fill_difference_draws.mean()
                        ),
                    "bootstrap_se_fill_difference_pp":
                        float(
                            fill_difference_draws.std(
                                ddof=1
                            )
                        ),
                    "fill_ci_lower_pp":
                        fill_ci_lower,
                    "fill_ci_upper_pp":
                        fill_ci_upper,
                    "fill_ci_excludes_zero":
                        fill_significant,
                    "higher_fill_model":
                        higher_fill_model
                })

        print(
            "Completed bootstrap:",
            f"q={quantile_level:.2f},",
            demand_group
        )


paired_bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ------------------------------------------------------------
# 5. Bootstrap checks
# ------------------------------------------------------------

expected_bootstrap_rows = (
    len(COST_SCENARIOS)
    * len(INVENTORY_QUANTILES)
    * len(DEMAND_GROUPS)
    * len(MODEL_PAIRS)
)

series_counts_correct = (
    paired_bootstrap_results.apply(
        lambda row:
            row["series_count"]
            == (
                500
                if row["demand_group"]
                == "Overall"
                else 125
            ),
        axis=1
    ).all()
)

bootstrap_numeric_columns = [
    "point_total_cost_difference_a_minus_b",
    "bootstrap_mean_cost_difference",
    "bootstrap_se_cost_difference",
    "cost_ci_lower",
    "cost_ci_upper",
    "point_fill_rate_difference_pp_a_minus_b",
    "bootstrap_mean_fill_difference_pp",
    "bootstrap_se_fill_difference_pp",
    "fill_ci_lower_pp",
    "fill_ci_upper_pp"
]

bootstrap_checks = pd.DataFrame({
    "check": [
        "Expected number of comparison rows",
        "Exactly 2000 bootstrap resamples",
        "Correct series counts",
        "No comparisons pair a model with itself",
        "Finite bootstrap results",
        "Cost interval lower bound not above upper bound",
        "Fill interval lower bound not above upper bound",
        "Exactly seven cost scenarios",
        "Exactly three quantile levels",
        "Exactly five demand groups"
    ],
    "passed": [
        len(paired_bootstrap_results)
        == expected_bootstrap_rows,

        paired_bootstrap_results[
            "bootstrap_resamples"
        ].eq(BOOTSTRAP_RESAMPLES).all(),

        series_counts_correct,

        (
            paired_bootstrap_results[
                "model_a"
            ]
            !=
            paired_bootstrap_results[
                "model_b"
            ]
        ).all(),

        np.isfinite(
            paired_bootstrap_results[
                bootstrap_numeric_columns
            ]
        ).all().all(),

        (
            paired_bootstrap_results[
                "cost_ci_lower"
            ]
            <=
            paired_bootstrap_results[
                "cost_ci_upper"
            ]
        ).all(),

        (
            paired_bootstrap_results[
                "fill_ci_lower_pp"
            ]
            <=
            paired_bootstrap_results[
                "fill_ci_upper_pp"
            ]
        ).all(),

        paired_bootstrap_results[
            "scenario"
        ].nunique() == len(COST_SCENARIOS),

        paired_bootstrap_results[
            "quantile_level"
        ].nunique() == 3,

        paired_bootstrap_results[
            "demand_group"
        ].nunique() == 5
    ]
})

if not bootstrap_checks[
    "passed"
].all():

    failed_checks = bootstrap_checks.loc[
        ~bootstrap_checks["passed"],
        "check"
    ].tolist()

    raise AssertionError(
        f"Bootstrap checks failed: "
        f"{failed_checks}"
    )


# ------------------------------------------------------------
# 6. Save bootstrap results
# ------------------------------------------------------------

paired_bootstrap_results.to_csv(
    OUTPUT_DIR
    / "inventory_paired_bootstrap_results.csv",
    index=False
)

bootstrap_checks.to_csv(
    OUTPUT_DIR
    / "inventory_paired_bootstrap_checks.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Display the main baseline comparisons
# ------------------------------------------------------------

main_bootstrap_results = (
    paired_bootstrap_results.loc[
        (
            paired_bootstrap_results[
                "scenario"
            ].eq("baseline")
        )
        &
        (
            paired_bootstrap_results[
                "demand_group"
            ].eq("Overall")
        )
        &
        (
            np.isclose(
                paired_bootstrap_results[
                    "quantile_level"
                ],
                0.99
            )
        )
    ]
    [
        [
            "model_a",
            "model_b",
            "point_total_cost_difference_a_minus_b",
            "cost_ci_lower",
            "cost_ci_upper",
            "lower_cost_model",
            "point_fill_rate_difference_pp_a_minus_b",
            "fill_ci_lower_pp",
            "fill_ci_upper_pp",
            "higher_fill_model"
        ]
    ]
)

print(
    "Paired bootstrap completed successfully."
)

print(
    "\nComparison rows:",
    len(paired_bootstrap_results)
)

print("\nBootstrap checks:")
print(bootstrap_checks)

print(
    "\nBaseline overall comparisons "
    "at q=0.99:"
)

print(
    main_bootstrap_results.to_string(
        index=False
    )
)

print("\nSaved file:")
print(
    OUTPUT_DIR
    / "inventory_paired_bootstrap_results.csv"
)


Completed bootstrap: q=0.90, Overall
Completed bootstrap: q=0.90, Smooth
Completed bootstrap: q=0.90, Erratic
Completed bootstrap: q=0.90, Intermittent
Completed bootstrap: q=0.90, Lumpy
Completed bootstrap: q=0.95, Overall
Completed bootstrap: q=0.95, Smooth
Completed bootstrap: q=0.95, Erratic
Completed bootstrap: q=0.95, Intermittent
Completed bootstrap: q=0.95, Lumpy
Completed bootstrap: q=0.99, Overall
Completed bootstrap: q=0.99, Smooth
Completed bootstrap: q=0.99, Erratic
Completed bootstrap: q=0.99, Intermittent
Completed bootstrap: q=0.99, Lumpy
Paired bootstrap completed successfully.

Comparison rows: 1050

Bootstrap checks:
                                             check  passed
0               Expected number of comparison rows    True
1                 Exactly 2000 bootstrap resamples    True
2                            Correct series counts    True
3          No comparisons pair a model with itself    True
4                         Finite bootstrap results    True
5 